# Trustworthy Multimodal Skin-Lesion Diagnosis on PAD-UFES-20
### Leakage-free, reliability-focused evaluation of image–metadata fusion (experimental pipeline)

**Study summary.** This notebook trains and evaluates image-only, metadata-only and four image–metadata fusion strategies (early concatenation, late/decision fusion, cross-attention, MetaBlock-style gating) on the PAD-UFES-20 dataset (Pacheco et al., *Data in Brief*, 2020; DOI 10.17632/zr7vgbcyr2.1) under 5-fold patient-grouped cross-validation. Every number, table, figure and caption is computed from the data by this notebook. Nothing is typed in by hand.

**Research questions**
- **RQ1 (discrimination).** Under a patient-grouped, leakage-free protocol, how do the fusion strategies compare with image-only and metadata-only baselines?
- **RQ2 (robustness).** How robust is each strategy when metadata is partly or fully missing at test time, and does modality-dropout training reduce the drop?
- **RQ3 (reliability).** Are the models well calibrated, and does selective prediction improve reliability?
- **RQ4 (fairness).** Does performance differ across Fitzpatrick skin types and other patient subgroups?

**How to run**
1. Put the Mendeley archive (or the extracted dataset) in Google Drive and set `DATA_PATH` in the configuration cell (Part 2). Alternatively set `DATA_SOURCE = "upload"`.
2. `Runtime → Change runtime type → T4 GPU`, then `Runtime → Restart and run all`.
3. If Colab disconnects, run all again. Finished fold/model runs are detected and skipped, and an interrupted run resumes from its last epoch checkpoint.
4. For a quick end-to-end debugging pass set `FAST_MODE = True`. Outputs then go to a separate `_FAST_DEBUG` folder and are **not for reported results**.

**Expected runtime (estimates, not measurements; the notebook logs the real times to `logs/` and prints a projection after the first training run)**

| Part | Content | Estimated time on a T4 |
|---|---|---|
| 1–2 | Environment, configuration | 2–4 min |
| 3 | Data extraction and image cache (first session only; cached afterwards) | 10–25 min |
| 4–5 | Splits, preprocessing | < 1 min |
| 6 | Training: 2 gradient-boosting models, 7 deep models × 5 folds, plus the optional leakage runs | roughly 4–8 h in total, normally spread over 2 or more free-tier sessions |
| 7–14 | Out-of-fold analysis, statistics, robustness, calibration, fairness, explainability | 10–25 min |
| 15–17 | Tables, figures, QC, summary | 5–10 min |

**Organisation.** Part numbers follow the specification (Parts 0–17). Every code cell is preceded by a Markdown cell with *Purpose / Why / Method choice / Output*.

## Part 1. Environment setup

**Purpose:** Install the small number of packages that are not guaranteed to be present in Colab, at pinned versions, and try to install the Arial font.

**Why:** Reproducibility requires known library versions. Journal figures require Arial; Colab does not ship it.

**Method choice:** `timm`, `lightgbm` and `python-docx` are pinned exactly. The heavy, CUDA-coupled packages that Colab pre-installs (PyTorch, torchvision, NumPy, pandas, SciPy, scikit-learn, matplotlib, Pillow) are **not** force-reinstalled: replacing Colab's PyTorch build takes several minutes per session and can break the CUDA pairing. Instead their minimum versions are asserted and the exact versions are recorded in `logs/requirements_lock.txt` and in Table T9, which is the information the reproducibility statement needs. This is a deliberate deviation from "pin everything"; set `STRICT_PIN_HEAVY` below if you want to force exact versions. Arial is installed from the Microsoft core fonts package; if that fails the notebook falls back to Liberation Sans (metric-compatible with Arial) and then DejaVu Sans and prints a warning.

**Output:** Installed packages; console log of what was installed. Nothing is saved yet (folders are created in Part 2).

In [ ]:
import importlib
import importlib.metadata as importlib_metadata
import subprocess
import sys

# Exact pins for the light packages this notebook installs itself.
PINNED_PACKAGES = {"timm": "1.0.31", "lightgbm": "4.7.0", "python-docx": "1.2.0"}
# Optional exact pins for the heavy packages, e.g. {"torch": "2.6.0"}. Empty = keep Colab's build.
STRICT_PIN_HEAVY = {}


def _installed_version(dist_name: str):
    """Return the installed version of a distribution, or None if it is not installed."""
    try:
        return importlib_metadata.version(dist_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def ensure_package(dist_name: str, version: str) -> None:
    """Install `dist_name==version` with pip unless exactly that version is already present.

    Raises a RuntimeError with the pip log if installation fails, so that a broken
    environment is never silently used.
    """
    have = _installed_version(dist_name)
    if have == version:
        print(f"[ok]      {dist_name}=={version} already installed")
        return
    print(f"[install] {dist_name}=={version} (found: {have})")
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"{dist_name}=={version}"],
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"pip could not install {dist_name}=={version}.\n{proc.stdout[-2000:]}\n{proc.stderr[-2000:]}\n"
            "Fix: check the internet connection, or change the pinned version in PINNED_PACKAGES "
            "and record the change in the paper's reproducibility statement."
        )


for _name, _ver in {**PINNED_PACKAGES, **STRICT_PIN_HEAVY}.items():
    ensure_package(_name, _ver)
importlib.invalidate_caches()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def try_install_arial() -> None:
    """Best-effort installation of Arial on Colab (Debian/Ubuntu).

    Recovery is meaningful here (a metric-compatible fallback font exists), so a failure
    only produces a warning; the font actually used is reported in Part 15 and in Table T9.
    """
    cmds = [
        "echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | debconf-set-selections",
        "apt-get -qq install -y ttf-mscorefonts-installer fonts-liberation > /dev/null 2>&1",
    ]
    for cmd in cmds:
        try:
            proc = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
        except subprocess.TimeoutExpired:
            print("[warning] Arial installation timed out; a fallback font will be used.")
            return
        if proc.returncode != 0:
            print("[warning] Arial installation command failed; a fallback font will be used.")
            return
    print("[ok]      font installation commands finished")


if IN_COLAB:
    try_install_arial()
else:
    print("[info]    not running in Colab: skipping font installation")

**Purpose:** Import all libraries, check the GPU, mount Google Drive, define the seeding function and print the software and hardware versions.

**Why:** A single import cell makes missing dependencies fail immediately. The version printout is the raw material for the reproducibility statement (Table T9).

**Method choice:** Seeds are set for Python, NumPy and PyTorch (CPU and CUDA), cuDNN is put in deterministic mode with benchmarking off, and `torch.use_deterministic_algorithms(True, warn_only=True)` is enabled. `warn_only=True` is used because a few CUDA operations have no deterministic implementation; forcing an error would make training impossible. **Remaining non-determinism:** mixed-precision arithmetic and those CUDA kernels can change results in the last decimals between runs and between GPU models; data order, splits, augmentation streams and all CPU analyses are exactly reproducible.

**Output:** Console printout; `ENV_INFO` dictionary (saved to `logs/environment.json` in Part 2).

In [ ]:
import copy
import datetime as dt
import gc
import glob
import hashlib
import io
import json
import math
import os
import platform
import random
import shutil
import time
import types
import warnings
import zipfile
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # required for deterministic cuBLAS

import joblib
import lightgbm as lgb
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import scipy
import scipy.optimize
import scipy.stats
import sklearn
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import Markdown, display
from PIL import Image
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2 as T

import docx  # python-docx
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Mm, Pt, RGBColor


def _version_tuple(v: str) -> Tuple[int, ...]:
    """Parse the leading numeric part of a version string ('2.6.0+cu124' -> (2, 6, 0))."""
    out = []
    for part in v.split("+")[0].split("."):
        digits = "".join(ch for ch in part if ch.isdigit())
        if digits == "":
            break
        out.append(int(digits))
    return tuple(out)


MIN_VERSIONS = {"torch": "2.3", "sklearn": "1.3", "pandas": "2.0", "matplotlib": "3.7", "numpy": "1.24"}
for _mod, _minv in MIN_VERSIONS.items():
    _have = importlib.import_module(_mod).__version__
    assert _version_tuple(_have) >= _version_tuple(_minv), (
        f"{_mod} {_have} is older than the minimum supported version {_minv}. "
        f"Fix: add an exact pin for it to STRICT_PIN_HEAVY in the first cell and restart the runtime."
    )


def seed_everything(seed: int) -> None:
    """Seed Python, NumPy and PyTorch (CPU + CUDA) and request deterministic kernels."""
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("=" * 78)
    print("WARNING: no GPU detected. Deep-model training on CPU is impractically slow for the\n"
          "full experiment. In Colab choose Runtime -> Change runtime type -> T4 GPU.")
    print("=" * 78)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

ENV_INFO: Dict[str, Any] = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "in_colab": IN_COLAB,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "lightgbm": lgb.__version__,
    "matplotlib": matplotlib.__version__,
    "Pillow": PIL.__version__,
    "python-docx": importlib_metadata.version("python-docx"),
    "device": DEVICE.type,
    "gpu_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU (CPU run)",
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if DEVICE.type == "cuda" else 0.0,
    "cuda": str(torch.version.cuda) if DEVICE.type == "cuda" else "not applicable",
    "cudnn": str(torch.backends.cudnn.version()) if DEVICE.type == "cuda" else "not applicable",
    "cpu_count": os.cpu_count(),
}
print("Software and hardware (for the reproducibility statement):")
for _k, _v in ENV_INFO.items():
    print(f"  {_k:<16s} {_v}")

## Part 2. Configuration

**Purpose:** One cell that holds every path, hyperparameter, backbone choice, seed, fold count, image size, epoch budget and analysis setting.

**Why:** A single source of truth makes the experiment auditable; the same dictionary is written to `logs/config.json`, hashed into every run record and exported as Table T7.

**Method choice:** Values that define the *results* are part of a configuration hash. If you change one of them after runs have been saved, the notebook stops with an explanation instead of silently mixing runs from two configurations. Batch size, gradient-accumulation steps and worker count are excluded from the hash so that you can lower memory use after an out-of-memory error without discarding finished runs (keep `BATCH_SIZE × GRAD_ACCUM_STEPS` constant when you do). Column names are **not** hard-coded: `COLUMN_CANDIDATES` lists spellings that are tried against the columns actually found in the file, and `COLUMN_OVERRIDES` lets you set any role explicitly after inspecting the printout in Part 3. `FAST_MODE` is for debugging only.

**Output:** The `CFG` namespace. **The only edit normally required is `DATA_PATH` (and `PROJECT_DIR` if you want another Drive folder).**

In [ ]:
CFG = types.SimpleNamespace(
    # ------------------------------------------------------------------ paths
    PROJECT_DIR="/content/drive/MyDrive/PAD_UFES20_Trustworthy",   # everything is saved here
    DATA_SOURCE="drive",        # "drive": DATA_PATH points to the archive or the extracted folder; "upload": browser upload
    DATA_PATH="/content/drive/MyDrive/PAD_UFES20_Trustworthy/Datasets/PAD-UFES-20.zip",   # dataset archive (a .zip file or a folder)
    METADATA_CSV=None,          # None -> auto-detect; set a full path only if auto-detection reports an ambiguity
    LOCAL_WORK_DIR="/content/pad_ufes_local",   # fast local scratch disk for extraction

    # ------------------------------------------------- dataset expectations
    # Counts stated by the dataset paper (Pacheco et al., 2020); asserted against the loaded file.
    # n_lesions is the number of distinct values in the lesion-id column; the number of distinct
    # (patient, lesion-id) pairs is reported separately (see Part 3).
    EXPECTED=dict(n_images=2298, n_patients=1373, n_lesions=1641, n_classes=6),
    COLUMN_OVERRIDES={},        # e.g. {"fitzpatrick": "fitspatrick"}; for list roles give a list of column names
    COLUMN_CANDIDATES=dict(     # spellings tried (case-insensitive) against the real header
        patient_id=["patient_id", "patient", "pat_id"],
        lesion_id=["lesion_id", "lesion"],
        img_id=["img_id", "image_id", "image", "img", "filename", "file_name"],
        label=["diagnostic", "diagnosis", "label", "class", "dx"],
        fitzpatrick=["fitspatrick", "fitzpatrick", "fitzpatrick_skin_type", "skin_type"],
        age=["age"],
        sex=["gender", "sex"],
        region=["region", "anatomical_region", "localization", "location"],
        biopsy=["biopsed", "biopsied", "biopsy"],
        diameter=["diameter_1", "diameter_2", "diameter"],                      # list role
        symptoms=["itch", "grew", "grow", "hurt", "changed", "bleed", "elevation"],     # list role
    ),
    LIST_ROLES=["diameter", "symptoms"],
    MISSING_TOKENS=["", "UNK", "UNKNOWN", "NAN", "NA", "N/A", "NONE", "NULL", "?"],   # compared after strip + upper-case
    # Roles/columns that must never be model inputs. The biopsy flag is excluded because, per the
    # dataset description, all cancers are biopsy-proven: it encodes the label (target leakage).
    METADATA_EXCLUDE_ROLES=["biopsy"],
    METADATA_EXCLUDE_COLUMNS=[],
    CANCER_CLASSES=["BCC", "SCC", "MEL"],       # malignant group for the cancer-vs-non-cancer analysis

    # ------------------------------------------------------- cross-validation
    SEED=42,
    N_FOLDS=5,
    N_REPEATS=1,                # 1 = single 5-fold CV. 3 = 3 x 5-fold (triples training time; enables the Wilcoxon test)
    INNER_VAL_SPLITS=5,         # inner patient-grouped split: 1/5 of each training fold is the validation set
    SPLIT_PROTOCOL_LABEL="5-fold patient-grouped stratified CV",   # used to flag literature rows with other protocols

    # ----------------------------------------------------------------- models
    MODELS_TO_RUN=["image_only", "meta_gbm", "meta_gbm_md",
                   "early_concat", "early_concat_md", "late_fusion", "late_fusion_md",
                   "cross_attn", "cross_attn_md", "metablock", "metablock_md"],
    BACKBONE="convnext_tiny",   # "convnext_tiny" | "efficientnet_b0" | "vit_small_dinov2"
    BACKBONE_REGISTRY=dict(
        convnext_tiny="convnext_tiny.fb_in1k",
        efficientnet_b0="efficientnet_b0.ra_in1k",
        vit_small_dinov2="vit_small_patch14_dinov2.lvd142m",
    ),
    PRETRAINED=True,
    IMG_SIZE=224,
    CACHE_SIZE=256,             # images are resized once to CACHE_SIZE x CACHE_SIZE and cached as uint8
    CACHE_RESIZE_MODE="squash", # "squash": keep all content, ignore aspect ratio; "center_crop": keep aspect, crop the centre
    EPOCHS=30,
    PATIENCE=8,                 # early stopping on validation balanced accuracy
    BATCH_SIZE=32,
    GRAD_ACCUM_STEPS=1,         # effective batch = BATCH_SIZE x GRAD_ACCUM_STEPS
    LR_BACKBONE=1e-4,
    LR_HEAD=1e-3,
    WEIGHT_DECAY=0.05,
    WARMUP_EPOCHS=2,
    GRAD_CLIP_NORM=1.0,
    DROPOUT=0.3,
    META_HIDDEN=128,
    FUSION_DIM=256,
    XATTN_HEADS=4,
    LABEL_SMOOTHING=0.0,        # kept at 0 so that calibration (RQ3) is not altered by the loss
    USE_AMP=True,
    NUM_WORKERS=2,
    # modality-dropout training (RQ2): per training sample, with probability MD_P_FULL all metadata
    # fields are masked; otherwise with probability MD_P_PARTIAL each field is masked with a rate ~ U(0, 1).
    MD_P_FULL=0.25,
    MD_P_PARTIAL=0.50,
    GBM_MD_N_AUG=4,             # masked copies of the training set for the gradient-boosting modality-dropout variant
    GBM_PARAMS=dict(n_estimators=2000, learning_rate=0.03, num_leaves=15, min_child_samples=10,
                    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0),
    GBM_EARLY_STOPPING_ROUNDS=100,
    KEEP_BEST_WEIGHTS="gradcam_model",   # "gradcam_model": keep best weights only for GRADCAM_MODEL (about 0.1 GB per fold);
                                # "all": keep them for every grouped-CV run (about 4 GB in total); "none": keep nothing
    REQUIRE_GPU=True,           # stop immediately if no GPU is attached (training on CPU would take days)

    # --------------------------------------------------------------- analyses
    N_BOOT=10000,               # patient-level cluster bootstrap resamples (>= 1000 required for reported results)
    CI_LEVEL=0.95,
    ECE_BINS=15,                # equal-width bins on the top-label confidence
    MCAR_RATES=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    N_MASK_SEEDS=5,
    # Structured missingness scenarios, defined on column ROLES resolved in Part 3.
    # "mask_roles": these roles are missing. "keep_roles": everything except these roles is missing.
    SCENARIOS=dict(
        no_diameter=dict(mask_roles=["diameter"]),
        no_symptoms=dict(mask_roles=["symptoms"]),
        smartphone_triage=dict(keep_roles=["age", "sex", "region", "symptoms"]),
    ),
    SELECTIVE_TARGET_COVERAGE=0.80,   # abstention threshold is chosen on validation data to reach this coverage
    MIN_SUBGROUP_N=30,          # subgroups with fewer images are reported but flagged and not interpreted
    AGE_BIN_EDGES=[0, 40, 60, 80, 200],
    RUN_RESTRICTED_SENSITIVITY=True,   # retrain metadata and fusion models on the low-missingness fields only (sensitivity analysis)
    RUN_SHORTCUT_AUDIT=True,    # Part 13: does the pattern of missing metadata reveal the diagnosis?
    AUDIT_ROW_MISSING_SHARE=0.5,    # an image counts as "metadata mostly missing" if at least this share of its fields is missing
    AUDIT_LOW_MISSING_MAX=0.05,     # fields missing in at most this share of images form the "low-missingness" field set
    RUN_LEAKAGE=True,           # Part 13: image-level StratifiedKFold vs patient-grouped split
    LEAKAGE_MODELS=["image_only", "meta_gbm"],
    RUN_EXPLAIN=True,           # Part 14: Grad-CAM and SHAP
    GRADCAM_MODEL="image_only",

    # ----------------------------------------------------- figures and tables
    FIG_WIDTHS_MM=dict(single_column=90.0, double_column=190.0),
    FIG_MAX_HEIGHT_MM=230.0,
    FIG_DPI=600,
    FONT_AXIS_PT=8.0,           # axis labels and legends
    FONT_TICK_PT=7.0,           # tick labels and in-plot annotations
    FONT_PANEL_PT=9.0,          # panel labels (a), (b), ...
    MIN_FONT_PT=7.0,
    SVG_TEXT_AS_PATHS=True,     # True: identical rendering everywhere; False: editable text (needs the font installed)
    TABLE_FONT="Times New Roman",
    TABLE_FONT_PT=9.0,
    DECIMALS=3,

    # -------------------------------------------------------------- debugging
    FAST_MODE=False,            # DEBUG ONLY - never use for reported results
)

**Purpose:** Validate the configuration, apply the `FAST_MODE` overrides, create the output folder tree, compute the configuration hash and save the configuration and environment records.

**Why:** Fail early on an invalid setting; keep debugging outputs physically separate from reportable outputs; give every run a fingerprint of the settings that produced it.

**Method choice:** The cell stops with instructions if no GPU is attached (`REQUIRE_GPU`), because a silent fall-back to CPU would make the run take days. It also prints the free space of the project folder. `FAST_MODE` writes to `PROJECT_DIR/_FAST_DEBUG/<configuration hash>/…` so that a debugging pass can never overwrite or be mistaken for the real experiment. The dataset folder and image cache are shared between the two modes because they do not depend on training settings. The folder layout follows the specification (`data, splits, checkpoints, predictions, results/tables, results/figures/{single_column,double_column}, results/captions, logs, qc_report`).

**Output:** `DIRS` (paths), `CFG_HASH`; files `logs/config.json`, `logs/environment.json`, `logs/requirements_lock.txt`.

In [ ]:
def finalise_config(cfg: types.SimpleNamespace) -> Tuple[Dict[str, Path], str]:
    """Validate `cfg`, apply FAST_MODE overrides, create folders and return (DIRS, CFG_HASH)."""
    assert cfg.DATA_SOURCE in ("drive", "upload"), "DATA_SOURCE must be 'drive' or 'upload'."
    assert cfg.BACKBONE in cfg.BACKBONE_REGISTRY, f"BACKBONE must be one of {list(cfg.BACKBONE_REGISTRY)}."
    assert cfg.N_FOLDS >= 2 and cfg.N_REPEATS >= 1 and cfg.INNER_VAL_SPLITS >= 2
    assert cfg.CACHE_SIZE >= cfg.IMG_SIZE, "CACHE_SIZE must be >= IMG_SIZE."
    assert cfg.CACHE_RESIZE_MODE in ("squash", "center_crop")
    assert 0.0 <= cfg.MD_P_FULL <= 1.0 and 0.0 <= cfg.MD_P_PARTIAL <= 1.0
    assert abs(cfg.MCAR_RATES[0]) < 1e-12 and abs(cfg.MCAR_RATES[-1] - 1.0) < 1e-12, \
        "MCAR_RATES must start at 0.0 and end at 1.0."
    assert cfg.GRAD_ACCUM_STEPS >= 1 and cfg.BATCH_SIZE >= 2
    if isinstance(cfg.KEEP_BEST_WEIGHTS, bool):            # accept the earlier True / False spelling
        cfg.KEEP_BEST_WEIGHTS = "all" if cfg.KEEP_BEST_WEIGHTS else "none"
    assert cfg.KEEP_BEST_WEIGHTS in ("gradcam_model", "all", "none"), "KEEP_BEST_WEIGHTS must be 'gradcam_model', 'all' or 'none'."
    if cfg.REQUIRE_GPU and DEVICE.type != "cuda":
        raise RuntimeError(
            "No GPU is attached to this runtime, so the notebook stops here instead of training on CPU for days.\n"
            "Fix: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save, then Runtime -> Restart and run all.\n"
            "If Colab says no GPU is currently available for your account, wait and try again later. "
            "(Set REQUIRE_GPU=False only to rerun the CPU-only analysis parts on finished runs.)")
    known = {"image_only", "meta_gbm", "meta_gbm_md", "early_concat", "early_concat_md", "late_fusion",
             "late_fusion_md", "cross_attn", "cross_attn_md", "metablock", "metablock_md"}
    unknown = set(cfg.MODELS_TO_RUN) - known
    assert not unknown, f"Unknown model names in MODELS_TO_RUN: {unknown}"
    for needed, deps in {"late_fusion": ["image_only", "meta_gbm"], "late_fusion_md": ["image_only", "meta_gbm_md"]}.items():
        if needed in cfg.MODELS_TO_RUN:
            missing = [d for d in deps if d not in cfg.MODELS_TO_RUN]
            assert not missing, f"{needed} needs {missing} in MODELS_TO_RUN."
    assert "image_only" in cfg.MODELS_TO_RUN, "image_only is the reference model for the statistical comparisons."

    if cfg.FAST_MODE:
        print("!" * 78)
        print("FAST_MODE is ON: 2 epochs, 200 bootstrap resamples, 2 mask seeds.\n"
              "DEBUGGING ONLY - outputs go to _FAST_DEBUG and must not be reported.")
        print("!" * 78)
        cfg.EPOCHS, cfg.PATIENCE, cfg.WARMUP_EPOCHS = 2, 2, 1
        cfg.N_BOOT, cfg.N_MASK_SEEDS = 200, 2
        cfg.GBM_PARAMS = dict(cfg.GBM_PARAMS, n_estimators=200)
        cfg.GBM_EARLY_STOPPING_ROUNDS = 20
    else:
        assert cfg.N_BOOT >= 1000, "N_BOOT must be >= 1000 for reported results."

    # Keys that do not change the scientific result are excluded from the hash.
    non_result_keys = {"PROJECT_DIR", "DATA_SOURCE", "DATA_PATH", "METADATA_CSV", "LOCAL_WORK_DIR", "BATCH_SIZE",
                       "GRAD_ACCUM_STEPS", "NUM_WORKERS", "KEEP_BEST_WEIGHTS", "REQUIRE_GPU", "MODELS_TO_RUN", "N_BOOT", "CI_LEVEL",
                       "ECE_BINS", "SELECTIVE_TARGET_COVERAGE", "MIN_SUBGROUP_N", "AGE_BIN_EDGES", "RUN_LEAKAGE",
                       "LEAKAGE_MODELS", "RUN_EXPLAIN", "GRADCAM_MODEL", "RUN_SHORTCUT_AUDIT", "AUDIT_ROW_MISSING_SHARE", "AUDIT_LOW_MISSING_MAX", "RUN_RESTRICTED_SENSITIVITY", "FIG_WIDTHS_MM", "FIG_MAX_HEIGHT_MM",
                       "FIG_DPI", "FONT_AXIS_PT", "FONT_TICK_PT", "FONT_PANEL_PT", "MIN_FONT_PT", "SVG_TEXT_AS_PATHS",
                       "TABLE_FONT", "TABLE_FONT_PT", "DECIMALS", "SPLIT_PROTOCOL_LABEL", "N_REPEATS"}
    hashed = {k: v for k, v in sorted(vars(cfg).items()) if k not in non_result_keys}
    hashed["EFFECTIVE_BATCH"] = cfg.BATCH_SIZE * cfg.GRAD_ACCUM_STEPS
    cfg_hash = hashlib.sha256(json.dumps(hashed, sort_keys=True, default=str).encode()).hexdigest()[:12]

    project = Path(cfg.PROJECT_DIR)
    # Debugging runs get one folder per configuration hash, so an old debugging run can never block a new one.
    out_root = project / "_FAST_DEBUG" / cfg_hash if cfg.FAST_MODE else project
    dirs = {
        "project": project,
        "data": project / "data",
        "out_root": out_root,
        "splits": out_root / "splits",
        "checkpoints": out_root / "checkpoints",
        "predictions": out_root / "predictions",
        "preprocessing": out_root / "checkpoints" / "preprocessing",
        "results": out_root / "results",
        "tables": out_root / "results" / "tables",
        "figures": out_root / "results" / "figures",
        "fig_single_column": out_root / "results" / "figures" / "single_column",
        "fig_double_column": out_root / "results" / "figures" / "double_column",
        "captions": out_root / "results" / "captions",
        "logs": out_root / "logs",
        "qc": out_root / "qc_report",
        "local": Path(cfg.LOCAL_WORK_DIR),
    }
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)
    if cfg.DATA_PATH is None:
        cfg.DATA_PATH = str(dirs["data"])

    return dirs, cfg_hash


DIRS, CFG_HASH = finalise_config(CFG)
seed_everything(CFG.SEED)


def save_json(obj: Any, path: Path) -> None:
    """Write JSON atomically (temporary file, then rename) so a disconnect cannot leave a half-written file."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str)
    os.replace(tmp, path)


def load_json(path: Path) -> Any:
    """Read a JSON file."""
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


save_json(dict(vars(CFG), CFG_HASH=CFG_HASH), DIRS["logs"] / "config.json")
save_json(ENV_INFO, DIRS["logs"] / "environment.json")
_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
(DIRS["logs"] / "requirements_lock.txt").write_text(_freeze.stdout if _freeze.returncode == 0 else "pip freeze failed\n")

try:
    _free_gb = shutil.disk_usage(DIRS["project"]).free / 1024**3
    print(f"Free space reported for the project folder: {_free_gb:.1f} GB"
          + ("  <-- LOW: the full experiment needs roughly 3 GB here; free up Drive space (and empty the Drive trash)." if _free_gb < 3.0 else ""))
except OSError as _exc:            # the figure is informational only; some mounts do not report it
    print(f"[info] free space could not be determined ({_exc})")
print(f"Project folder : {DIRS['project']}")
print(f"Output root    : {DIRS['out_root']}")
print(f"Config hash    : {CFG_HASH}")
print(f"Backbone       : {CFG.BACKBONE} -> {CFG.BACKBONE_REGISTRY[CFG.BACKBONE]} (pretrained={CFG.PRETRAINED})")

## Part 3. Data loading, verification and exploratory analysis

**Purpose:** Locate the dataset (Drive folder, Drive archive or browser upload), extract any archives (including archives nested inside the main archive), find the metadata file and index every image file.

**Why:** A download URL can break, so none is used. The archive layout is not assumed: files are discovered by searching, and the notebook stops with a clear message if the metadata file is missing or ambiguous.

**Method choice:** Archives are extracted to Colab's local disk (fast) rather than to Drive (slow for thousands of small files). The metadata CSV is identified as the CSV whose header contains an image-identifier column and a label column from `COLUMN_CANDIDATES`; if several different CSVs qualify, the notebook stops and asks for `CFG.METADATA_CSV`. A browser upload of the full archive (about 3 GB) is slow and can time out; placing the archive in Drive is the more reliable option.

**Output:** `METADATA_PATH`, `IMAGE_INDEX` (file name → path). A copy of the metadata file is stored in `data/metadata_copy.csv`.

In [ ]:
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
GENERATED_PREFIXES = ("image_cache", "image_manifest", "metadata_copy")   # files this notebook writes into data/


def extract_archives(source: Path, dest: Path, max_depth: int = 3) -> int:
    """Extract every .zip under `source` (a file or a folder) into `dest`, following nested zips.

    A marker file makes the step idempotent across sessions on the same machine.
    Returns the number of archives extracted in this call.
    """
    source, dest = Path(source), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    n_done = 0
    queue: List[Tuple[Path, int]] = []
    if source.is_file() and source.suffix.lower() == ".zip":
        queue.append((source, 0))
    elif source.is_dir():
        queue.extend((p, 0) for p in sorted(source.rglob("*.zip")))
    seen = set()
    while queue:
        zpath, depth = queue.pop(0)
        key = (zpath.name, zpath.stat().st_size)
        if key in seen or depth > max_depth:
            continue
        seen.add(key)
        target = dest / f"{zpath.stem}__{zpath.stat().st_size}"
        marker = target / ".extracted_ok"
        if not marker.exists():
            print(f"  extracting {zpath.name} ({zpath.stat().st_size / 1024**2:.0f} MB) ...")
            if not zipfile.is_zipfile(zpath):
                raise RuntimeError(f"{zpath} is not a valid zip archive (incomplete upload?). Re-copy the file.")
            with zipfile.ZipFile(zpath) as zf:
                zf.extractall(target)
            marker.write_text("ok")
            n_done += 1
        queue.extend((p, depth + 1) for p in sorted(target.rglob("*.zip")))
    return n_done


def find_files(roots: Sequence[Path], suffixes: Tuple[str, ...]) -> List[Path]:
    """Recursively list files with the given suffixes under several roots (sorted, de-duplicated)."""
    found = set()
    for root in roots:
        root = Path(root)
        if root.is_dir():
            for p in root.rglob("*"):
                if p.is_file() and p.suffix.lower() in suffixes and "__MACOSX" not in p.parts:
                    found.add(p)
    return sorted(found)


def _header_has_role(columns: Sequence[str], role: str) -> bool:
    """True if any candidate spelling (or override) for `role` is present in `columns` (case-insensitive)."""
    lower = {str(c).strip().lower() for c in columns}
    cands = list(CFG.COLUMN_CANDIDATES.get(role, []))
    if role in CFG.COLUMN_OVERRIDES:
        ov = CFG.COLUMN_OVERRIDES[role]
        cands = list(ov) if isinstance(ov, (list, tuple)) else [ov]
    return any(c.lower() in lower for c in cands)


def locate_metadata_csv(roots: Sequence[Path]) -> Path:
    """Return the path of the metadata CSV, or raise with instructions if none/ambiguous."""
    if CFG.METADATA_CSV is not None:
        p = Path(CFG.METADATA_CSV)
        if not p.is_file():
            raise FileNotFoundError(f"CFG.METADATA_CSV = {p} does not exist.")
        return p
    candidates = []
    for p in find_files(roots, (".csv",)):
        if p.name.startswith(GENERATED_PREFIXES):
            continue
        try:
            header = pd.read_csv(p, nrows=0).columns
        except (pd.errors.ParserError, pd.errors.EmptyDataError, UnicodeDecodeError):
            continue   # not a readable table: cannot be the metadata file
        if _header_has_role(header, "img_id") and _header_has_role(header, "label"):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError(
            "No metadata CSV with an image-id column and a label column was found under:\n  "
            + "\n  ".join(str(r) for r in roots)
            + "\nFix: check CFG.DATA_PATH, or set CFG.METADATA_CSV, or add the real column names to CFG.COLUMN_OVERRIDES."
        )
    digests = {hashlib.md5(p.read_bytes()).hexdigest() for p in candidates}
    if len(digests) > 1:
        raise RuntimeError("Several different candidate metadata files were found:\n  "
                           + "\n  ".join(map(str, candidates)) + "\nFix: set CFG.METADATA_CSV to the correct one.")
    return candidates[0]


data_path = Path(CFG.DATA_PATH)
if CFG.DATA_SOURCE == "upload":
    if not IN_COLAB:
        raise RuntimeError("DATA_SOURCE='upload' needs Google Colab. Use DATA_SOURCE='drive' and set DATA_PATH.")
    from google.colab import files as colab_files
    upload_dir = DIRS["local"] / "upload"
    upload_dir.mkdir(parents=True, exist_ok=True)
    if not any(upload_dir.iterdir()):
        print("Select the Mendeley archive (.zip) in the file dialog ...")
        for _name, _blob in colab_files.upload().items():
            (upload_dir / _name).write_bytes(_blob)
    data_path = upload_dir
if not data_path.exists():
    raise FileNotFoundError(f"DATA_PATH does not exist: {data_path}\nFix: set CFG.DATA_PATH in the configuration cell "
                            "to the Drive folder (or .zip file) that holds PAD-UFES-20.")

EXTRACT_DIR = DIRS["local"] / "extracted"
n_extracted = extract_archives(data_path, EXTRACT_DIR)
SEARCH_ROOTS = [data_path if data_path.is_dir() else data_path.parent, EXTRACT_DIR, DIRS["data"]]
print(f"Archives extracted in this session: {n_extracted}")

METADATA_COPY = DIRS["data"] / "metadata_copy.csv"
try:
    METADATA_PATH = locate_metadata_csv(SEARCH_ROOTS)
    if METADATA_PATH.resolve() != METADATA_COPY.resolve():
        shutil.copyfile(METADATA_PATH, METADATA_COPY)
except FileNotFoundError:
    if METADATA_COPY.is_file():      # later sessions: the raw archive may have been removed, the copy remains
        METADATA_PATH = METADATA_COPY
        print("[info] raw metadata file not found; using the copy saved in an earlier session.")
    else:
        raise

IMAGE_INDEX: Dict[str, Path] = {}
_dupes: Dict[str, List[Path]] = {}
for _p in find_files(SEARCH_ROOTS, IMAGE_EXTENSIONS):
    if _p.name in IMAGE_INDEX:
        _dupes.setdefault(_p.name, [IMAGE_INDEX[_p.name]]).append(_p)
    else:
        IMAGE_INDEX[_p.name] = _p
print(f"Metadata file : {METADATA_PATH}")
print(f"Image files   : {len(IMAGE_INDEX)} unique file names found"
      + (f" ({len(_dupes)} names occur more than once; the first path is used)" if _dupes else ""))

**Purpose:** Load the metadata file, print what is actually in it (columns, dtypes, unique values, missing-value encodings), resolve the column roles, and assert the dataset counts.

**Why:** The specification forbids assuming file structure. The printout is the evidence for every later processing decision and should be checked once after the first run.

**Method choice:** The file is read with `keep_default_na=False` and all cells as text first, so that string encodings of missingness such as `UNK`, empty strings or `NaN` are seen exactly as stored rather than being silently converted by pandas. Roles are resolved by case-insensitive exact match against `COLUMN_CANDIDATES`, or taken from `COLUMN_OVERRIDES`. Required roles (`patient_id`, `lesion_id`, `img_id`, `label`) raise an error if unresolved; optional roles only disable the analyses that need them, which are then reported as "n/a". Counts are asserted against `CFG.EXPECTED` and a failure stops the notebook.

**Output:** `RAW` (text table as stored), `ROLES`, `DF` (one row per image, sorted by image id, with standardised helper columns), `CLASS_NAMES`; file `logs/metadata_inspection.csv`.

In [ ]:
RAW = pd.read_csv(METADATA_PATH, dtype=str, keep_default_na=False)
RAW.columns = [str(c).strip() for c in RAW.columns]
assert RAW.shape[0] > 0, "The metadata file has no rows."
assert len(set(RAW.columns)) == len(RAW.columns), f"Duplicate column names in metadata: {list(RAW.columns)}"
MISSING_TOKEN_SET = {t.strip().upper() for t in CFG.MISSING_TOKENS}


def is_missing_token(series: pd.Series) -> pd.Series:
    """Boolean mask: True where the stored text is one of the configured missing-value encodings."""
    return series.astype(str).str.strip().str.upper().isin(MISSING_TOKEN_SET)


_rows = []
for _c in RAW.columns:
    _s = RAW[_c].astype(str)
    _miss = is_missing_token(_s)
    _tokens = sorted(_s[_miss].str.strip().unique().tolist())
    _vals = _s[~_miss]
    _num = pd.to_numeric(_vals, errors="coerce")
    _uniq = sorted(_vals.unique().tolist())
    _rows.append(dict(
        column=_c, n_unique_non_missing=len(_uniq), n_missing=int(_miss.sum()),
        pct_missing=round(100.0 * float(_miss.mean()), 2),
        missing_encodings_found=" | ".join(repr(t) for t in _tokens) if _tokens else "(none)",
        all_numeric=bool(len(_vals) > 0 and _num.notna().all()),
        example_values=", ".join(_uniq[:8]) + (" ..." if len(_uniq) > 8 else ""),
    ))
INSPECTION = pd.DataFrame(_rows)
INSPECTION.to_csv(DIRS["logs"] / "metadata_inspection.csv", index=False)
print(f"Metadata shape as stored: {RAW.shape[0]} rows x {RAW.shape[1]} columns\n")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 70, "display.width", 250):
    display(INSPECTION)


def resolve_roles(columns: Sequence[str]) -> Dict[str, Any]:
    """Map each semantic role to the real column name(s); list roles map to a (possibly empty) list."""
    lower_to_real = {c.lower(): c for c in columns}
    roles: Dict[str, Any] = {}
    for role, cands in CFG.COLUMN_CANDIDATES.items():
        is_list = role in CFG.LIST_ROLES
        if role in CFG.COLUMN_OVERRIDES:
            ov = CFG.COLUMN_OVERRIDES[role]
            ov_list = list(ov) if isinstance(ov, (list, tuple)) else [ov]
            bad = [o for o in ov_list if o not in columns]
            if bad:
                raise KeyError(f"COLUMN_OVERRIDES['{role}'] names columns that do not exist: {bad}. "
                               f"Available columns: {list(columns)}")
            roles[role] = ov_list if is_list else ov_list[0]
            continue
        hits = [lower_to_real[c.lower()] for c in cands if c.lower() in lower_to_real]
        roles[role] = hits if is_list else (hits[0] if hits else None)
    return roles


ROLES = resolve_roles(list(RAW.columns))
print("\nResolved column roles (check these against the table above):")
for _r, _c in ROLES.items():
    print(f"  {_r:<12s} -> {_c if _c not in (None, []) else 'NOT FOUND'}")
for _r in ("patient_id", "lesion_id", "img_id", "label"):
    if ROLES[_r] is None:
        raise KeyError(f"Required role '{_r}' could not be matched to a column. "
                       f"Fix: set CFG.COLUMN_OVERRIDES['{_r}'] to the real column name. Columns: {list(RAW.columns)}")
for _r in ("fitzpatrick", "age", "sex", "region", "biopsy"):
    if ROLES[_r] is None:
        print(f"[warning] optional role '{_r}' not found: analyses that need it will be reported as n/a.")

for _r in ("patient_id", "lesion_id", "img_id", "label"):
    _n = int(is_missing_token(RAW[ROLES[_r]]).sum())
    if _n:
        raise ValueError(f"Column '{ROLES[_r]}' ({_r}) has {_n} missing values; identifiers and labels must be complete.")

DF = RAW.copy()
DF["_img"] = DF[ROLES["img_id"]].str.strip()
DF["_patient"] = DF[ROLES["patient_id"]].str.strip()
DF["_lesion"] = DF[ROLES["patient_id"]].str.strip() + "::" + DF[ROLES["lesion_id"]].str.strip()
DF["_label"] = DF[ROLES["label"]].str.strip().str.upper()
if DF["_img"].duplicated().any():
    raise ValueError(f"Duplicate image identifiers: {DF.loc[DF['_img'].duplicated(), '_img'].head(10).tolist()}")
DF = DF.sort_values("_img", kind="mergesort").reset_index(drop=True)   # one fixed row order for the whole study

CLASS_NAMES: List[str] = sorted(DF["_label"].unique().tolist())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
DF["y"] = DF["_label"].map(CLASS_TO_IDX).astype(int)
N, K = len(DF), len(CLASS_NAMES)
Y = DF["y"].to_numpy()

# Lesions. A lesion is identified by the pair (patient id, lesion id). The number of distinct lesion-id
# values alone can be smaller, because the same lesion-id value can occur under different patient ids;
# both counts are computed and reported, and nothing is merged or dropped.
N_LESION_IDS = int(DF[ROLES["lesion_id"]].str.strip().nunique())
N_LESION_PAIRS = int(DF["_lesion"].nunique())
_pat_per_lesion_id = DF.groupby(DF[ROLES["lesion_id"]].str.strip())["_patient"].nunique()
N_SHARED_LESION_IDS = int((_pat_per_lesion_id > 1).sum())
OBSERVED = dict(n_images=N, n_patients=int(DF["_patient"].nunique()), n_lesions=N_LESION_IDS, n_classes=K)
print(f"\nObserved counts: {OBSERVED}")
print(f"Distinct lesion-id values: {N_LESION_IDS}; distinct (patient id, lesion id) pairs: {N_LESION_PAIRS}; "
      f"lesion-id values that occur under more than one patient id: {N_SHARED_LESION_IDS}")
if N_SHARED_LESION_IDS:
    print("[note] Lesion-id values are not unique across patients in this file. Lesions are therefore identified by the\n"
          "       (patient id, lesion id) pair, and that pair count is the number of lesions reported in Table T1.\n"
          "       Splits are grouped by patient, so this does not affect the leakage protection.")
_bad = {k: (OBSERVED[k], CFG.EXPECTED[k]) for k in CFG.EXPECTED if OBSERVED[k] != CFG.EXPECTED[k]}
if _bad:
    raise AssertionError(
        f"Dataset counts differ from the expected values (observed, expected): {_bad}.\n"
        "Fix: confirm that the complete PAD-UFES-20 metadata file is used and that the roles above are correct. "
        "Only if you have verified a different dataset version, update CFG.EXPECTED and report it in the paper."
    )
_missing_cancer = [c for c in CFG.CANCER_CLASSES if c not in CLASS_NAMES]
if _missing_cancer:
    raise AssertionError(f"CANCER_CLASSES {_missing_cancer} are not among the class codes found in the file: {CLASS_NAMES}. "
                         "Fix: edit CFG.CANCER_CLASSES to match the codes printed above.")
CANCER_IDX = np.array([CLASS_TO_IDX[c] for c in CFG.CANCER_CLASSES], dtype=int)
Y_CANCER = np.isin(Y, CANCER_IDX).astype(int)
PATIENTS = DF["_patient"].to_numpy()
IMG_IDS = DF["_img"].to_numpy()
print(f"Classes (alphabetical index order): {CLASS_TO_IDX}")
print(f"Cancer group: {CFG.CANCER_CLASSES}; non-cancer group: {[c for c in CLASS_NAMES if c not in CFG.CANCER_CLASSES]}")

**Purpose:** Verify that every metadata row has an image file, then build (or reload) the resized image cache.

**Why:** A row without an image, or an unreadable image, would otherwise surface hours later inside a training loop. Decoding the original smartphone photographs on Colab's two CPU cores in every epoch would dominate the runtime; a one-off resize to a small `uint8` array removes that bottleneck and makes each epoch depend on the GPU only.

**Method choice:** Images are resized once to `CACHE_SIZE × CACHE_SIZE` with bicubic interpolation (Pillow reducing filter, anti-aliased). The default `"squash"` mode keeps the whole photograph and ignores the aspect ratio, which avoids cutting off lesions that are not centred; `"center_crop"` is available as an alternative. Training-time augmentation then crops to `IMG_SIZE`. The cache and a manifest (file name, original width and height) are stored in `data/` on Drive, so later sessions do not need the raw archive. Any missing or unreadable file stops the notebook and is listed.

The cell also looks for **exact duplicate images** (identical pixel content after resizing). Duplicates inside one patient are harmless under patient-grouped splitting; a duplicate shared by two patients would leak across a patient-grouped split, so such patients are merged into one group for splitting and for the bootstrap (`CV_GROUPS`). Near-duplicates (re-photographed lesions) are not detected by this test.

**Output:** `IMAGES` (`uint8` array, N × S × S × 3, in RAM), `IMAGE_MANIFEST`, `DUPLICATES`, `CV_GROUPS`; file `logs/duplicate_images.csv`; files `data/image_cache_<S>_<mode>.npy` and `data/image_manifest_<S>_<mode>.csv`.

In [ ]:
def match_image(img_id: str) -> Optional[Path]:
    """Find the file for a metadata image id: exact file name first, then the id with a known extension."""
    if img_id in IMAGE_INDEX:
        return IMAGE_INDEX[img_id]
    for ext in IMAGE_EXTENSIONS:
        if img_id + ext in IMAGE_INDEX:
            return IMAGE_INDEX[img_id + ext]
    return None


def load_resized(path: Path, size: int, mode: str) -> Tuple[np.ndarray, int, int]:
    """Read one image as RGB and resize it to (size, size); returns (array, original width, original height)."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size
        if mode == "center_crop":
            s = min(w, h)
            left, top = (w - s) // 2, (h - s) // 2
            im = im.crop((left, top, left + s, top + s))
        im = im.resize((size, size), resample=Image.BICUBIC)
        return np.asarray(im, dtype=np.uint8), w, h


_cache_tag = f"{CFG.CACHE_SIZE}_{CFG.CACHE_RESIZE_MODE}"
CACHE_NPY = DIRS["data"] / f"image_cache_{_cache_tag}.npy"
CACHE_MANIFEST = DIRS["data"] / f"image_manifest_{_cache_tag}.csv"

_cache_ok = False
if CACHE_NPY.is_file() and CACHE_MANIFEST.is_file():
    IMAGE_MANIFEST = pd.read_csv(CACHE_MANIFEST, dtype={"img_id": str})
    _cache_ok = (len(IMAGE_MANIFEST) == N) and (IMAGE_MANIFEST["img_id"].to_numpy() == IMG_IDS).all()
    if not _cache_ok:
        print("[info] existing image cache does not match the metadata rows; it will be rebuilt.")

if _cache_ok:
    print(f"Loading image cache {CACHE_NPY.name} ...")
    IMAGES = np.load(CACHE_NPY)
    print("Every metadata row has a cached image (verified against the cache manifest written when the cache was built).")
else:
    _paths = [match_image(i) for i in IMG_IDS]
    _missing = [i for i, p in zip(IMG_IDS, _paths) if p is None]
    _unused = sorted(set(IMAGE_INDEX) - {p.name for p in _paths if p is not None})
    print(f"Metadata rows: {N}; rows with an image file: {N - len(_missing)}; rows WITHOUT an image file: {len(_missing)}")
    print(f"Image files not referenced by any metadata row: {len(_unused)}" + (f" (e.g. {_unused[:5]})" if _unused else ""))
    if _missing:
        pd.DataFrame({"img_id": _missing}).to_csv(DIRS["logs"] / "missing_image_files.csv", index=False)
        raise FileNotFoundError(
            f"{len(_missing)} metadata rows have no image file (first: {_missing[:5]}); full list in logs/missing_image_files.csv.\n"
            "Fix: make sure all image archives were copied completely to CFG.DATA_PATH and rerun this cell."
        )
    _tmp = CACHE_NPY.with_name(CACHE_NPY.name + ".tmp.npy")
    IMAGES = np.lib.format.open_memmap(_tmp, mode="w+", dtype=np.uint8, shape=(N, CFG.CACHE_SIZE, CFG.CACHE_SIZE, 3))
    _wh, _t0 = [], time.time()
    for _i, _p in enumerate(_paths):
        try:
            IMAGES[_i], _w, _h = load_resized(_p, CFG.CACHE_SIZE, CFG.CACHE_RESIZE_MODE)
        except (OSError, ValueError) as exc:
            raise RuntimeError(f"Image file could not be read: {_p} ({exc}). Fix: re-copy the dataset; the file is corrupt.") from exc
        _wh.append((_w, _h))
        if (_i + 1) % 500 == 0 or _i + 1 == N:
            print(f"  cached {_i + 1}/{N} images ({time.time() - _t0:.0f} s)")
    IMAGES.flush()
    del IMAGES
    os.replace(_tmp, CACHE_NPY)
    IMAGE_MANIFEST = pd.DataFrame({"img_id": IMG_IDS, "file_name": [p.name for p in _paths],
                                   "orig_width": [w for w, _ in _wh], "orig_height": [h for _, h in _wh]})
    IMAGE_MANIFEST.to_csv(CACHE_MANIFEST, index=False)
    IMAGES = np.load(CACHE_NPY)

assert IMAGES.shape == (N, CFG.CACHE_SIZE, CFG.CACHE_SIZE, 3) and IMAGES.dtype == np.uint8
print(f"Image cache: {IMAGES.shape}, {IMAGES.nbytes / 1024**2:.0f} MB in RAM")

# ---- exact duplicate images, and the grouping unit for cross-validation and the bootstrap
_digest = np.array([hashlib.md5(IMAGES[i].tobytes()).hexdigest() for i in range(N)])
_dup = pd.DataFrame({"img_id": IMG_IDS, "patient": PATIENTS, "label": DF["_label"].to_numpy(), "digest": _digest})
_dup = _dup[_dup.duplicated("digest", keep=False)].sort_values(["digest", "img_id"])
_dup.to_csv(DIRS["logs"] / "duplicate_images.csv", index=False)
_g = _dup.groupby("digest")
DUPLICATES = dict(n_groups=int(_g.ngroups), n_images=int(len(_dup)),
                  n_groups_cross_patient=int((_g["patient"].nunique() > 1).sum()) if len(_dup) else 0,
                  n_groups_label_conflict=int((_g["label"].nunique() > 1).sum()) if len(_dup) else 0)
# Patients that share an identical image are put into one group, so that a duplicate can never sit on
# both sides of a split. Without such duplicates every group is exactly one patient.
_parent = {p: p for p in np.unique(PATIENTS)}


def _find(p: str) -> str:
    """Union-find root with path compression."""
    while _parent[p] != p:
        _parent[p] = _parent[_parent[p]]
        p = _parent[p]
    return p


if len(_dup):
    for _, _members in _g["patient"]:
        _ps = sorted(set(_members))
        for _other in _ps[1:]:
            _parent[_find(_other)] = _find(_ps[0])
CV_GROUPS = np.array([_find(p) for p in PATIENTS])
N_CV_GROUPS = int(len(np.unique(CV_GROUPS)))
GROUPS_ARE_PATIENTS = N_CV_GROUPS == OBSERVED["n_patients"]
print(f"Exact duplicate images (identical after resizing): {DUPLICATES['n_images']} images in {DUPLICATES['n_groups']} groups; "
      f"groups spanning more than one patient: {DUPLICATES['n_groups_cross_patient']}; groups with conflicting labels: {DUPLICATES['n_groups_label_conflict']}")
print(f"Grouping unit for splits and bootstrap: {N_CV_GROUPS} groups"
      + (" (one per patient)." if GROUPS_ARE_PATIENTS else f" ({OBSERVED['n_patients']} patients; patients sharing an identical image were merged)."))

**Purpose:** Convert the metadata to a clean, typed table of model input fields, with all missing-value encodings mapped to a real missing value.

**Why:** Models and imputers need a consistent representation of "missing". Identifier, label and leakage-prone columns must be excluded from the inputs.

**Method choice:** This step is *stateless*: it only recodes each cell (trim, upper-case, map the configured missing tokens to `NaN`) and decides whether a column is numeric (all non-missing entries parse as numbers) or categorical. No statistic is estimated here, so it is safe to apply to the whole dataset; everything that estimates parameters (medians, scales, category sets) is fitted on training folds only in Part 5. The biopsy flag is excluded from the inputs because it encodes the label (all cancers are biopsy-proven according to the dataset description). Constant columns are dropped and reported. Fitzpatrick type is kept as an input by default because it is clinically available; it is additionally used as the grouping variable in RQ4.

**Output:** `META` (typed feature table), `NUM_COLS`, `CAT_COLS`, `FIELD_NAMES`, `FIELD_ROLE_MAP`; file `logs/metadata_fields.csv`.

In [ ]:
_exclude = {ROLES["patient_id"], ROLES["lesion_id"], ROLES["img_id"], ROLES["label"]}
for _r in CFG.METADATA_EXCLUDE_ROLES:
    _c = ROLES.get(_r)
    if _c:
        _exclude.update(_c if isinstance(_c, list) else [_c])
_exclude.update(CFG.METADATA_EXCLUDE_COLUMNS)

NUM_COLS: List[str] = []
CAT_COLS: List[str] = []
_dropped: List[Tuple[str, str]] = []
_meta_cols: Dict[str, pd.Series] = {}
for _c in RAW.columns:
    if _c in _exclude:
        _dropped.append((_c, "identifier, label or excluded role"))
        continue
    _s = DF[_c].astype(str).str.strip()
    _miss = is_missing_token(_s)
    _vals = _s[~_miss]
    if _vals.nunique() <= 1:
        _dropped.append((_c, f"constant or empty ({_vals.nunique()} distinct non-missing values)"))
        continue
    _num = pd.to_numeric(_vals, errors="coerce")
    if _num.notna().all():
        _col = pd.to_numeric(_s.where(~_miss), errors="coerce").astype(float)
        NUM_COLS.append(_c)
    else:
        # explicit object array with np.nan so that the result is identical under pandas 2 and pandas 3
        _col = pd.Series(np.where(_miss.to_numpy(), np.nan, _s.str.upper().to_numpy(dtype=object)), index=DF.index, dtype=object)
        CAT_COLS.append(_c)
    _meta_cols[_c] = _col

META = pd.DataFrame(_meta_cols)[NUM_COLS + CAT_COLS]
FIELD_NAMES: List[str] = NUM_COLS + CAT_COLS
N_FIELDS = len(FIELD_NAMES)
assert N_FIELDS >= 1, "No usable metadata fields were found."
MISSING_MATRIX = META.isna().to_numpy()            # (N, N_FIELDS) natural missingness

FIELD_ROLE_MAP: Dict[str, str] = {}
for _r, _c in ROLES.items():
    for _name in (_c if isinstance(_c, list) else [_c]):
        if _name in FIELD_NAMES:
            FIELD_ROLE_MAP[_name] = _r

_fields = pd.DataFrame({
    "field": FIELD_NAMES,
    "type": ["numeric"] * len(NUM_COLS) + ["categorical"] * len(CAT_COLS),
    "role": [FIELD_ROLE_MAP.get(f, "") for f in FIELD_NAMES],
    "n_missing": MISSING_MATRIX.sum(axis=0),
    "pct_missing": np.round(100.0 * MISSING_MATRIX.mean(axis=0), 2),
    "n_categories": [np.nan] * len(NUM_COLS) + [int(META[c].nunique()) for c in CAT_COLS],
})
_fields.to_csv(DIRS["logs"] / "metadata_fields.csv", index=False)
print(f"Model input fields: {N_FIELDS} ({len(NUM_COLS)} numeric, {len(CAT_COLS)} categorical)")
display(_fields)
print("Columns not used as inputs:")
for _c, _why in _dropped:
    print(f"  {_c:<24s} {_why}")

**Purpose:** Exploratory analysis: class distribution, images per lesion and per patient, metadata missingness (overall and per class), Fitzpatrick distribution, image sizes.

**Why:** These computed statistics populate Table T1 and Figure F1, and they determine which later analyses are statistically meaningful (for example, which skin-type subgroups are large enough to interpret).

**Method choice:** Plain counts and proportions computed from the loaded table. Missingness is reported per class as well as overall because missingness that depends on the diagnosis is itself predictive. **Read the missingness-by-class table carefully:** if a block of fields is complete for some diagnoses and largely missing for others, the missingness pattern is a shortcut to the label; Part 13 contains an audit that quantifies this, and it affects the interpretation of RQ1, RQ2 and RQ4.

**Output:** `EDA` dictionary; files `logs/eda_*.csv`.

In [ ]:
EDA: Dict[str, Any] = {}
_cls = DF["_label"].value_counts().reindex(CLASS_NAMES)
EDA["class_counts"] = pd.DataFrame({
    "class": CLASS_NAMES, "n_images": _cls.to_numpy(),
    "pct_images": 100.0 * _cls.to_numpy() / N,
    "n_lesions": [int(DF.loc[DF["_label"] == c, "_lesion"].nunique()) for c in CLASS_NAMES],
    "n_patients": [int(DF.loc[DF["_label"] == c, "_patient"].nunique()) for c in CLASS_NAMES],
})
EDA["images_per_patient"] = DF.groupby("_patient").size().to_numpy()
EDA["images_per_lesion"] = DF.groupby("_lesion").size().to_numpy()
EDA["lesions_per_patient"] = DF.groupby("_patient")["_lesion"].nunique().to_numpy()
EDA["patients_multi_class"] = int((DF.groupby("_patient")["_label"].nunique() > 1).sum())
EDA["missing_by_class"] = pd.DataFrame(
    [[float(MISSING_MATRIX[Y == k, j].mean()) for j in range(N_FIELDS)] for k in range(K)],
    index=CLASS_NAMES, columns=FIELD_NAMES)
EDA["missing_overall"] = pd.Series(MISSING_MATRIX.mean(axis=0), index=FIELD_NAMES)

if ROLES["fitzpatrick"] is not None:
    _f = DF[ROLES["fitzpatrick"]].astype(str).str.strip()
    _fnum = pd.to_numeric(_f.where(~is_missing_token(_f)), errors="coerce")
    # Integer-valued codes are shown without a decimal point ("2", not "2.0").
    FITZ = _fnum.map(lambda v: "Missing" if pd.isna(v) else (str(int(v)) if float(v).is_integer() else str(v)))
    if _fnum.isna().sum() != is_missing_token(_f).sum():
        raise ValueError(f"Column '{ROLES['fitzpatrick']}' has non-numeric, non-missing entries; inspect it and "
                         "extend CFG.MISSING_TOKENS or recode it before continuing.")
    DF["_fitz"] = FITZ.to_numpy()
    _levels = sorted([l for l in DF["_fitz"].unique() if l != "Missing"], key=float) + (["Missing"] if (DF["_fitz"] == "Missing").any() else [])
    EDA["fitz_levels"] = _levels
    EDA["fitz_counts"] = pd.DataFrame({
        "fitzpatrick": _levels,
        "n_images": [int((DF["_fitz"] == l).sum()) for l in _levels],
        "n_patients": [int(DF.loc[DF["_fitz"] == l, "_patient"].nunique()) for l in _levels],
    })
else:
    EDA["fitz_levels"], EDA["fitz_counts"] = [], None

if ROLES["biopsy"] is not None:
    _b = DF[ROLES["biopsy"]].astype(str).str.strip().str.upper()
    EDA["biopsy_by_class"] = pd.crosstab(DF["_label"], _b).reindex(CLASS_NAMES)
else:
    EDA["biopsy_by_class"] = None

if {"orig_width", "orig_height"}.issubset(IMAGE_MANIFEST.columns):
    EDA["image_size"] = IMAGE_MANIFEST[["orig_width", "orig_height"]].describe().loc[["min", "50%", "max"]]
else:
    EDA["image_size"] = None

EDA["class_counts"].to_csv(DIRS["logs"] / "eda_class_counts.csv", index=False)
EDA["missing_by_class"].to_csv(DIRS["logs"] / "eda_missing_by_class.csv")
print("Class distribution:")
display(EDA["class_counts"])
for _name in ("images_per_patient", "images_per_lesion", "lesions_per_patient"):
    _a = EDA[_name]
    print(f"{_name:<20s} min={_a.min()}  median={np.median(_a):g}  mean={_a.mean():.3f}  max={_a.max()}  "
          f"share with >1: {100.0 * (_a > 1).mean():.1f}%")
print(f"Patients with lesions of more than one class: {EDA['patients_multi_class']}")
print("\nMetadata missingness by class (fraction of images):")
display(EDA["missing_by_class"].round(3))
if EDA["fitz_counts"] is not None:
    print("Fitzpatrick skin type distribution:")
    display(EDA["fitz_counts"])
if EDA["biopsy_by_class"] is not None:
    print(f"Biopsy flag ('{ROLES['biopsy']}') by class (not used as a model input):")
    display(EDA["biopsy_by_class"])
if EDA["image_size"] is not None:
    print("Original image size in pixels (min / median / max):")
    display(EDA["image_size"])

## Part 4. Patient-grouped splits

**Purpose:** Create the outer 5-fold test splits and, inside every training fold, a validation split; assert that no patient is shared between the parts; save the assignments.

**Why:** Several images of the same lesion and several lesions of the same patient exist in PAD-UFES-20. If images of one patient fall on both sides of a split, the test estimate is optimistically biased (patient-level leakage). The validation part is needed for early stopping, temperature scaling and threshold selection, none of which may touch the test fold.

**Method choice:** `StratifiedGroupKFold` (scikit-learn) with the patient as the group (`CV_GROUPS`; patients that share an identical image, if any, are merged into one group in Part 3) keeps patients intact while approximately preserving class proportions. Grouping by patient also keeps all images of a lesion together, because a lesion belongs to one patient. The inner validation split applies the same splitter to the training fold and takes the first of `INNER_VAL_SPLITS` parts (one fifth by default), giving roughly 64% / 16% / 20% train / validation / test. For the optional leakage analysis (Part 13) a second, deliberately *ungrouped* scheme (`StratifiedKFold` on images) is generated. Assignments are saved once; on later runs they are regenerated and compared with the saved file, and any difference stops the notebook, because all models must use identical splits.

**Output:** `SPLITS` table and `get_split()`; file `splits/fold_assignments.csv` (one row per scheme × repeat × fold × image with its role) and `splits/fold_class_counts.csv`.

In [ ]:
SCHEMES: List[str] = ["grouped"] + (["image_level"] if CFG.RUN_LEAKAGE else [])


def make_splits(scheme: str, repeat: int) -> pd.DataFrame:
    """Return a long table (scheme, repeat, fold, img_id, role) for one scheme and one repetition.

    'grouped'     : outer and inner splits grouped by patient (the protocol used for all reported results).
    'image_level' : outer and inner splits on images, ignoring patients (leakage demonstration only).
    """
    seed = CFG.SEED + 1000 * repeat
    idx_all = np.arange(N)
    if scheme == "grouped":
        outer = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=seed)
        outer_iter = outer.split(idx_all, Y, groups=CV_GROUPS)
    elif scheme == "image_level":
        outer = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=seed)
        outer_iter = outer.split(idx_all, Y)
    else:
        raise ValueError(f"Unknown split scheme: {scheme}")
    rows = []
    for fold, (trainval_idx, test_idx) in enumerate(outer_iter):
        if scheme == "grouped":
            inner = StratifiedGroupKFold(n_splits=CFG.INNER_VAL_SPLITS, shuffle=True, random_state=seed + 1 + fold)
            tr_rel, va_rel = next(inner.split(trainval_idx, Y[trainval_idx], groups=CV_GROUPS[trainval_idx]))
        else:
            inner = StratifiedKFold(n_splits=CFG.INNER_VAL_SPLITS, shuffle=True, random_state=seed + 1 + fold)
            tr_rel, va_rel = next(inner.split(trainval_idx, Y[trainval_idx]))
        role = np.empty(N, dtype=object)
        role[trainval_idx[tr_rel]] = "train"
        role[trainval_idx[va_rel]] = "val"
        role[test_idx] = "test"
        rows.append(pd.DataFrame({"scheme": scheme, "repeat": repeat, "fold": fold, "img_id": IMG_IDS, "role": role}))
    return pd.concat(rows, ignore_index=True)


_generated = pd.concat([make_splits(s, r) for s in SCHEMES for r in range(CFG.N_REPEATS)], ignore_index=True)
SPLIT_CSV = DIRS["splits"] / "fold_assignments.csv"
if SPLIT_CSV.is_file():
    _saved = pd.read_csv(SPLIT_CSV, dtype={"img_id": str, "scheme": str, "role": str})
    _keys = ["scheme", "repeat", "fold", "img_id"]
    _merged = _generated.merge(_saved, on=_keys, how="left", suffixes=("", "_saved"))
    if _merged["role_saved"].isna().any():
        _new_rows = _merged["role_saved"].isna()
        # New schemes/repeats may be added later (e.g. N_REPEATS raised); existing rows must be unchanged.
        if not (_merged.loc[~_new_rows, "role"] == _merged.loc[~_new_rows, "role_saved"]).all():
            raise RuntimeError("Regenerated splits differ from splits/fold_assignments.csv (see fix below).")
        print(f"[info] {int(_new_rows.sum())} new assignment rows are added to the saved split file.")
        _generated.to_csv(SPLIT_CSV, index=False)
    elif not (_merged["role"] == _merged["role_saved"]).all():
        raise RuntimeError(
            "The regenerated splits differ from the saved file splits/fold_assignments.csv.\n"
            "Cause: the data, the seed, the fold settings or the scikit-learn version changed after runs were saved.\n"
            "Fix: restore the original settings; or, to start a new experiment, move the whole output folder away."
        )
    else:
        print("Saved split file found and identical to the regenerated splits (reproducibility check passed).")
else:
    _generated.to_csv(SPLIT_CSV, index=False)
SPLITS = _generated
_IMG_POS = pd.Series(np.arange(N), index=IMG_IDS)
_SPLIT_CACHE: Dict[Tuple[str, int, int], Tuple[np.ndarray, np.ndarray, np.ndarray]] = {}


def get_split(scheme: str, repeat: int, fold: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return sorted row positions (train_idx, val_idx, test_idx) into DF for one split."""
    key = (scheme, repeat, fold)
    if key not in _SPLIT_CACHE:
        sub = SPLITS[(SPLITS["scheme"] == scheme) & (SPLITS["repeat"] == repeat) & (SPLITS["fold"] == fold)]
        assert len(sub) == N, f"Split {key} does not cover all images."
        pos = _IMG_POS.loc[sub["img_id"].to_numpy()].to_numpy()
        roles = sub["role"].to_numpy()
        _SPLIT_CACHE[key] = tuple(np.sort(pos[roles == r]) for r in ("train", "val", "test"))
    return _SPLIT_CACHE[key]


_count_rows = []
for _s in SCHEMES:
    for _r in range(CFG.N_REPEATS):
        _test_cover = np.zeros(N, dtype=int)
        for _f in range(CFG.N_FOLDS):
            _tr, _va, _te = get_split(_s, _r, _f)
            assert len(set(_tr) | set(_va) | set(_te)) == N and len(_tr) + len(_va) + len(_te) == N, "Parts overlap or are incomplete."
            _test_cover[_te] += 1
            if _s == "grouped":
                _ptr, _pva, _pte = set(PATIENTS[_tr]), set(PATIENTS[_va]), set(PATIENTS[_te])
                assert not (_ptr & _pte), f"Patient overlap train/test in repeat {_r} fold {_f}"
                assert not (_pva & _pte), f"Patient overlap val/test in repeat {_r} fold {_f}"
                assert not (_ptr & _pva), f"Patient overlap train/val in repeat {_r} fold {_f}"
            for _part, _idx in (("train", _tr), ("val", _va), ("test", _te)):
                _cnt = np.bincount(Y[_idx], minlength=K)
                if (_cnt == 0).any():
                    raise AssertionError(
                        f"Class {[CLASS_NAMES[i] for i in np.where(_cnt == 0)[0]]} is absent from the {_part} part of "
                        f"{_s} repeat {_r} fold {_f}. Fix: change CFG.SEED (and report the seed used).")
                _count_rows.append(dict(scheme=_s, repeat=_r, fold=_f, part=_part, n_images=len(_idx),
                                        n_patients=int(len(set(PATIENTS[_idx]))),
                                        **{c: int(_cnt[i]) for i, c in enumerate(CLASS_NAMES)}))
        assert (_test_cover == 1).all(), "Every image must be in exactly one test fold per repeat."
FOLD_COUNTS = pd.DataFrame(_count_rows)
FOLD_COUNTS.to_csv(DIRS["splits"] / "fold_class_counts.csv", index=False)
print("Zero patient overlap between train, validation and test in every grouped fold: asserted.")
print("Every image is a test image exactly once per repeat: asserted.\n")
display(FOLD_COUNTS[(FOLD_COUNTS["scheme"] == "grouped") & (FOLD_COUNTS["repeat"] == 0)].drop(columns=["scheme", "repeat"]))
if CFG.RUN_LEAKAGE:
    _shared = [len(set(PATIENTS[get_split("image_level", 0, f)[0]]) & set(PATIENTS[get_split("image_level", 0, f)[2]]))
               for f in range(CFG.N_FOLDS)]
    print(f"For comparison, patients shared between train and test under the image-level scheme, per fold: {_shared}")

## Part 5. Preprocessing pipelines (fitted on training folds only)

**Purpose:** For every split, fit the metadata preprocessing (imputation with missingness indicators, scaling, one-hot encoding) on the **training part only**, save the fitted pipeline, and prepare the machinery for masking metadata fields (used in RQ2 and in modality-dropout training). Class weights for the loss are computed from the same training part.

**Why:** Imputation medians, scaling constants and category sets are estimated parameters. Estimating them on validation or test rows leaks information.

**Method choice:** A scikit-learn `ColumnTransformer` with three blocks: numeric fields → median imputation → standardisation; numeric fields → `MissingIndicator` (one binary indicator per field); categorical fields → constant imputation with an explicit `__MISSING__` category → one-hot encoding. An explicit missing category and indicators are used instead of silent imputation because, in clinical data, the fact that a value was not recorded can carry information, and because the fusion networks need a defined input when a field is absent. Every block is column-wise, so the transformed representation of "field *j* is missing" is the same vector for every patient. This allows a field to be masked directly in the transformed space, which is exactly equivalent to deleting the raw value and re-applying the pipeline; the equivalence is asserted below and again in the QC cell. Class weights are `n / (K · n_k)` computed on the training part (the "balanced" heuristic, as implemented in scikit-learn's `compute_class_weight`).

For the sensitivity analysis (Part 13) a restricted view `PREP_R` is derived from the same fitted pipelines: it keeps only the columns of the fields that are missing in at most `AUDIT_LOW_MISSING_MAX` of the images. No additional fitting takes place.

**Output:** `PREP[(scheme, repeat, fold)]` with the transformed matrix for all rows, the per-field column groups, the "missing" vector and the class weights; files `checkpoints/preprocessing/<scheme>_r<r>_f<f>.joblib`, `..._fit_ids.csv`, and `logs/class_weights.csv`.

In [ ]:
MISSING_CATEGORY = "__MISSING__"


def build_preprocessor() -> ColumnTransformer:
    """Unfitted metadata pipeline; all blocks are column-wise (required for exact field masking)."""
    blocks = []
    if NUM_COLS:
        blocks.append(("num", Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                                        ("scale", StandardScaler())]), NUM_COLS))
        blocks.append(("num_missing", MissingIndicator(features="all"), NUM_COLS))
    if CAT_COLS:
        blocks.append(("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=MISSING_CATEGORY)),
                                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT_COLS))
    return ColumnTransformer(blocks, remainder="drop", sparse_threshold=0.0)


def field_groups(pre: ColumnTransformer) -> List[np.ndarray]:
    """For each raw field (FIELD_NAMES order) return the indices of its columns in the transformed matrix."""
    n_num = len(NUM_COLS)
    groups: List[np.ndarray] = [np.array([j, n_num + j]) for j in range(n_num)]   # scaled value + indicator
    offset = 2 * n_num
    if CAT_COLS:
        for cats in pre.named_transformers_["cat"].named_steps["onehot"].categories_:
            groups.append(np.arange(offset, offset + len(cats)))
            offset += len(cats)
    return groups


def all_missing_frame(n_rows: int = 1) -> pd.DataFrame:
    """A raw metadata frame in which every field is missing."""
    data = {c: np.full(n_rows, np.nan, dtype=float) for c in NUM_COLS}
    data.update({c: np.array([np.nan] * n_rows, dtype=object) for c in CAT_COLS})
    return pd.DataFrame(data)[FIELD_NAMES]


def mask_raw(meta: pd.DataFrame, field_mask: np.ndarray) -> pd.DataFrame:
    """Set raw fields to missing where field_mask[i, j] is True (reference implementation for the equivalence test)."""
    out = meta.copy()
    for j, c in enumerate(FIELD_NAMES):
        col = out[c].to_numpy(copy=True)
        col[field_mask[:, j]] = np.nan
        out[c] = col
    return out


def apply_field_mask(X: np.ndarray, field_mask: np.ndarray, prep: Dict[str, Any]) -> np.ndarray:
    """Mask fields in the transformed space: columns of masked fields are replaced by the 'missing' vector."""
    assert field_mask.shape == (X.shape[0], N_FIELDS)
    col_mask = (field_mask.astype(np.float32) @ prep["G"]) > 0.5          # (n, D)
    return np.where(col_mask, prep["missvec"][None, :], X).astype(np.float32)


def ids_digest(ids: Sequence[str]) -> str:
    """Order-independent fingerprint of a set of image ids."""
    return hashlib.sha256("|".join(sorted(map(str, ids))).encode()).hexdigest()[:16]


PREP: Dict[Tuple[str, int, int], Dict[str, Any]] = {}
_cw_rows = []
for _s in SCHEMES:
    for _r in range(CFG.N_REPEATS):
        for _f in range(CFG.N_FOLDS):
            _tr, _va, _te = get_split(_s, _r, _f)
            _pre = build_preprocessor().fit(META.iloc[_tr])                 # training rows only
            _X = _pre.transform(META).astype(np.float32)
            _groups = field_groups(_pre)
            assert sum(len(g) for g in _groups) == _X.shape[1], "Field groups do not cover the transformed columns."
            _G = np.zeros((N_FIELDS, _X.shape[1]), dtype=np.float32)
            for _j, _g in enumerate(_groups):
                _G[_j, _g] = 1.0
            _missvec = _pre.transform(all_missing_frame(1)).astype(np.float32)[0]
            _counts = np.bincount(Y[_tr], minlength=K).astype(float)
            _cw = len(_tr) / (K * _counts)
            _prep = dict(pipeline=_pre, X=_X, groups=_groups, G=_G, missvec=_missvec, class_weights=_cw.astype(np.float32),
                         fit_ids_digest=ids_digest(IMG_IDS[_tr]), n_fit=len(_tr))
            # Equivalence test: masking in the transformed space == masking the raw table and re-transforming.
            _rng = np.random.default_rng([CFG.SEED, _r, _f, 7])
            _rows = _rng.choice(N, size=min(64, N), replace=False)
            _fm = _rng.random((len(_rows), N_FIELDS)) < 0.5
            _ref = _pre.transform(mask_raw(META.iloc[_rows], _fm)).astype(np.float32)
            assert np.allclose(apply_field_mask(_X[_rows], _fm, _prep), _ref, atol=1e-6), \
                "Transformed-space masking is not equivalent to raw masking; the preprocessing is not column-wise."
            _base = DIRS["preprocessing"] / f"{_s}_r{_r}_f{_f}"
            joblib.dump(_pre, str(_base) + ".joblib")
            pd.DataFrame({"img_id": IMG_IDS[_tr]}).to_csv(str(_base) + "_fit_ids.csv", index=False)
            PREP[(_s, _r, _f)] = _prep
            _cw_rows.append(dict(scheme=_s, repeat=_r, fold=_f, n_train=len(_tr),
                                 **{f"weight_{c}": float(_cw[i]) for i, c in enumerate(CLASS_NAMES)}))
CLASS_WEIGHTS = pd.DataFrame(_cw_rows)
CLASS_WEIGHTS.to_csv(DIRS["logs"] / "class_weights.csv", index=False)
META_DIM = {k: v["X"].shape[1] for k, v in PREP.items()}

# ---- restricted metadata for the sensitivity analysis: only fields that are (almost) never missing, so that
# the pattern of missing fields cannot act as a proxy for the diagnosis. The restricted matrix is a column
# subset of the same training-fold-fitted pipeline output; nothing is refitted.
LOW_MISSING_FIELDS: List[str] = [f for f, r in zip(FIELD_NAMES, MISSING_MATRIX.mean(axis=0)) if r <= CFG.AUDIT_LOW_MISSING_MAX]
LOW_MISSING_IDX = [FIELD_NAMES.index(f) for f in LOW_MISSING_FIELDS]
PREP_R: Dict[Tuple[str, int, int], Dict[str, Any]] = {}
if LOW_MISSING_FIELDS:
    for _key, _p in PREP.items():
        _cols = np.concatenate([_p["groups"][j] for j in LOW_MISSING_IDX])
        _new_groups, _off = [], 0
        for _j in LOW_MISSING_IDX:
            _new_groups.append(np.arange(_off, _off + len(_p["groups"][_j])))
            _off += len(_p["groups"][_j])
        PREP_R[_key] = dict(X=np.ascontiguousarray(_p["X"][:, _cols]), groups=_new_groups, G=np.ascontiguousarray(_p["G"][:, _cols]),
                            missvec=_p["missvec"][_cols], class_weights=_p["class_weights"], fields=list(LOW_MISSING_FIELDS))
print(f"Low-missingness fields (missing in at most {100 * CFG.AUDIT_LOW_MISSING_MAX:.0f}% of images), used by the restricted-metadata "
      f"sensitivity models: {LOW_MISSING_FIELDS if LOW_MISSING_FIELDS else 'none'}")
print(f"Fitted {len(PREP)} preprocessing pipelines on training rows only; raw-vs-transformed masking equivalence asserted.")
print(f"Transformed metadata dimension per split: min {min(META_DIM.values())}, max {max(META_DIM.values())}")
print("\nClass weights for the weighted cross-entropy (grouped scheme, repeat 0):")
display(CLASS_WEIGHTS[(CLASS_WEIGHTS["scheme"] == "grouped") & (CLASS_WEIGHTS["repeat"] == 0)].drop(columns=["scheme", "repeat"]).round(4))

## Part 6. Models and training

**Purpose:** Shared infrastructure for all model runs: run bookkeeping (done-detection, configuration-hash check), the standard prediction file format, the test-time metadata masking conditions for RQ2, and the masking distribution used for modality-dropout training.

**Why:** Every model must be evaluated on identical test rows under identical masks, and every result must be reproducible from saved prediction files without retraining (Part 7). Completed runs must be skipped after a Colab disconnect, but never reused if the result-defining configuration changed.

**Method choice:** *Test-time missingness (RQ2).* (i) MCAR: each metadata field of each test image is independently set to missing with probability *p* ∈ `MCAR_RATES`, on top of the missingness already present; each intermediate rate is repeated with `N_MASK_SEEDS` seeds. (ii) Structured scenarios from `CFG.SCENARIOS`, defined on column roles. Masks depend only on (seed, repeat, fold, rate, mask seed), so all models see exactly the same masks. *Modality dropout (training).* Following the idea of ModDrop (Neverova et al., *IEEE TPAMI*, 2016), a training sample loses all metadata with probability `MD_P_FULL`; otherwise, with probability `MD_P_PARTIAL`, each field is dropped with a per-sample rate drawn from U(0, 1). The mixture exposes the model to the whole range of missingness levels that RQ2 evaluates, rather than to one fixed rate.

**Output:** Helper functions; `ROBUST_CONDITIONS` table saved to `logs/robust_conditions.csv`; `SCENARIO_FIELDS`.

In [ ]:
MODEL_KIND = {  # model name -> (family, uses modality dropout)
    "image_only": ("image_only", False), "meta_gbm": ("meta_gbm", False), "meta_gbm_md": ("meta_gbm", True),
    "early_concat": ("early_concat", False), "early_concat_md": ("early_concat", True),
    "late_fusion": ("late_fusion", False), "late_fusion_md": ("late_fusion", True),
    "cross_attn": ("cross_attn", False), "cross_attn_md": ("cross_attn", True),
    "metablock": ("metablock", False), "metablock_md": ("metablock", True),
}
DEEP_FAMILIES = ("image_only", "early_concat", "cross_attn", "metablock")
MODELS: List[str] = [m for m in MODEL_KIND if m in CFG.MODELS_TO_RUN]          # fixed canonical order

# Sensitivity models ("_r" = restricted metadata): the same architectures and settings, but the metadata input
# is limited to the low-missingness fields (PREP_R). They are analysed separately (Part 13) and are not part of MODELS.
RESTRICTED_MODELS = {"meta_gbm_r": "meta_gbm", "early_concat_r": "early_concat", "late_fusion_r": "late_fusion",
                     "cross_attn_r": "cross_attn", "metablock_r": "metablock"}
for _name, _base in RESTRICTED_MODELS.items():
    MODEL_KIND[_name] = MODEL_KIND[_base]
SENS_MODELS: List[str] = []
if CFG.RUN_RESTRICTED_SENSITIVITY and LOW_MISSING_FIELDS and "meta_gbm" in MODELS:
    SENS_MODELS = [n for n, b in RESTRICTED_MODELS.items() if b in MODELS]
elif CFG.RUN_RESTRICTED_SENSITIVITY:
    print("[info] restricted-metadata sensitivity analysis not possible: it needs meta_gbm in MODELS_TO_RUN and at least one low-missingness field.")


def get_prep(model: str, scheme: str, repeat: int, fold: int) -> Dict[str, Any]:
    """Preprocessed metadata for a run: the restricted field set for '_r' models, all fields otherwise."""
    return PREP_R[(scheme, repeat, fold)] if model in RESTRICTED_MODELS else PREP[(scheme, repeat, fold)]


DEEP_MODELS = [m for m in MODELS if MODEL_KIND[m][0] in DEEP_FAMILIES]
GBM_MODELS = [m for m in MODELS if MODEL_KIND[m][0] == "meta_gbm"]
LATE_MODELS = [m for m in MODELS if MODEL_KIND[m][0] == "late_fusion"]
USES_METADATA = {m: MODEL_KIND[m][0] != "image_only" for m in MODEL_KIND}
SENS_DEEP = [m for m in SENS_MODELS if MODEL_KIND[m][0] in DEEP_FAMILIES]
SENS_GBM = [m for m in SENS_MODELS if MODEL_KIND[m][0] == "meta_gbm"]
SENS_LATE = [m for m in SENS_MODELS if MODEL_KIND[m][0] == "late_fusion"]


def needs_robust(scheme: str, model: str) -> bool:
    """Masked-test predictions (RQ2) are stored for the main metadata-using models under the grouped scheme only."""
    return scheme == "grouped" and USES_METADATA[model] and model not in RESTRICTED_MODELS


def run_seed(repeat: int, fold: int) -> int:
    """Deterministic seed of one training run."""
    return int(CFG.SEED + 1000 * repeat + fold)


def run_files(scheme: str, model: str, repeat: int, fold: int) -> Dict[str, Path]:
    """All file paths belonging to one (scheme, model, repeat, fold) run."""
    pred_dir = DIRS["predictions"] / scheme / model
    ckpt_dir = DIRS["checkpoints"] / scheme / model
    pred_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    stem = f"r{repeat}_f{fold}"
    return dict(val=pred_dir / f"{stem}_val.csv", test=pred_dir / f"{stem}_test.csv",
                robust=pred_dir / f"{stem}_robust.npz", shap=pred_dir / f"{stem}_shap.npz",
                done=pred_dir / f"{stem}_DONE.json",
                last=ckpt_dir / f"{stem}_last.pt", best=ckpt_dir / f"{stem}_best.pt")


def is_done(scheme: str, model: str, repeat: int, fold: int) -> bool:
    """True if the run finished under the current configuration; raises if it finished under another one."""
    fp = run_files(scheme, model, repeat, fold)
    if not fp["done"].is_file():
        return False
    rec = load_json(fp["done"])
    if rec.get("cfg_hash") != CFG_HASH:
        raise RuntimeError(
            f"Run {scheme}/{model}/r{repeat}_f{fold} was produced with configuration hash {rec.get('cfg_hash')}, "
            f"but the current hash is {CFG_HASH}.\nA result-defining setting changed after runs were saved.\n"
            "Fix: restore the original configuration (see logs/config.json), or move the output folder away "
            "to start a clean experiment. Runs from two configurations are never mixed."
        )
    needed = [fp["val"], fp["test"]] + ([fp["robust"]] if needs_robust(scheme, model) else [])
    return all(p.is_file() for p in needed)


def log_softmax_np(logits: np.ndarray) -> np.ndarray:
    """Numerically stable log-softmax along the last axis."""
    z = logits - logits.max(axis=-1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))


def softmax_np(logits: np.ndarray) -> np.ndarray:
    """Softmax along the last axis."""
    return np.exp(log_softmax_np(logits))


def save_predictions(path: Path, idx: np.ndarray, logits: np.ndarray, scheme: str, model: str,
                     repeat: int, fold: int, seed: int) -> None:
    """Write one prediction file (one row per image) in the standard format used by all analyses."""
    assert logits.shape == (len(idx), K) and np.isfinite(logits).all(), "Invalid logits."
    probs = softmax_np(logits.astype(np.float64))
    out = pd.DataFrame({"img_id": IMG_IDS[idx], "patient_id": PATIENTS[idx], "lesion_id": DF["_lesion"].to_numpy()[idx],
                        "scheme": scheme, "model": model, "repeat": repeat, "fold": fold, "seed": seed,
                        "y_true": Y[idx], "label": [CLASS_NAMES[i] for i in Y[idx]]})
    for k, c in enumerate(CLASS_NAMES):
        out[f"logit_{c}"] = logits[:, k].astype(np.float64)
    for k, c in enumerate(CLASS_NAMES):
        out[f"prob_{c}"] = probs[:, k]
    tmp = path.with_name(path.name + ".tmp")
    out.to_csv(tmp, index=False, float_format="%.8g")
    os.replace(tmp, path)


def balanced_accuracy_fast(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> float:
    """Mean per-class recall over the classes present in y_true (validated against scikit-learn in Part 8)."""
    cm = np.bincount(y_true * n_classes + y_pred, minlength=n_classes * n_classes).reshape(n_classes, n_classes)
    support = cm.sum(axis=1)
    present = support > 0
    return float((np.diag(cm)[present] / support[present]).mean()) if present.any() else float("nan")


# ----------------------------------------------------------------------------- RQ2 masking conditions
SCENARIO_FIELDS: Dict[str, List[str]] = {}
for _name, _spec in CFG.SCENARIOS.items():
    assert ("mask_roles" in _spec) != ("keep_roles" in _spec), f"Scenario {_name}: give exactly one of mask_roles / keep_roles."
    if "mask_roles" in _spec:
        _masked = [f for f in FIELD_NAMES if FIELD_ROLE_MAP.get(f) in _spec["mask_roles"]]
        _unresolved = [r for r in _spec["mask_roles"] if not any(FIELD_ROLE_MAP.get(f) == r for f in FIELD_NAMES)]
    else:
        _masked = [f for f in FIELD_NAMES if FIELD_ROLE_MAP.get(f) not in _spec["keep_roles"]]
        _unresolved = [r for r in _spec["keep_roles"] if not any(FIELD_ROLE_MAP.get(f) == r for f in FIELD_NAMES)]
    if _unresolved or not _masked:
        print(f"[warning] scenario '{_name}' skipped: roles {_unresolved} have no input field in this file.")
        continue
    SCENARIO_FIELDS[_name] = _masked
    print(f"Scenario '{_name}': {len(_masked)} of {N_FIELDS} fields set to missing -> {_masked}")

_conds = []
for _rate in CFG.MCAR_RATES:
    _n_seeds = 1 if (_rate <= 0.0 or _rate >= 1.0) else CFG.N_MASK_SEEDS     # 0% and 100% are deterministic
    for _ms in range(_n_seeds):
        _conds.append(dict(cond_id=f"mcar_{_rate:.2f}_s{_ms}", kind="mcar", name="", rate=float(_rate), mask_seed=_ms))
for _name in SCENARIO_FIELDS:
    _conds.append(dict(cond_id=f"scenario_{_name}", kind="scenario", name=_name, rate=float("nan"), mask_seed=0))
ROBUST_CONDITIONS = pd.DataFrame(_conds)
ROBUST_CONDITIONS.to_csv(DIRS["logs"] / "robust_conditions.csv", index=False)
COND_IDS: List[str] = ROBUST_CONDITIONS["cond_id"].tolist()


def condition_mask(cond: Dict[str, Any], repeat: int, fold: int, n_rows: int) -> np.ndarray:
    """Boolean (n_rows, N_FIELDS) mask of fields set to missing under one test-time condition."""
    if cond["kind"] == "mcar":
        rng = np.random.default_rng([CFG.SEED, repeat, fold, int(round(cond["rate"] * 1000)), int(cond["mask_seed"]), 11])
        return rng.random((n_rows, N_FIELDS)) < cond["rate"]
    cols = np.isin(np.array(FIELD_NAMES), SCENARIO_FIELDS[cond["name"]])
    return np.tile(cols[None, :], (n_rows, 1))


def sample_md_mask(n_rows: int, rng: np.random.Generator) -> np.ndarray:
    """Draw modality-dropout field masks (NumPy version; the GPU version in the training loop uses the same law)."""
    full = rng.random(n_rows) < CFG.MD_P_FULL
    partial = (~full) & (rng.random(n_rows) < CFG.MD_P_PARTIAL)
    rates = rng.random(n_rows)
    per_field = rng.random((n_rows, N_FIELDS)) < rates[:, None]
    return full[:, None] | (partial[:, None] & per_field)


EPOCH_LOG_CSV = DIRS["logs"] / "epoch_log.csv"


def append_csv_row(path: Path, row: Dict[str, Any]) -> None:
    """Append one record to a CSV log (the header is written when the file is created)."""
    pd.DataFrame([row]).to_csv(path, mode="a", header=not Path(path).is_file(), index=False)


print(f"\nModels in this experiment: {MODELS}")
print(f"Restricted-metadata sensitivity models: {SENS_MODELS if SENS_MODELS else 'none'}")
print(f"Test-time conditions for RQ2: {len(COND_IDS)} ({(ROBUST_CONDITIONS['kind'] == 'mcar').sum()} MCAR, {len(SCENARIO_FIELDS)} structured)")

**Purpose:** Train the metadata-only baseline (gradient boosting) and its missingness-augmented variant for every split; save validation, test and masked-test predictions and the SHAP contributions.

**Why:** The metadata-only model is the baseline that shows how much diagnostic signal the clinical fields carry without the image, and it is the metadata branch of late fusion.

**Method choice:** LightGBM (Ke et al., *NeurIPS*, 2017): gradient-boosted trees are a strong default for small tabular data and train in seconds on CPU. The same fold-wise preprocessing as for the networks is used so that all models share one definition of "missing". Class imbalance is handled with the same class weights as the networks (as sample weights). The number of trees is chosen by early stopping on the validation part (multi-class log-loss); the test fold is not used. No hyperparameter search is run for any model in this study, which keeps the comparison fair in tuning effort but means results reflect fixed, reasonable settings rather than tuned optima (a limitation to state in the paper). *Modality-dropout variant (`meta_gbm_md`):* trees cannot use stochastic dropout during optimisation, so the training set is augmented with `GBM_MD_N_AUG` masked copies drawn from the modality-dropout distribution (and the early-stopping set likewise); this is the closest tabular equivalent and is described as "missingness augmentation" in the outputs. SHAP values are exact TreeSHAP contributions (Lundberg et al., *Nature Machine Intelligence*, 2020) obtained from LightGBM's built-in `pred_contrib`, which avoids a dependency on the `shap` package.

**Output:** `predictions/<scheme>/meta_gbm[_md]/r<r>_f<f>_{val,test}.csv`, `_robust.npz` (masked-test logits and masked-validation logits), `_shap.npz` (per-field contributions on the test fold), `_DONE.json`.

In [ ]:
import inspect


def fit_lgbm(X_tr: np.ndarray, y_tr: np.ndarray, X_es: np.ndarray, y_es: np.ndarray, class_weights: np.ndarray, seed: int):
    """Fit one class-weighted multi-class LightGBM model with early stopping on the given validation data."""
    cw = np.asarray(class_weights, dtype=np.float64)
    clf = lgb.LGBMClassifier(objective="multiclass", random_state=seed, n_jobs=min(2, os.cpu_count() or 1), verbose=-1,
                             deterministic=True, force_col_wise=True, **CFG.GBM_PARAMS)
    # LightGBM renamed the validation-set arguments (eval_set -> eval_X / eval_y); support both spellings.
    if "eval_X" in inspect.signature(clf.fit).parameters:
        eval_kwargs = dict(eval_X=(X_es,), eval_y=(y_es,))
    else:
        eval_kwargs = dict(eval_set=[(X_es, y_es)])
    clf.fit(X_tr, y_tr, sample_weight=cw[y_tr], eval_sample_weight=[cw[y_es]], eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(CFG.GBM_EARLY_STOPPING_ROUNDS, verbose=False)], **eval_kwargs)
    assert list(clf.classes_) == list(range(K)), "LightGBM did not see all classes in the training part."
    return clf


def run_gbm(model: str, scheme: str, repeat: int, fold: int) -> None:
    """Train one LightGBM metadata model and save all its predictions."""
    fp = run_files(scheme, model, repeat, fold)
    use_md = MODEL_KIND[model][1]
    prep = get_prep(model, scheme, repeat, fold)
    tr, va, te = get_split(scheme, repeat, fold)
    seed = run_seed(repeat, fold)
    t0 = time.time()
    rng = np.random.default_rng([seed, 21])
    X_tr, y_tr = prep["X"][tr], Y[tr]
    X_es, y_es = prep["X"][va], Y[va]
    # Masked copies of the validation part (used for early stopping of the *_md variant and for the
    # late-fusion weight of late_fusion_md). Validation rows only - never test rows.
    val_aug_X = np.stack([apply_field_mask(prep["X"][va], sample_md_mask(len(va), np.random.default_rng([seed, 99, a])), prep)
                          for a in range(CFG.GBM_MD_N_AUG)])
    if use_md:
        X_tr = np.concatenate([X_tr] + [apply_field_mask(prep["X"][tr], sample_md_mask(len(tr), rng), prep)
                                        for _ in range(CFG.GBM_MD_N_AUG)])
        y_tr = np.tile(Y[tr], CFG.GBM_MD_N_AUG + 1)
        X_es = np.concatenate([X_es] + list(val_aug_X))
        y_es = np.tile(Y[va], CFG.GBM_MD_N_AUG + 1)
    clf = fit_lgbm(X_tr, y_tr, X_es, y_es, prep["class_weights"], seed)

    def logp(X: np.ndarray) -> np.ndarray:
        """Log-probabilities (used as logits by all later analyses, including temperature scaling)."""
        return np.log(np.clip(clf.predict_proba(X), 1e-12, 1.0))

    save_predictions(fp["val"], va, logp(prep["X"][va]), scheme, model, repeat, fold, seed)
    save_predictions(fp["test"], te, logp(prep["X"][te]), scheme, model, repeat, fold, seed)
    if needs_robust(scheme, model):
        rob = np.stack([logp(apply_field_mask(prep["X"][te], condition_mask(c, repeat, fold, len(te)), prep))
                        for c in ROBUST_CONDITIONS.to_dict("records")]).astype(np.float32)
        val_aug = np.stack([logp(x) for x in val_aug_X]).astype(np.float32)
        np.savez_compressed(fp["robust"], cond_ids=np.array(COND_IDS), logits=rob, val_aug_logits=val_aug)
        if not use_md:
            contrib = clf.booster_.predict(prep["X"][te], pred_contrib=True, num_iteration=clf.best_iteration_)
            D = prep["X"].shape[1]
            contrib = np.asarray(contrib).reshape(len(te), K, D + 1)[:, :, :D]      # drop the bias term
            per_field = np.stack([contrib[:, :, g].sum(axis=2) for g in prep["groups"]], axis=2)   # (n, K, N_FIELDS)
            np.savez_compressed(fp["shap"], shap=per_field.astype(np.float32), img_id=IMG_IDS[te], fields=np.array(FIELD_NAMES))
    val_pred = clf.predict_proba(prep["X"][va]).argmax(1)
    rec = dict(scheme=scheme, model=model, repeat=repeat, fold=fold, seed=seed, cfg_hash=CFG_HASH,
               best_iteration=int(clf.best_iteration_ or CFG.GBM_PARAMS["n_estimators"]),
               val_balanced_accuracy=balanced_accuracy_fast(Y[va], val_pred, K),
               n_train=len(tr), n_val=len(va), n_test=len(te), seconds=round(time.time() - t0, 1),
               val_ids_digest=ids_digest(IMG_IDS[va]), finished_utc=dt.datetime.now(dt.timezone.utc).isoformat())
    save_json(rec, fp["done"])


for _s in SCHEMES:
    for _m in GBM_MODELS + (SENS_GBM if _s == "grouped" else []):
        if _s == "image_level" and _m not in CFG.LEAKAGE_MODELS:
            continue
        for _r in range(CFG.N_REPEATS):
            for _f in range(CFG.N_FOLDS):
                if is_done(_s, _m, _r, _f):
                    continue
                run_gbm(_m, _s, _r, _f)
        print(f"[done] {_s}/{_m}")

**Purpose:** Dataset class, augmentation and data loaders for the image models.

**Why:** Training needs random augmentation to limit over-fitting on about 1,500 training images per fold; evaluation needs a deterministic view.

**Method choice:** Training view: random resized crop to `IMG_SIZE` (scale 0.6–1.0), horizontal and vertical flips (lesion photographs have no canonical orientation) and mild colour jitter. Colour jitter is kept mild because lesion colour is diagnostically relevant. Evaluation view: a plain resize of the cached image to `IMG_SIZE` with no cropping, so that no part of the lesion is discarded; no test-time augmentation. Normalisation uses the mean and standard deviation stored with the pretrained weights and is done on the GPU. The data-loader shuffling generator is re-seeded from (run seed, epoch) at the start of every epoch, so an interrupted run that resumes at epoch *e* sees the same batches as an uninterrupted run.

**Output:** `LesionDataset`, `make_loader()`.

In [ ]:
class LesionDataset(Dataset):
    """Images from the in-memory cache plus the fold-specific transformed metadata."""

    def __init__(self, idx: np.ndarray, meta_X: np.ndarray, train: bool) -> None:
        self.idx = np.asarray(idx)
        self.meta = torch.from_numpy(np.ascontiguousarray(meta_X[self.idx])).float()
        self.y = torch.from_numpy(Y[self.idx]).long()
        if train:
            self.tf = T.Compose([
                T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.6, 1.0), ratio=(0.8, 1.25), antialias=True),
                T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
            ])
        else:
            self.tf = T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE), antialias=True)

    def __len__(self) -> int:
        return len(self.idx)

    def __getitem__(self, i: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        img = torch.from_numpy(np.array(IMAGES[self.idx[i]], copy=True)).permute(2, 0, 1)   # uint8, C x H x W
        return self.tf(img), self.meta[i], self.y[i]


def _worker_init(worker_id: int) -> None:
    """Give every data-loader worker its own reproducible NumPy / Python seed."""
    s = torch.initial_seed() % (2**32)
    np.random.seed(s)
    random.seed(s)


def make_loader(idx: np.ndarray, meta_X: np.ndarray, train: bool, generator: Optional[torch.Generator] = None) -> DataLoader:
    """Build a DataLoader; training loaders shuffle and drop the last incomplete batch (BatchNorm needs > 1 sample)."""
    ds = LesionDataset(idx, meta_X, train)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE if train else 2 * CFG.BATCH_SIZE, shuffle=train, drop_last=train,
                      num_workers=CFG.NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                      # "fork" lets workers see the dataset class and image cache defined in this notebook; newer
                      # Python versions default to a start method that cannot import notebook-defined classes.
                      multiprocessing_context=("fork" if CFG.NUM_WORKERS > 0 else None),
                      worker_init_fn=_worker_init, generator=generator)

**Purpose:** Define the image backbone wrapper and the four network heads: image-only, early concatenation, cross-attention fusion and MetaBlock-style gated fusion.

**Why:** RQ1 compares fusion strategies; to attribute differences to the fusion mechanism, all networks share the same backbone, the same pooled-feature definition, the same optimiser and the same training budget.

**Method choice:**
- *Backbone.* ConvNeXt-Tiny by default (Liu et al., *CVPR*, 2022), with EfficientNet-B0 (Tan & Le, *ICML*, 2019) and ViT-S/14 DINOv2 (Oquab et al., *TMLR*, 2024) selectable; all from `timm` with pretrained weights. The backbone returns a token sequence (spatial positions for CNNs, patch tokens for ViT), so that all heads work with every backbone. The backbone is fine-tuned end to end with a smaller learning rate than the new layers.
- *Early concatenation.* Pooled image features are concatenated with an MLP embedding of the metadata and classified by an MLP: the standard feature-level fusion baseline.
- *Cross-attention.* Each metadata field becomes one token (a field-specific linear embedding); the field tokens attend over the image tokens with multi-head attention (Vaswani et al., *NeurIPS*, 2017), and the attended field representation is concatenated with the pooled image representation. This lets each clinical field select the image regions it is combined with.
- *MetaBlock-style gating.* After Pacheco & Krohling (*IEEE JBHI*, 2021, 25(9), 3554–3563, DOI 10.1109/JBHI.2021.3062002): the metadata vector produces two channel-wise vectors, f(m) and g(m), and the image feature map V is transformed as σ(tanh(V ⊙ f(m)) + g(m)) before pooling and classification. **Stated deviations / items to verify against the paper before submission:** (i) the gating formula is implemented from my reading of the paper and its public code; (ii) f and g are plain linear layers **without batch normalisation**, initialised so that the gate starts as f = 1, g = 0. A batch-normalised f is centred on zero across the batch, so the sign applied to each image channel varies from patient to patient with the metadata; in the test runs of this pipeline that version could not use the image at all (with all metadata removed it stayed at chance level even after modality-dropout training, while the image-only network on the same data was nearly perfect), and the identity-initialised version removed the problem; (iii) the image tokens are layer-normalised before the gate, because the token features returned by the backbones are unnormalised; (iv) the original inserts a feature-reducer layer before the classifier and used other backbones and training settings, whereas the shared classifier head of this study is used. Because of these differences the model is named "MetaBlock-style" in all outputs and must not be described as a reproduction of MetaBlock.
- *Late fusion* is not a network: it combines the predictions of the image-only network and the metadata model (cell after training).

**Output:** `build_model()` returning a `FusionNet`.

In [ ]:
class Backbone(nn.Module):
    """timm feature extractor that returns tokens of shape (batch, n_tokens, channels) for CNNs and ViTs."""

    def __init__(self, name: str, pretrained: bool, img_size: int) -> None:
        super().__init__()
        kwargs: Dict[str, Any] = dict(pretrained=pretrained, num_classes=0)
        if name.startswith("vit"):
            kwargs["img_size"] = img_size          # ViT position embeddings are resampled to the training resolution
        self.net = timm.create_model(name, **kwargs)
        self.dim = int(self.net.num_features)
        self.n_prefix = int(getattr(self.net, "num_prefix_tokens", 0))
        cfg = timm.data.resolve_model_data_config(self.net)
        self.register_buffer("mean", torch.tensor(cfg["mean"]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(cfg["std"]).view(1, 3, 1, 1), persistent=False)

    def forward(self, x_uint8: torch.Tensor) -> torch.Tensor:
        x = (x_uint8.float() / 255.0 - self.mean) / self.std
        feats = self.net.forward_features(x)
        if feats.ndim == 4:                        # CNN: (B, C, H, W) -> (B, H*W, C)
            feats = feats.flatten(2).transpose(1, 2)
        return feats


class FieldTokenizer(nn.Module):
    """Embed every metadata field (its group of transformed columns) into one d-dimensional token."""

    def __init__(self, groups: List[np.ndarray], meta_dim: int, d: int) -> None:
        super().__init__()
        max_len = max(len(g) for g in groups)
        idx = torch.full((len(groups), max_len), meta_dim, dtype=torch.long)     # meta_dim = index of a zero pad column
        for j, g in enumerate(groups):
            idx[j, : len(g)] = torch.as_tensor(g, dtype=torch.long)
        self.register_buffer("idx", idx)
        self.weight = nn.Parameter(torch.randn(len(groups), max_len, d) * (1.0 / math.sqrt(max_len)))
        self.bias = nn.Parameter(torch.zeros(len(groups), d))

    def forward(self, meta: torch.Tensor) -> torch.Tensor:
        padded = torch.cat([meta, meta.new_zeros(meta.shape[0], 1)], dim=1)
        grouped = padded[:, self.idx]                                            # (B, n_fields, max_len)
        return torch.einsum("bfm,fmd->bfd", grouped, self.weight) + self.bias


class FusionNet(nn.Module):
    """Image backbone plus one of the heads: image_only | early_concat | cross_attn | metablock."""

    def __init__(self, family: str, backbone: Backbone, meta_dim: int, groups: List[np.ndarray], n_classes: int) -> None:
        super().__init__()
        assert family in DEEP_FAMILIES
        self.family, self.backbone = family, backbone
        C, H, d, p = backbone.dim, CFG.META_HIDDEN, CFG.FUSION_DIM, CFG.DROPOUT
        if family == "image_only":
            self.pool_norm = nn.LayerNorm(C)
            self.head = nn.Sequential(nn.Dropout(p), nn.Linear(C, n_classes))
        elif family == "early_concat":
            self.pool_norm = nn.LayerNorm(C)
            self.meta_mlp = nn.Sequential(nn.Linear(meta_dim, H), nn.ReLU(), nn.Dropout(p), nn.Linear(H, H), nn.ReLU())
            self.head = nn.Sequential(nn.Dropout(p), nn.Linear(C + H, d), nn.ReLU(), nn.Dropout(p), nn.Linear(d, n_classes))
        elif family == "metablock":
            # The backbone's token features are unnormalised (timm applies the final norm inside its own head) and
            # can be large; without this normalisation tanh saturates and the gate passes almost no image information.
            self.token_norm = nn.LayerNorm(C)
            self.fb = nn.Linear(meta_dim, C)
            self.gb = nn.Linear(meta_dim, C)
            # Start as a near-identity gate (f = 1, g = 0) so that the pretrained image features pass through
            # from the first step and the metadata modulation is learned on top of them.
            nn.init.normal_(self.fb.weight, std=0.01)
            nn.init.ones_(self.fb.bias)
            nn.init.zeros_(self.gb.weight)
            nn.init.zeros_(self.gb.bias)
            self.head = nn.Sequential(nn.Dropout(p), nn.Linear(C, n_classes))
        elif family == "cross_attn":
            self.img_proj = nn.Sequential(nn.Linear(C, d), nn.LayerNorm(d))
            self.tokenizer = FieldTokenizer(groups, meta_dim, d)
            self.q_norm = nn.LayerNorm(d)
            self.attn = nn.MultiheadAttention(d, CFG.XATTN_HEADS, dropout=0.1, batch_first=True)
            self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
            self.ffn = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
            self.drop = nn.Dropout(0.1)
            self.head = nn.Sequential(nn.Dropout(p), nn.Linear(2 * d, d), nn.ReLU(), nn.Dropout(p), nn.Linear(d, n_classes))

    def forward_from_tokens(self, tokens: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:
        """Classify from backbone tokens; separated from the backbone so that tokens can be cached (RQ2) and hooked (Grad-CAM)."""
        if self.family == "image_only":
            return self.head(self.pool_norm(tokens.mean(dim=1)))
        if self.family == "early_concat":
            return self.head(torch.cat([self.pool_norm(tokens.mean(dim=1)), self.meta_mlp(meta)], dim=1))
        if self.family == "metablock":
            feats = self.token_norm(tokens)
            gated = torch.sigmoid(torch.tanh(feats * self.fb(meta).unsqueeze(1)) + self.gb(meta).unsqueeze(1))
            return self.head(gated.mean(dim=1))
        img = self.img_proj(tokens)
        q = self.q_norm(self.tokenizer(meta))
        attended, _ = self.attn(q, img, img, need_weights=False)
        q = self.norm1(q + self.drop(attended))
        q = self.norm2(q + self.drop(self.ffn(q)))
        return self.head(torch.cat([img.mean(dim=1), q.mean(dim=1)], dim=1))

    def forward(self, x_uint8: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:
        return self.forward_from_tokens(self.backbone(x_uint8), meta)


def build_model(family: str, prep: Dict[str, Any]) -> FusionNet:
    """Create a FusionNet for one split (the metadata dimension and field groups are split-specific)."""
    backbone = Backbone(CFG.BACKBONE_REGISTRY[CFG.BACKBONE], CFG.PRETRAINED, CFG.IMG_SIZE)
    return FusionNet(family, backbone, prep["X"].shape[1], prep["groups"], K)


def count_parameters(model: nn.Module) -> int:
    """Number of trainable parameters."""
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

**Purpose:** The training engine: optimiser and schedule, mixed precision, weighted loss, modality dropout, validation, early stopping, per-epoch checkpointing with automatic resume, and prediction on validation, test and masked-test data.

**Why:** Colab sessions disconnect; a run must continue from its last completed epoch. The test fold must be isolated from every fitting decision.

**Method choice:**
- *Optimiser:* AdamW (Loshchilov & Hutter, *ICLR*, 2019) with linear warm-up and cosine decay, a lower learning rate for the pretrained backbone than for the new layers, no weight decay on biases and normalisation parameters, and gradient-norm clipping.
- *Imbalance:* class-weighted cross-entropy with the training-fold weights from Part 5 (the single imbalance method used in this study; no resampling).
- *Mixed precision:* `torch.amp` autocast with a gradient scaler roughly halves memory and time on a T4.
- *Model selection:* the epoch with the highest **validation** balanced accuracy is kept (ties broken by lower validation loss); training stops after `PATIENCE` epochs without improvement. For the modality-dropout variants the validation data are left unmasked, so that both variants are selected by the same criterion.
- *Test isolation:* the training function receives only the training and validation indices. The test indices are used for inference alone, after the best epoch has been fixed; the validation ids used are fingerprinted into the run record and re-checked by the QC cell.
- *Resume:* after every epoch the model, optimiser, scheduler, scaler, early-stopping state and history are written to Drive (to a temporary file that then replaces the previous checkpoint). Because files deleted through the Colab mount go to the Drive trash and still count against the quota, the previous checkpoint is emptied before it is removed; otherwise hundreds of superseded checkpoints would fill a free Drive account during the full experiment. An unreadable checkpoint (cut off by a disconnect) restarts only the affected run. Random streams are re-seeded from (run seed, epoch) at each epoch start, so a resumed run follows the same data order.
- *RQ2 efficiency:* the image tokens of the test fold are computed once per run and the fusion head is evaluated on them under every masking condition; this is mathematically identical to a full forward pass because the backbone does not see the metadata.

**Output:** `train_deep_run()`; per run: prediction files, `_DONE.json`, optional best weights; `logs/epoch_log.csv` (the run summary `logs/run_summary.csv` is rebuilt from the `_DONE.json` records in Part 7).

In [ ]:
def drive_safe_remove(path: Path) -> None:
    """Delete a file without leaving its content in the Google Drive trash.

    Files removed through the Colab Drive mount are moved to the Drive trash, where they still count
    against the storage quota. Emptying the file first means only a zero-byte item reaches the trash.
    """
    path = Path(path)
    if path.is_file():
        with open(path, "wb"):
            pass
        path.unlink()


def drive_safe_replace(tmp: Path, dst: Path) -> None:
    """Replace `dst` by the completely written file `tmp` (see drive_safe_remove for why the old file is emptied)."""
    drive_safe_remove(dst)
    os.replace(tmp, dst)


def recover_interrupted_write(dst: Path) -> None:
    """If a session died between removing `dst` and renaming its finished temporary file, complete the rename."""
    tmp = Path(dst).with_name(Path(dst).name + ".tmp")
    if tmp.is_file() and not Path(dst).is_file():
        os.replace(tmp, dst)
    elif tmp.is_file():
        drive_safe_remove(tmp)                      # a stale, possibly partial temporary file


def _amp_enabled() -> bool:
    """Mixed precision is used only on CUDA."""
    return bool(CFG.USE_AMP and DEVICE.type == "cuda")


def md_mask_torch(n_rows: int, G: torch.Tensor, gen: torch.Generator) -> torch.Tensor:
    """GPU version of sample_md_mask: returns a boolean (n_rows, D) mask over transformed columns."""
    dev = G.device
    full = torch.rand(n_rows, device=dev, generator=gen) < CFG.MD_P_FULL
    partial = (~full) & (torch.rand(n_rows, device=dev, generator=gen) < CFG.MD_P_PARTIAL)
    rates = torch.rand(n_rows, device=dev, generator=gen)
    per_field = torch.rand(n_rows, G.shape[0], device=dev, generator=gen) < rates[:, None]
    field_mask = full[:, None] | (partial[:, None] & per_field)
    return (field_mask.float() @ G) > 0.5


@torch.no_grad()
def encode_tokens(model: FusionNet, idx: np.ndarray, meta_X: np.ndarray) -> torch.Tensor:
    """Backbone tokens for the given rows, kept on the device (evaluation view, no augmentation)."""
    model.eval()
    out = []
    for xb, _, _ in make_loader(idx, meta_X, train=False):
        with torch.autocast(device_type=DEVICE.type, enabled=_amp_enabled()):
            out.append(model.backbone(xb.to(DEVICE, non_blocking=True)))
    return torch.cat(out)


@torch.no_grad()
def head_logits(model: FusionNet, tokens: torch.Tensor, meta: np.ndarray) -> np.ndarray:
    """Logits of the fusion head for cached tokens and a (possibly masked) metadata matrix."""
    model.eval()
    meta_t = torch.from_numpy(np.ascontiguousarray(meta)).float().to(DEVICE)
    out = []
    bs = 2 * CFG.BATCH_SIZE
    for s in range(0, tokens.shape[0], bs):
        with torch.autocast(device_type=DEVICE.type, enabled=_amp_enabled()):
            out.append(model.forward_from_tokens(tokens[s:s + bs], meta_t[s:s + bs]).float().cpu())
    return torch.cat(out).numpy()


def train_deep_run(model_name: str, scheme: str, repeat: int, fold: int) -> Dict[str, Any]:
    """Train (or resume) one network run, then save validation / test / masked-test predictions."""
    family, use_md = MODEL_KIND[model_name]
    fp = run_files(scheme, model_name, repeat, fold)
    prep = get_prep(model_name, scheme, repeat, fold)
    tr, va, te = get_split(scheme, repeat, fold)      # `te` is used for inference only, after training has finished
    seed = run_seed(repeat, fold)
    seed_everything(seed)
    t_start = time.time()

    model = build_model(family, prep).to(DEVICE)
    decay, no_decay, bb_decay, bb_no_decay = [], [], [], []
    for name, p in model.named_parameters():
        is_bb = name.startswith("backbone.")
        if p.ndim <= 1:                                   # biases and normalisation parameters: no weight decay
            (bb_no_decay if is_bb else no_decay).append(p)
        else:
            (bb_decay if is_bb else decay).append(p)
    optimizer = torch.optim.AdamW([
        dict(params=bb_decay, lr=CFG.LR_BACKBONE, weight_decay=CFG.WEIGHT_DECAY),
        dict(params=bb_no_decay, lr=CFG.LR_BACKBONE, weight_decay=0.0),
        dict(params=decay, lr=CFG.LR_HEAD, weight_decay=CFG.WEIGHT_DECAY),
        dict(params=no_decay, lr=CFG.LR_HEAD, weight_decay=0.0),
    ])
    loader_gen = torch.Generator()
    train_loader = make_loader(tr, prep["X"], train=True, generator=loader_gen)
    val_loader = make_loader(va, prep["X"], train=False)
    if len(train_loader) == 0:
        raise RuntimeError("The training part is smaller than one batch; lower CFG.BATCH_SIZE.")
    steps_per_epoch = math.ceil(len(train_loader) / CFG.GRAD_ACCUM_STEPS)
    total_steps, warm_steps = CFG.EPOCHS * steps_per_epoch, CFG.WARMUP_EPOCHS * steps_per_epoch

    def lr_factor(step: int) -> float:
        """Linear warm-up followed by cosine decay to zero."""
        if step < warm_steps:
            return (step + 1) / max(1, warm_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, (step - warm_steps) / max(1, total_steps - warm_steps))))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=_amp_enabled())
    class_w = torch.tensor(prep["class_weights"], dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=CFG.LABEL_SMOOTHING)
    G_t = torch.tensor(prep["G"], device=DEVICE)
    miss_t = torch.tensor(prep["missvec"], device=DEVICE)
    md_gen = torch.Generator(device=DEVICE)

    state = dict(epoch=0, best_ba=-1.0, best_loss=float("inf"), best_epoch=-1, bad_epochs=0, history=[], seconds=0.0)
    recover_interrupted_write(fp["last"])
    recover_interrupted_write(fp["best"])
    ckpt = None
    if fp["last"].is_file():
        try:
            ckpt = torch.load(fp["last"], map_location=DEVICE, weights_only=False)
        except (RuntimeError, EOFError, OSError, zipfile.BadZipFile, ValueError) as exc:
            # A checkpoint cut off by a disconnect cannot be used; restarting this one run is the valid recovery.
            print(f"    [info] resume checkpoint is unreadable ({type(exc).__name__}); this run restarts from epoch 0.")
            drive_safe_remove(fp["last"])
    if ckpt is not None and not fp["best"].is_file() and ckpt["state"]["best_epoch"] >= 0:
        print("    [info] best-weights file is missing; this run restarts from epoch 0.")
        drive_safe_remove(fp["last"])
        ckpt = None
    if ckpt is not None:
        if ckpt["cfg_hash"] != CFG_HASH:
            raise RuntimeError(f"Checkpoint {fp['last']} belongs to another configuration ({ckpt['cfg_hash']} vs {CFG_HASH}). "
                               "Fix: restore the configuration or delete this checkpoint.")
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        state = ckpt["state"]
        print(f"    resumed from epoch {state['epoch']} (best validation BA so far {state['best_ba']:.4f})")

    stopped_early = state["bad_epochs"] >= CFG.PATIENCE
    for epoch in range(state["epoch"], CFG.EPOCHS):
        if stopped_early:
            break
        t_ep = time.time()
        seed_everything(seed * 1000 + epoch)                 # identical streams with or without a resume
        loader_gen.manual_seed(seed * 1000 + epoch)
        md_gen.manual_seed(seed * 1000 + epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        run_loss, n_seen = 0.0, 0
        n_batches = len(train_loader)
        for i, (xb, mb, yb) in enumerate(train_loader):
            xb, mb, yb = xb.to(DEVICE, non_blocking=True), mb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            if use_md:
                mb = torch.where(md_mask_torch(mb.shape[0], G_t, md_gen), miss_t.unsqueeze(0), mb)
            with torch.autocast(device_type=DEVICE.type, enabled=_amp_enabled()):
                logits = model(xb, mb)
            loss = criterion(logits.float(), yb)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite training loss in {model_name} r{repeat} f{fold} epoch {epoch}. "
                                         "Fix: lower LR_BACKBONE / LR_HEAD or set USE_AMP=False.")
            scaler.scale(loss / CFG.GRAD_ACCUM_STEPS).backward()
            if (i + 1) % CFG.GRAD_ACCUM_STEPS == 0 or (i + 1) == n_batches:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP_NORM)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                # With mixed precision the scaler skips an optimiser step whose gradients are not finite
                # (it then lowers its scale); the schedule advances only when a step was really taken.
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
            run_loss += float(loss.detach()) * yb.shape[0]
            n_seen += yb.shape[0]

        # ---- validation (validation part only; unmasked metadata)
        model.eval()
        v_logits = []
        with torch.no_grad():
            for xb, mb, _ in val_loader:
                with torch.autocast(device_type=DEVICE.type, enabled=_amp_enabled()):
                    v_logits.append(model(xb.to(DEVICE, non_blocking=True), mb.to(DEVICE, non_blocking=True)).float().cpu())
        v_logits_t = torch.cat(v_logits)
        val_loss = float(F.cross_entropy(v_logits_t, torch.from_numpy(Y[va]).long(), weight=class_w.cpu()))
        val_ba = balanced_accuracy_fast(Y[va], v_logits_t.argmax(1).numpy(), K)

        improved = (val_ba > state["best_ba"] + 1e-9) or (abs(val_ba - state["best_ba"]) <= 1e-9 and val_loss < state["best_loss"])
        if improved:
            state.update(best_ba=val_ba, best_loss=val_loss, best_epoch=epoch, bad_epochs=0)
            tmp = fp["best"].with_name(fp["best"].name + ".tmp")
            torch.save({"model": model.state_dict(), "cfg_hash": CFG_HASH, "epoch": epoch}, tmp)
            drive_safe_replace(tmp, fp["best"])
        else:
            state["bad_epochs"] += 1
        state["epoch"] = epoch + 1
        state["seconds"] += time.time() - t_ep
        row = dict(scheme=scheme, model=model_name, repeat=repeat, fold=fold, epoch=epoch,
                   train_loss=run_loss / max(1, n_seen), val_loss=val_loss, val_balanced_accuracy=val_ba,
                   lr_backbone=optimizer.param_groups[0]["lr"], lr_head=optimizer.param_groups[2]["lr"],
                   improved=bool(improved), seconds=round(time.time() - t_ep, 1))
        state["history"].append(row)
        append_csv_row(EPOCH_LOG_CSV, row)
        tmp = fp["last"].with_name(fp["last"].name + ".tmp")
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(), "state": state, "cfg_hash": CFG_HASH}, tmp)
        drive_safe_replace(tmp, fp["last"])
        print(f"    epoch {epoch + 1:>2d}/{CFG.EPOCHS}  train loss {row['train_loss']:.4f}  val loss {val_loss:.4f}  "
              f"val BA {val_ba:.4f}{'  *' if improved else ''}  ({row['seconds']:.0f} s)")
        stopped_early = state["bad_epochs"] >= CFG.PATIENCE

    # ---- inference with the selected (best-validation) weights
    best = torch.load(fp["best"], map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model"])
    X = prep["X"]
    val_tokens = encode_tokens(model, va, X)
    save_predictions(fp["val"], va, head_logits(model, val_tokens, X[va]), scheme, model_name, repeat, fold, seed)
    del val_tokens
    test_tokens = encode_tokens(model, te, X)
    save_predictions(fp["test"], te, head_logits(model, test_tokens, X[te]), scheme, model_name, repeat, fold, seed)
    if needs_robust(scheme, model_name):
        rob = np.stack([head_logits(model, test_tokens, apply_field_mask(X[te], condition_mask(c, repeat, fold, len(te)), prep))
                        for c in ROBUST_CONDITIONS.to_dict("records")]).astype(np.float32)
        np.savez_compressed(fp["robust"], cond_ids=np.array(COND_IDS), logits=rob)
    rec = dict(scheme=scheme, model=model_name, repeat=repeat, fold=fold, seed=seed, cfg_hash=CFG_HASH,
               backbone=CFG.BACKBONE_REGISTRY[CFG.BACKBONE], n_parameters=count_parameters(model),
               best_epoch=int(state["best_epoch"]) + 1, epochs_run=int(state["epoch"]), stopped_early=bool(stopped_early),
               val_balanced_accuracy=float(state["best_ba"]), n_train=len(tr), n_val=len(va), n_test=len(te),
               seconds=round(state["seconds"], 1), val_ids_digest=ids_digest(IMG_IDS[va]),
               finished_utc=dt.datetime.now(dt.timezone.utc).isoformat())
    save_json(rec, fp["done"])
    drive_safe_remove(fp["last"])                                # the resume checkpoint is no longer needed
    keep_best = scheme == "grouped" and (CFG.KEEP_BEST_WEIGHTS == "all" or
                                         (CFG.KEEP_BEST_WEIGHTS == "gradcam_model" and model_name == CFG.GRADCAM_MODEL))
    if not keep_best:
        drive_safe_remove(fp["best"])
    del model, optimizer, scaler, test_tokens
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    rec["wall_seconds_this_session"] = time.time() - t_start
    return rec

**Purpose:** Run every network for every split (image-only, the three trainable fusion networks, and their modality-dropout variants; plus the image-level leakage runs if enabled), skipping finished runs.

**Why:** This is the long-running part of the study. It is written so that "Run all" after a disconnect continues where the previous session stopped.

**Method choice:** Runs are ordered fold by fold within a repeat, so that a complete set of models for fold 0 exists early and intermediate analyses are possible. After each run the measured duration is used to print a projection of the remaining training time. A CUDA out-of-memory error is converted into an instruction (halve `BATCH_SIZE`, double `GRAD_ACCUM_STEPS`): these two settings are excluded from the configuration hash, so finished runs stay valid.

**Output:** All run files listed above. Drive usage: about 0.1 GB per kept best-weights file (by default only the Grad-CAM model, five files; see `KEEP_BEST_WEIGHTS`), plus one temporary resume checkpoint of about 0.35 GB for the run in progress.

In [ ]:
_todo = []
for _r in range(CFG.N_REPEATS):
    for _f in range(CFG.N_FOLDS):
        for _m in DEEP_MODELS:
            _todo.append(("grouped", _m, _r, _f))
if CFG.RUN_LEAKAGE:
    for _r in range(CFG.N_REPEATS):
        for _f in range(CFG.N_FOLDS):
            for _m in [m for m in DEEP_MODELS if m in CFG.LEAKAGE_MODELS]:
                _todo.append(("image_level", _m, _r, _f))
for _r in range(CFG.N_REPEATS):                      # restricted-metadata sensitivity models (primary scheme only)
    for _f in range(CFG.N_FOLDS):
        for _m in SENS_DEEP:
            _todo.append(("grouped", _m, _r, _f))
_pending = [t for t in _todo if not is_done(*t)]
print(f"Network runs: {len(_todo)} in total, {len(_todo) - len(_pending)} already finished, {len(_pending)} to do.")
_durations: List[float] = []
for _n, (_s, _m, _r, _f) in enumerate(_pending):
    print(f"\n[{_n + 1}/{len(_pending)}] {_s} / {_m} / repeat {_r} / fold {_f}")
    try:
        _rec = train_deep_run(_m, _s, _r, _f)
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError(
            "CUDA out of memory. Fix: in the configuration cell halve BATCH_SIZE and double GRAD_ACCUM_STEPS "
            "(the effective batch size and the configuration hash stay the same), then Runtime -> Restart and run all. "
            "Finished runs are kept and the interrupted run resumes from its last epoch."
        ) from exc
    _durations.append(_rec["wall_seconds_this_session"])
    _left = len(_pending) - (_n + 1)
    print(f"    finished: best epoch {_rec['best_epoch']}, validation BA {_rec['val_balanced_accuracy']:.4f}, "
          f"{_durations[-1] / 60:.1f} min. Projected time for the remaining {_left} runs: "
          f"{_left * float(np.mean(_durations)) / 3600:.1f} h (based on the runs of this session).")
print("\nAll network runs are finished.")

**Purpose:** Late (decision-level) fusion of the image-only network and the metadata model.

**Why:** Late fusion is the simplest multimodal strategy and needs no joint training, so it is a required reference point for RQ1. Because the two branches are independent, it is also the natural strategy when metadata may be absent.

**Method choice:** Weighted log-linear pooling: log p ∝ w · log p_image + (1 − w) · log p_metadata, with one weight *w* per split chosen on the **validation** part from a fixed grid (0, 0.05, …, 1) to maximise balanced accuracy (ties resolved towards the value closest to 0.5). Log-linear pooling is chosen over averaging probabilities because it corresponds to multiplying the two models' evidence. *Modality-dropout variant (`late_fusion_md`):* the metadata branch is the missingness-augmented model `meta_gbm_md`, and *w* is selected on the validation part together with its masked copies, so that the weight reflects the missingness the model is expected to meet. A single global weight cannot adapt to the missingness of an individual case; this is a known limitation of static late fusion and is part of what RQ2 measures. The test fold is not used for selecting *w*.

**Output:** Prediction files for `late_fusion` and `late_fusion_md` in the standard format; the chosen weight is stored in each `_DONE.json`.

In [ ]:
def read_logits(path: Path) -> Tuple[pd.DataFrame, np.ndarray]:
    """Load a prediction file and return (table, logits array in CLASS_NAMES order)."""
    tab = pd.read_csv(path, dtype={"img_id": str, "patient_id": str, "lesion_id": str})
    return tab, tab[[f"logit_{c}" for c in CLASS_NAMES]].to_numpy(dtype=np.float64)


def run_late_fusion(model: str, repeat: int, fold: int) -> None:
    """Combine image-only and metadata-model predictions for one grouped split."""
    use_md = MODEL_KIND[model][1]
    restricted = model in RESTRICTED_MODELS
    meta_model = "meta_gbm_r" if restricted else ("meta_gbm_md" if use_md else "meta_gbm")
    fp = run_files("grouped", model, repeat, fold)
    f_img, f_meta = run_files("grouped", "image_only", repeat, fold), run_files("grouped", meta_model, repeat, fold)
    _, va, te = get_split("grouped", repeat, fold)
    t_iv, l_iv = read_logits(f_img["val"])
    t_mv, l_mv = read_logits(f_meta["val"])
    t_it, l_it = read_logits(f_img["test"])
    t_mt, l_mt = read_logits(f_meta["test"])
    assert (t_iv["img_id"].to_numpy() == IMG_IDS[va]).all() and (t_mv["img_id"].to_numpy() == IMG_IDS[va]).all()
    assert (t_it["img_id"].to_numpy() == IMG_IDS[te]).all() and (t_mt["img_id"].to_numpy() == IMG_IDS[te]).all()
    lp_iv, lp_mv, lp_it, lp_mt = map(log_softmax_np, (l_iv, l_mv, l_it, l_mt))
    rob = None
    if not restricted:
        rob = np.load(f_meta["robust"], allow_pickle=False)
        assert list(rob["cond_ids"]) == COND_IDS, "Masked-test conditions of the metadata model do not match the current configuration."

    # ---- weight selection on validation data only
    sel_img, sel_meta, sel_y = lp_iv, lp_mv, Y[va]
    if use_md:
        aug = log_softmax_np(rob["val_aug_logits"].astype(np.float64))            # (n_aug, n_val, K), masked validation copies
        sel_img = np.concatenate([lp_iv] * (1 + aug.shape[0]))
        sel_meta = np.concatenate([lp_mv] + list(aug))
        sel_y = np.tile(Y[va], 1 + aug.shape[0])
    grid = np.round(np.arange(0.0, 1.0001, 0.05), 2)
    scores = np.array([balanced_accuracy_fast(sel_y, (w * sel_img + (1 - w) * sel_meta).argmax(1), K) for w in grid])
    best = np.flatnonzero(scores >= scores.max() - 1e-12)
    w = float(grid[best[np.argmin(np.abs(grid[best] - 0.5))]])

    seed = run_seed(repeat, fold)
    save_predictions(fp["val"], va, log_softmax_np(w * lp_iv + (1 - w) * lp_mv), "grouped", model, repeat, fold, seed)
    save_predictions(fp["test"], te, log_softmax_np(w * lp_it + (1 - w) * lp_mt), "grouped", model, repeat, fold, seed)
    if rob is not None:
        rob_meta = log_softmax_np(rob["logits"].astype(np.float64))               # (n_cond, n_test, K)
        fused = log_softmax_np(w * lp_it[None] + (1 - w) * rob_meta).astype(np.float32)
        np.savez_compressed(fp["robust"], cond_ids=np.array(COND_IDS), logits=fused)
    rec = dict(scheme="grouped", model=model, repeat=repeat, fold=fold, seed=seed, cfg_hash=CFG_HASH, image_weight=w,
               metadata_branch=meta_model, weight_selection_score=float(scores.max()),
               val_balanced_accuracy=balanced_accuracy_fast(Y[va], (w * lp_iv + (1 - w) * lp_mv).argmax(1), K), n_val=len(va), n_test=len(te),
               val_ids_digest=ids_digest(IMG_IDS[va]), finished_utc=dt.datetime.now(dt.timezone.utc).isoformat())
    save_json(rec, fp["done"])


for _m in LATE_MODELS + SENS_LATE:
    _w = []
    for _r in range(CFG.N_REPEATS):
        for _f in range(CFG.N_FOLDS):
            # Late fusion is cheap and depends on two other runs, so it is always recomputed from their saved files.
            run_late_fusion(_m, _r, _f)
            _w.append(load_json(run_files("grouped", _m, _r, _f)["done"])["image_weight"])
    print(f"[done] {_m}: image-branch weight per split = {_w}")

## Part 7. Out-of-fold prediction storage

**Purpose:** Collect the per-run prediction files into aligned arrays, verify their completeness and integrity, fit the temperature for every run on its validation part, and write one consolidated out-of-fold (OOF) file per model.

**Why:** All analyses in Parts 8–14 read these stored predictions, so they can be rerun, extended or audited without retraining. Integrity checks here (every image predicted exactly once per repeat, labels identical to the metadata) protect all later numbers.

**Method choice:** Temperature scaling (Guo et al., *ICML*, 2017) learns a single scalar *T* per run by minimising the negative log-likelihood of the **validation** predictions, then divides the test logits by *T*. It is the standard post-hoc calibration baseline, cannot change the predicted class, and with one parameter does not over-fit a small validation set. For the gradient-boosting and late-fusion models the stored "logits" are log-probabilities, to which the same procedure applies. The models singled out in figures ("best fusion model") are chosen by mean **validation** balanced accuracy, not by test performance, to avoid selecting on the test folds.

**Output:** `LOGITS`, `PROBS`, `PROBS_TS`, `TEST_FOLD`, `VALS`, `TEMPS`, `ROB`; files `predictions/oof_<scheme>_<model>.csv`, `logs/run_summary.csv`, `logs/temperatures.csv`, `logs/model_selection_validation.csv`.

In [ ]:
PRIMARY_REPEAT = 0     # RQ2-RQ4, figures and bootstrap intervals use repeat 0; fold statistics use all repeats
LOGIT_COLS = [f"logit_{c}" for c in CLASS_NAMES]


def fit_temperature(val_logits: np.ndarray, y_val: np.ndarray) -> float:
    """Temperature T > 0 minimising the validation negative log-likelihood of softmax(logits / T)."""
    def nll(log_t: float) -> float:
        lp = log_softmax_np(val_logits / math.exp(log_t))
        return float(-lp[np.arange(len(y_val)), y_val].mean())
    res = scipy.optimize.minimize_scalar(nll, bounds=(-3.0, 3.0), method="bounded", options={"xatol": 1e-6})
    return float(math.exp(res.x))


SCHEME_MODELS: Dict[str, List[str]] = {"grouped": list(MODELS) + list(SENS_MODELS)}   # sensitivity models are loaded too, analysed in Part 13
if CFG.RUN_LEAKAGE:
    SCHEME_MODELS["image_level"] = [m for m in MODELS if m in CFG.LEAKAGE_MODELS and MODEL_KIND[m][0] in ("image_only", "meta_gbm")]

LOGITS: Dict[Tuple[str, str], np.ndarray] = {}      # (scheme, model) -> (R, N, K) test logits in DF row order
PROBS: Dict[Tuple[str, str], np.ndarray] = {}
PROBS_TS: Dict[Tuple[str, str], np.ndarray] = {}
TEST_FOLD: Dict[str, np.ndarray] = {}               # scheme -> (R, N) test fold of every image
VALS: Dict[Tuple[str, str, int, int], Tuple[np.ndarray, np.ndarray]] = {}   # -> (val_idx, val_logits)
TEMPS: Dict[Tuple[str, str, int, int], float] = {}
_done_records, _temp_rows = [], []

for _s, _models in SCHEME_MODELS.items():
    TEST_FOLD[_s] = np.full((CFG.N_REPEATS, N), -1, dtype=int)
    for _r in range(CFG.N_REPEATS):
        for _f in range(CFG.N_FOLDS):
            TEST_FOLD[_s][_r, get_split(_s, _r, _f)[2]] = _f
    assert (TEST_FOLD[_s] >= 0).all()
    for _m in _models:
        _L = np.full((CFG.N_REPEATS, N, K), np.nan)
        _T = np.ones((CFG.N_REPEATS, N))
        for _r in range(CFG.N_REPEATS):
            for _f in range(CFG.N_FOLDS):
                _fp = run_files(_s, _m, _r, _f)
                if not is_done(_s, _m, _r, _f):
                    raise FileNotFoundError(f"Run {_s}/{_m}/r{_r}_f{_f} is not finished. Fix: rerun Part 6 (Run all).")
                _, _va, _te = get_split(_s, _r, _f)
                _tt, _lt = read_logits(_fp["test"])
                _tv, _lv = read_logits(_fp["val"])
                assert (_tt["img_id"].to_numpy() == IMG_IDS[_te]).all(), f"Test ids of {_fp['test']} do not match the split."
                assert (_tv["img_id"].to_numpy() == IMG_IDS[_va]).all(), f"Validation ids of {_fp['val']} do not match the split."
                assert (_tt["y_true"].to_numpy() == Y[_te]).all() and (_tv["y_true"].to_numpy() == Y[_va]).all(), "Stored labels differ from the metadata."
                _rec = load_json(_fp["done"])
                assert _rec["val_ids_digest"] == ids_digest(IMG_IDS[_va]), "Validation fingerprint mismatch."
                _temp = fit_temperature(_lv, Y[_va])                         # validation data only
                TEMPS[(_s, _m, _r, _f)] = _temp
                VALS[(_s, _m, _r, _f)] = (_va, _lv)
                _L[_r, _te] = _lt
                _T[_r, _te] = _temp
                _done_records.append(_rec)
                _temp_rows.append(dict(scheme=_s, model=_m, repeat=_r, fold=_f, temperature=_temp, n_val=len(_va)))
        assert np.isfinite(_L).all(), f"{_s}/{_m}: some images have no out-of-fold prediction."
        LOGITS[(_s, _m)] = _L
        PROBS[(_s, _m)] = softmax_np(_L)
        PROBS_TS[(_s, _m)] = softmax_np(_L / _T[:, :, None])
        _rows = []
        for _r in range(CFG.N_REPEATS):
            _o = pd.DataFrame({"img_id": IMG_IDS, "patient_id": PATIENTS, "lesion_id": DF["_lesion"].to_numpy(), "scheme": _s,
                               "model": _m, "repeat": _r, "fold": TEST_FOLD[_s][_r], "seed": [run_seed(_r, f) for f in TEST_FOLD[_s][_r]],
                               "y_true": Y, "label": DF["_label"].to_numpy(), "temperature": _T[_r]})
            for _k, _c in enumerate(CLASS_NAMES):
                _o[f"logit_{_c}"] = _L[_r, :, _k]
            for _k, _c in enumerate(CLASS_NAMES):
                _o[f"prob_{_c}"] = PROBS[(_s, _m)][_r, :, _k]
            for _k, _c in enumerate(CLASS_NAMES):
                _o[f"prob_ts_{_c}"] = PROBS_TS[(_s, _m)][_r, :, _k]
            _rows.append(_o)
        pd.concat(_rows, ignore_index=True).to_csv(DIRS["predictions"] / f"oof_{_s}_{_m}.csv", index=False, float_format="%.8g")

RUN_SUMMARY = pd.DataFrame(_done_records)
RUN_SUMMARY.to_csv(DIRS["logs"] / "run_summary.csv", index=False)
TEMPERATURES = pd.DataFrame(_temp_rows)
TEMPERATURES.to_csv(DIRS["logs"] / "temperatures.csv", index=False)

# Masked-test logits for RQ2: (R, n_conditions, N, K). The image-only model ignores metadata, so its
# prediction is identical under every condition and its test logits are repeated.
ROB: Dict[str, np.ndarray] = {}
for _m in MODELS:
    _A = np.full((CFG.N_REPEATS, len(COND_IDS), N, K), np.nan, dtype=np.float32)
    for _r in range(CFG.N_REPEATS):
        for _f in range(CFG.N_FOLDS):
            _te = get_split("grouped", _r, _f)[2]
            if USES_METADATA[_m]:
                _z = np.load(run_files("grouped", _m, _r, _f)["robust"], allow_pickle=False)
                assert list(_z["cond_ids"]) == COND_IDS, f"{_m}: stored masking conditions differ from the configuration."
                _A[_r][:, _te, :] = _z["logits"]
            else:
                _A[_r][:, _te, :] = LOGITS[("grouped", _m)][_r, _te][None]
    assert np.isfinite(_A).all()
    ROB[_m] = _A
    if USES_METADATA[_m]:   # consistency: the 0%-missing condition must reproduce the standard test prediction
        _c0 = COND_IDS.index("mcar_0.00_s0")
        assert np.allclose(softmax_np(_A[:, _c0].astype(np.float64)), PROBS[("grouped", _m)], atol=2e-3), \
            f"{_m}: predictions at 0% masking differ from the stored test predictions."

# ---- model selection for figures, on validation data only
_sel = (RUN_SUMMARY[RUN_SUMMARY["scheme"] == "grouped"].groupby("model")["val_balanced_accuracy"]
        .agg(["mean", "std", "count"]).reindex(MODELS).reset_index())
_sel.columns = ["model", "val_ba_mean", "val_ba_sd", "n_runs"]
_sel.to_csv(DIRS["logs"] / "model_selection_validation.csv", index=False)
FUSION_MODELS = [m for m in MODELS if MODEL_KIND[m][0] in ("early_concat", "late_fusion", "cross_attn", "metablock")]
SELECTED = {"image_only": "image_only"}
if FUSION_MODELS:
    _fs = _sel[_sel["model"].isin(FUSION_MODELS)]
    SELECTED["best_fusion"] = str(_fs.sort_values(["val_ba_mean", "model"], ascending=[False, True])["model"].iloc[0])
if "meta_gbm" in MODELS:
    SELECTED["meta_only"] = "meta_gbm"
print("Mean validation balanced accuracy per model (used only to choose which fusion model is shown in figures):")
display(_sel.round(4))
print(f"Models singled out in figures: {SELECTED}")
print(f"Temperatures (validation-fitted), grouped scheme: min {TEMPERATURES['temperature'].min():.3f}, "
      f"median {TEMPERATURES['temperature'].median():.3f}, max {TEMPERATURES['temperature'].max():.3f}")

## Part 8. RQ1 — discrimination

**Purpose:** Define all discrimination metrics, validate them against scikit-learn, and build the patient-level bootstrap resamples that every later interval uses.

**Why:** The bootstrap calls each metric thousands of times, so fast NumPy implementations are used; because they are custom code, they are checked against the reference implementation before any result is computed (and again in the QC cell).

**Method choice:** Balanced accuracy (mean per-class recall) is the primary metric because the classes are imbalanced and it is the metric most often reported on PAD-UFES-20. Macro-F1, per-class recall, one-vs-rest AUROC per class and their macro average are reported as required. Cancer-vs-non-cancer sensitivity and specificity are derived from the six-class prediction (an image counts as "predicted cancer" if its arg-max class is BCC, SCC or MEL), which needs no threshold; the binary AUROC uses the summed probability of the cancer classes. **Uncertainty:** images of one patient are not independent, so resampling images would give intervals that are too narrow. The cluster bootstrap resamples *patients* with replacement and takes all their images (Davison & Hinkley, *Bootstrap Methods and their Application*, 1997), with percentile intervals. The same resamples are reused for every model, which makes all between-model differences paired.

**Output:** `compute_metrics()`, `BOOT_IDX`, `BOOT_W`; console confirmation of the validation against scikit-learn.

In [ ]:
from sklearn import metrics as skm

METRIC_LABELS = {"balanced_accuracy": "Balanced accuracy", "macro_f1": "Macro-F1", "auroc_macro_ovr": "Macro AUROC (OvR)",
                 "accuracy": "Accuracy", "cancer_sensitivity": "Cancer sensitivity", "cancer_specificity": "Cancer specificity",
                 "cancer_auroc": "Cancer AUROC"}
PRIMARY_METRICS = ["balanced_accuracy", "macro_f1", "auroc_macro_ovr", "cancer_sensitivity", "cancer_specificity"]


def auc_binary_fast(is_pos: np.ndarray, score: np.ndarray) -> float:
    """AUROC via the Mann-Whitney statistic with mid-ranks for ties; NaN if one class is absent."""
    n_pos = int(is_pos.sum())
    n_neg = len(is_pos) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = scipy.stats.rankdata(score)
    return float((ranks[is_pos].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def compute_metrics(y: np.ndarray, P: np.ndarray, with_auc: bool = True) -> Dict[str, float]:
    """All discrimination metrics for labels y (n,) and probabilities P (n, K)."""
    pred = P.argmax(axis=1)
    cm = np.bincount(y * K + pred, minlength=K * K).reshape(K, K).astype(float)
    support, predicted, tp = cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    out: Dict[str, float] = {}
    with np.errstate(divide="ignore", invalid="ignore"):
        recall = np.where(support > 0, tp / support, np.nan)
        in_union = (support + predicted) > 0
        f1 = np.where(in_union, 2 * tp / (support + predicted), np.nan)     # 2TP / (2TP + FP + FN)
    out["accuracy"] = float(tp.sum() / cm.sum())
    out["balanced_accuracy"] = float(np.nanmean(recall)) if (support > 0).any() else float("nan")
    out["macro_f1"] = float(np.nanmean(f1))
    for k, c in enumerate(CLASS_NAMES):
        out[f"recall_{c}"] = float(recall[k])
    y_c, pred_c = np.isin(y, CANCER_IDX), np.isin(pred, CANCER_IDX)
    n_c, n_nc = int(y_c.sum()), int((~y_c).sum())
    out["cancer_sensitivity"] = float((y_c & pred_c).sum() / n_c) if n_c else float("nan")
    out["cancer_specificity"] = float((~y_c & ~pred_c).sum() / n_nc) if n_nc else float("nan")
    if with_auc:
        aucs = [auc_binary_fast(y == k, P[:, k]) for k in range(K)]
        for k, c in enumerate(CLASS_NAMES):
            out[f"auroc_{c}"] = aucs[k]
        out["auroc_macro_ovr"] = float(np.mean(aucs)) if not np.isnan(aucs).any() else float("nan")
        out["cancer_auroc"] = auc_binary_fast(y_c, P[:, CANCER_IDX].sum(axis=1))
    return out


def validate_metrics_against_sklearn(seed: int = 0, n: int = 400) -> Dict[str, bool]:
    """Compare every custom metric with scikit-learn on a random multi-class case that contains tied scores."""
    rng = np.random.default_rng(seed)
    y = rng.integers(0, K, size=n)
    y[:K] = np.arange(K)                                    # every class present
    P = rng.dirichlet(np.ones(K), size=n)
    P = np.round(P, 2) + 1e-9                               # rounding creates ties
    P = P / P.sum(axis=1, keepdims=True)
    pred = P.argmax(axis=1)
    m = compute_metrics(y, P)
    y_c, pred_c = np.isin(y, CANCER_IDX), np.isin(pred, CANCER_IDX)
    rec = skm.recall_score(y, pred, average=None, labels=list(range(K)), zero_division=0)
    ref = {
        "accuracy": skm.accuracy_score(y, pred),
        "balanced_accuracy": skm.balanced_accuracy_score(y, pred),
        "macro_f1": skm.f1_score(y, pred, average="macro"),
        "auroc_macro_ovr": skm.roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(K))),
        "cancer_sensitivity": skm.recall_score(y_c, pred_c),
        "cancer_specificity": skm.recall_score(~y_c, ~pred_c),
        "cancer_auroc": skm.roc_auc_score(y_c, P[:, CANCER_IDX].sum(axis=1)),
    }
    for k, c in enumerate(CLASS_NAMES):
        ref[f"recall_{c}"] = rec[k]
        ref[f"auroc_{c}"] = skm.roc_auc_score(y == k, P[:, k])
    checks = {name: bool(abs(m[name] - val) < 1e-9) for name, val in ref.items()}
    checks["balanced_accuracy_fast"] = bool(abs(balanced_accuracy_fast(y, pred, K) - ref["balanced_accuracy"]) < 1e-9)
    return checks


_checks = validate_metrics_against_sklearn()
assert all(_checks.values()), f"Custom metrics disagree with scikit-learn: {[k for k, v in _checks.items() if not v]}"
print(f"All {len(_checks)} custom metric functions agree with scikit-learn (tolerance 1e-9).")


def make_boot_indices(groups: np.ndarray, n_boot: int, seed: int) -> List[np.ndarray]:
    """Cluster bootstrap: each resample draws len(unique groups) groups with replacement and keeps all their rows."""
    uniq, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    bounds = np.concatenate([[0], np.cumsum(np.bincount(inv, minlength=len(uniq)))])
    members = [order[bounds[g]:bounds[g + 1]] for g in range(len(uniq))]
    rng = np.random.default_rng([seed, 31])
    return [np.concatenate([members[g] for g in rng.integers(0, len(uniq), size=len(uniq))]) for _ in range(n_boot)]


BOOT_IDX = make_boot_indices(CV_GROUPS, CFG.N_BOOT, CFG.SEED)
# The same resamples as a weight matrix (how often each image occurs), for vectorised metrics in RQ2 and RQ4.
BOOT_W = np.stack([np.bincount(ix, minlength=N) for ix in BOOT_IDX]).astype(np.float32)
ALPHA = 1.0 - CFG.CI_LEVEL


def percentile_ci(samples: np.ndarray) -> Tuple[float, float, int]:
    """Percentile interval over the finite bootstrap values; returns (lower, upper, number of valid resamples)."""
    v = np.asarray(samples, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) < max(20, 0.5 * len(samples)):
        return float("nan"), float("nan"), int(len(v))
    lo, hi = np.percentile(v, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
    return float(lo), float(hi), int(len(v))


print(f"Cluster bootstrap: {CFG.N_BOOT} resamples of {N_CV_GROUPS} {'patients' if GROUPS_ARE_PATIENTS else 'patient groups'} "
      f"(mean resample size {np.mean([len(i) for i in BOOT_IDX]):.0f} images).")

**Purpose:** Compute the RQ1 results: per-fold metrics (mean ± SD across folds), pooled out-of-fold point estimates and patient-level bootstrap 95% confidence intervals for every model.

**Why:** Mean ± SD across folds shows the variability over data splits; the bootstrap interval on the pooled out-of-fold predictions quantifies sampling uncertainty with patients as the independent unit. The two answer different questions and both are reported.

**Method choice:** The fold SD from five folds is itself an imprecise estimate and folds share training data, so it is reported as a descriptive spread, not used for inference. Inference uses the pooled out-of-fold predictions, in which every image is predicted exactly once by a model that never saw its patient. With melanoma represented by few images, per-class recall for that class has a wide interval; the table reports the interval rather than hiding it.

**Output:** `FOLD_METRICS`, `POOLED`, `BOOT` (bootstrap distributions per model), `RQ1`; files `logs/rq1_fold_metrics.csv`, `logs/rq1_summary_long.csv`.

In [ ]:
def bootstrap_metrics(y: np.ndarray, P: np.ndarray, idx_list: List[np.ndarray]) -> pd.DataFrame:
    """Metric values for every bootstrap resample (rows) of one model's predictions."""
    return pd.DataFrame([compute_metrics(y[ix], P[ix]) for ix in idx_list])


_fold_rows = []
POOLED: Dict[Tuple[str, str], Dict[str, float]] = {}
BOOT: Dict[Tuple[str, str], pd.DataFrame] = {}
_t0 = time.time()
for (_s, _m), _P in PROBS.items():
    for _r in range(CFG.N_REPEATS):
        for _f in range(CFG.N_FOLDS):
            _te = np.flatnonzero(TEST_FOLD[_s][_r] == _f)
            _fold_rows.append(dict(scheme=_s, model=_m, repeat=_r, fold=_f, n_test=len(_te), **compute_metrics(Y[_te], _P[_r, _te])))
    POOLED[(_s, _m)] = compute_metrics(Y, _P[PRIMARY_REPEAT])
    BOOT[(_s, _m)] = bootstrap_metrics(Y, _P[PRIMARY_REPEAT], BOOT_IDX)
FOLD_METRICS = pd.DataFrame(_fold_rows)
FOLD_METRICS.to_csv(DIRS["logs"] / "rq1_fold_metrics.csv", index=False)
ALL_METRICS = [c for c in FOLD_METRICS.columns if c not in ("scheme", "model", "repeat", "fold", "n_test")]

_rows = []
for (_s, _m) in PROBS:
    _fm = FOLD_METRICS[(FOLD_METRICS["scheme"] == _s) & (FOLD_METRICS["model"] == _m)]
    for _met in ALL_METRICS:
        _lo, _hi, _nv = percentile_ci(BOOT[(_s, _m)][_met].to_numpy())
        _rows.append(dict(scheme=_s, model=_m, metric=_met, pooled=POOLED[(_s, _m)][_met],
                          fold_mean=float(_fm[_met].mean()), fold_sd=float(_fm[_met].std(ddof=1)), n_folds=int(len(_fm)),
                          ci_low=_lo, ci_high=_hi, n_boot_valid=_nv))
RQ1 = pd.DataFrame(_rows)
RQ1.to_csv(DIRS["logs"] / "rq1_summary_long.csv", index=False)
print(f"RQ1 metrics and bootstrap distributions computed in {time.time() - _t0:.0f} s.\n")
_show = RQ1[(RQ1["scheme"] == "grouped") & RQ1["metric"].isin(PRIMARY_METRICS)].copy()
_show["value"] = _show.apply(lambda r: f"{r.pooled:.3f} [{r.ci_low:.3f}–{r.ci_high:.3f}]; {r.fold_mean:.3f} ± {r.fold_sd:.3f}", axis=1)
print("Cell format: pooled out-of-fold estimate [95% CI]; fold mean ± SD")
display(_show.pivot(index="model", columns="metric", values="value").reindex(MODELS)[PRIMARY_METRICS])

## Part 9. Statistical comparison

**Purpose:** Test the pre-specified model comparisons: every model against the image-only reference, and every modality-dropout variant against its standard counterpart.

**Why:** Differences between models must be reported with effect sizes, confidence intervals and multiplicity-adjusted p-values, using tests that are valid for this design.

**Method choice:**
- **(a) Paired patient-level bootstrap (primary).** For each comparison the difference in the pooled out-of-fold metric is computed on the same patient resamples for both models. The effect size is the difference with its percentile interval; the two-sided p-value is 2 · min(P(Δ* ≤ 0), P(Δ* ≥ 0)) with the add-one correction, so its smallest attainable value is about 2 / (N_BOOT + 1).
- **(b) DeLong's test** (DeLong et al., *Biometrics*, 1988; fast algorithm of Sun & Xu, *IEEE Signal Processing Letters*, 2014) for the cancer-vs-non-cancer AUROC on pooled predictions. Two caveats are printed in the table notes: DeLong's variance assumes independent observations, whereas images are clustered within patients, and pooled out-of-fold scores come from five different fitted models. It is therefore reported as a secondary analysis; the cluster bootstrap is the primary one.
- **(c) Wilcoxon signed-rank test on fold-level balanced accuracy** is run **only** when `N_REPEATS ≥ 2`. With five folds the smallest possible two-sided p-value is 2 / 2⁵ = 0.0625, so the test cannot reach 0.05 and would be uninformative. Even with repeated cross-validation, folds overlap in training data, which makes the test anti-conservative (Nadeau & Bengio, *Machine Learning*, 2003); it is reported as supplementary.
- **Multiplicity:** Holm's step-down procedure (Holm, *Scandinavian Journal of Statistics*, 1979) over all comparisons, separately for each metric/test.

**Output:** `STATS` table; file `logs/rq1_statistical_comparisons.csv`.

In [ ]:
def holm_adjust(p: Sequence[float]) -> np.ndarray:
    """Holm step-down adjusted p-values; NaN entries are ignored and stay NaN."""
    p = np.asarray(p, dtype=float)
    adj = np.full_like(p, np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if len(ok) == 0:
        return adj
    order = ok[np.argsort(p[ok], kind="stable")]
    m = len(order)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


assert np.allclose(holm_adjust([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06]), "Holm adjustment failed its unit test."


def paired_bootstrap(boot_a: np.ndarray, boot_b: np.ndarray, point_a: float, point_b: float) -> Dict[str, float]:
    """Difference a - b with percentile CI and two-sided bootstrap p-value (paired resamples)."""
    d = np.asarray(boot_a, dtype=float) - np.asarray(boot_b, dtype=float)
    d = d[np.isfinite(d)]
    lo, hi, n_valid = percentile_ci(d)
    if n_valid == 0 or not np.isfinite(lo):
        return dict(diff=point_a - point_b, ci_low=np.nan, ci_high=np.nan, p=np.nan)
    p = 2.0 * min((np.sum(d <= 0) + 1) / (len(d) + 1), (np.sum(d >= 0) + 1) / (len(d) + 1))
    return dict(diff=point_a - point_b, ci_low=lo, ci_high=hi, p=float(min(1.0, p)))


def delong_test(y_bin: np.ndarray, score_a: np.ndarray, score_b: np.ndarray) -> Dict[str, float]:
    """DeLong test for two correlated AUROCs (fast mid-rank algorithm). Returns AUCs, z and two-sided p."""
    pos = y_bin.astype(bool)
    m, n = int(pos.sum()), int((~pos).sum())
    if m < 2 or n < 2:
        return dict(auc_a=np.nan, auc_b=np.nan, z=np.nan, p=np.nan)
    S = np.vstack([score_a, score_b]).astype(float)
    X, Z = S[:, pos], S[:, ~pos]
    tx = scipy.stats.rankdata(X, axis=1)
    tz_ = scipy.stats.rankdata(Z, axis=1)
    tall = scipy.stats.rankdata(np.hstack([X, Z]), axis=1)
    aucs = tall[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tall[:, :m] - tx) / n
    v10 = 1.0 - (tall[:, m:] - tz_) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if not np.isfinite(var) or var <= 1e-18:
        return dict(auc_a=float(aucs[0]), auc_b=float(aucs[1]), z=np.nan, p=np.nan)
    z = (aucs[0] - aucs[1]) / math.sqrt(var)
    return dict(auc_a=float(aucs[0]), auc_b=float(aucs[1]), z=float(z), p=float(2 * scipy.stats.norm.sf(abs(z))))


# DeLong's AUC must equal the reference AUROC.
_rng = np.random.default_rng(1)
_yb, _sa, _sb = _rng.integers(0, 2, 300), _rng.random(300), _rng.random(300)
_dl = delong_test(_yb, _sa, _sb)
assert abs(_dl["auc_a"] - skm.roc_auc_score(_yb, _sa)) < 1e-9 and abs(_dl["auc_b"] - skm.roc_auc_score(_yb, _sb)) < 1e-9

REFERENCE_MODEL = "image_only"
COMPARISONS: List[Tuple[str, str, str]] = [(m, REFERENCE_MODEL, "vs image-only") for m in MODELS if m != REFERENCE_MODEL]
COMPARISONS += [(m, m[:-3], "modality dropout vs standard") for m in MODELS if m.endswith("_md") and m[:-3] in MODELS]
_n_fold_obs = CFG.N_REPEATS * CFG.N_FOLDS
_rows = []
for _a, _b, _family in COMPARISONS:
    _row: Dict[str, Any] = dict(model_a=_a, model_b=_b, family=_family)
    for _met in ("balanced_accuracy", "auroc_macro_ovr", "macro_f1"):
        _res = paired_bootstrap(BOOT[("grouped", _a)][_met].to_numpy(), BOOT[("grouped", _b)][_met].to_numpy(),
                                POOLED[("grouped", _a)][_met], POOLED[("grouped", _b)][_met])
        _row.update({f"{_met}_diff": _res["diff"], f"{_met}_ci_low": _res["ci_low"], f"{_met}_ci_high": _res["ci_high"], f"{_met}_p_boot": _res["p"]})
    _pa, _pb = PROBS[("grouped", _a)][PRIMARY_REPEAT], PROBS[("grouped", _b)][PRIMARY_REPEAT]
    _d = delong_test(Y_CANCER, _pa[:, CANCER_IDX].sum(axis=1), _pb[:, CANCER_IDX].sum(axis=1))
    _row.update(cancer_auroc_a=_d["auc_a"], cancer_auroc_b=_d["auc_b"], cancer_auroc_diff=_d["auc_a"] - _d["auc_b"],
                delong_z=_d["z"], delong_p=_d["p"])
    _fa = FOLD_METRICS[(FOLD_METRICS["scheme"] == "grouped") & (FOLD_METRICS["model"] == _a)].sort_values(["repeat", "fold"])["balanced_accuracy"].to_numpy()
    _fb = FOLD_METRICS[(FOLD_METRICS["scheme"] == "grouped") & (FOLD_METRICS["model"] == _b)].sort_values(["repeat", "fold"])["balanced_accuracy"].to_numpy()
    if CFG.N_REPEATS >= 2 and np.any(_fa != _fb):
        _row["wilcoxon_p"] = float(scipy.stats.wilcoxon(_fa, _fb, zero_method="wilcox", alternative="two-sided").pvalue)
    else:
        _row["wilcoxon_p"] = np.nan
    _row["wilcoxon_n"] = _n_fold_obs
    _rows.append(_row)
STATS = pd.DataFrame(_rows)
for _col in ("balanced_accuracy_p_boot", "auroc_macro_ovr_p_boot", "macro_f1_p_boot", "delong_p", "wilcoxon_p"):
    STATS[_col + "_holm"] = holm_adjust(STATS[_col].to_numpy())
STATS.to_csv(DIRS["logs"] / "rq1_statistical_comparisons.csv", index=False)
WILCOXON_NOTE = (f"Wilcoxon signed-rank test on fold-level balanced accuracy over {_n_fold_obs} folds." if CFG.N_REPEATS >= 2 else
                 f"Wilcoxon signed-rank test not run: with {_n_fold_obs} folds the minimum two-sided p-value is "
                 f"{2.0 / 2 ** _n_fold_obs:.4f}; set N_REPEATS >= 2 to enable it.")
print(WILCOXON_NOTE)
print(f"Smallest attainable bootstrap p-value: {2.0 / (CFG.N_BOOT + 1):.5f}; comparisons: {len(COMPARISONS)} (Holm-adjusted per metric).\n")
display(STATS[["model_a", "model_b", "balanced_accuracy_diff", "balanced_accuracy_ci_low", "balanced_accuracy_ci_high",
               "balanced_accuracy_p_boot", "balanced_accuracy_p_boot_holm", "delong_z", "delong_p", "delong_p_holm"]].round(4))

## Part 10. RQ2 — robustness to missing metadata

**Purpose:** Evaluate every model when metadata is partly or fully missing at test time (random missingness at 0–100% and the structured scenarios), and compare standard training with modality-dropout training.

**Why:** In deployment, clinical fields are often not recorded. A fusion model that is more accurate with complete metadata but collapses below the image-only model when fields are absent is not reliable. RQ2 measures that trade-off directly.

**Method choice:** Predictions under every masking condition were stored during Part 6, so this cell only computes metrics. For each missingness rate the balanced accuracy is averaged over the mask seeds; its 95% interval is a patient-level bootstrap of that seed-averaged value, which captures sampling uncertainty (the dominant source) rather than only mask-to-mask variation. Summary quantities: balanced accuracy at 0%, 50% and 100% missingness; the drop from 0% to 100%; the mean over the rate grid (normalised area under the robustness curve, trapezoidal rule); and, for each fusion family, the paired difference between the modality-dropout and the standard variant at 100% missingness and at 0% (the latter shows what modality dropout costs when metadata is complete). Masking is applied on top of the missingness already present in the data, so "0%" means "as recorded", not "complete". **Interpretation caveat:** in this dataset the pattern of missing metadata is strongly associated with the diagnosis (Part 3 and the audit in Part 13), so a model can read "fields missing" as evidence for a benign lesion. Masking therefore does two things at once: it removes information and it pushes cases towards the pattern typical of benign lesions. Cancer sensitivity and specificity at 0% and 100% missingness are reported next to balanced accuracy so that this effect is visible.

**Output:** `ROBUST_CURVES`, `ROBUST_SUMMARY`, `ROBUST_MD_EFFECT`; files `logs/rq2_curves.csv`, `logs/rq2_summary.csv`, `logs/rq2_md_effect.csv`.

In [ ]:
ONEHOT_Y = np.eye(K, dtype=np.float32)[Y]                   # (N, K)


def weighted_ba(correct: np.ndarray, weights: np.ndarray, class_onehot: np.ndarray = ONEHOT_Y) -> np.ndarray:
    """Balanced accuracy for many bootstrap resamples at once.

    correct      : (N,) 0/1 array, prediction correct or not.
    weights      : (B, N) bootstrap multiplicities (optionally already multiplied by a subgroup mask).
    Returns (B,) balanced accuracies over the classes that are present in each resample.
    """
    num = (weights * correct[None, :]) @ class_onehot       # (B, K) weighted correct per class
    den = weights @ class_onehot                            # (B, K) weighted support per class
    with np.errstate(divide="ignore", invalid="ignore"):
        rec = np.where(den > 0, num / den, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)      # resamples without any image of the subgroup give NaN by design
        return np.nanmean(rec, axis=1)


def weighted_rate(event: np.ndarray, within: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Proportion of `event` among the rows where `within` is true, for every row of the weight matrix (NaN if empty)."""
    within = within.astype(np.float32)
    den = weights @ within
    num = weights @ (event.astype(np.float32) * within)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 0, num / den, np.nan)


IS_CANCER = Y_CANCER.astype(bool)
ONES_W = np.ones((1, N), dtype=np.float32)
_cond_rate_groups: Dict[float, List[int]] = {}
for _i, _c in ROBUST_CONDITIONS.iterrows():
    if _c["kind"] == "mcar":
        _cond_rate_groups.setdefault(round(float(_c["rate"]), 4), []).append(_i)
RATES = sorted(_cond_rate_groups)

_curve_rows = []
ROB_BOOT: Dict[Tuple[str, str], np.ndarray] = {}            # (model, condition group) -> (B,) seed-averaged bootstrap BA
ROB_POINT: Dict[Tuple[str, str], float] = {}
for _m in MODELS:
    _pred = ROB[_m][PRIMARY_REPEAT].argmax(axis=2)          # (n_conditions, N)
    _correct = (_pred == Y[None, :]).astype(np.float32)
    _groups = {f"rate_{r:.2f}": idxs for r, idxs in _cond_rate_groups.items()}
    _groups.update({f"scenario_{n}": [COND_IDS.index(f"scenario_{n}")] for n in SCENARIO_FIELDS})
    for _g, _idxs in _groups.items():
        _point = float(np.mean([weighted_ba(_correct[i], ONES_W)[0] for i in _idxs]))
        _boot = np.mean([weighted_ba(_correct[i], BOOT_W) for i in _idxs], axis=0)
        _lo, _hi, _ = percentile_ci(_boot)
        ROB_BOOT[(_m, _g)], ROB_POINT[(_m, _g)] = _boot, _point
        _curve_rows.append(dict(model=_m, condition=_g, kind="mcar" if _g.startswith("rate_") else "scenario",
                                rate=float(_g.split("_")[1]) if _g.startswith("rate_") else np.nan,
                                n_mask_seeds=len(_idxs), balanced_accuracy=_point, ci_low=_lo, ci_high=_hi))
ROBUST_CURVES = pd.DataFrame(_curve_rows)
ROBUST_CURVES.to_csv(DIRS["logs"] / "rq2_curves.csv", index=False)


def _trapz_mean(values: np.ndarray, x: np.ndarray) -> np.ndarray:
    """Trapezoidal area under values(x) divided by the x-range; works on (..., len(x)) arrays."""
    dx = np.diff(x)
    return ((values[..., 1:] + values[..., :-1]) / 2.0 * dx).sum(axis=-1) / (x[-1] - x[0])


_rate_keys = [f"rate_{r:.2f}" for r in RATES]
_x = np.array(RATES)
_mid = f"rate_{min(RATES, key=lambda r: abs(r - 0.5)):.2f}"
_sum_rows = []
for _m in MODELS:
    _b = np.stack([ROB_BOOT[(_m, k)] for k in _rate_keys], axis=1)          # (B, n_rates)
    _p = np.array([ROB_POINT[(_m, k)] for k in _rate_keys])
    _row = dict(model=_m, ba_0=_p[0], ba_mid=ROB_POINT[(_m, _mid)], ba_100=_p[-1],
                drop_0_to_100=_p[0] - _p[-1], aurc_robust=float(_trapz_mean(_p, _x)))
    for _name, _samples in (("ba_0", _b[:, 0]), ("ba_mid", ROB_BOOT[(_m, _mid)]), ("ba_100", _b[:, -1]),
                            ("drop_0_to_100", _b[:, 0] - _b[:, -1]), ("aurc_robust", _trapz_mean(_b, _x))):
        _row[f"{_name}_ci_low"], _row[f"{_name}_ci_high"], _ = percentile_ci(_samples)
    for _n in SCENARIO_FIELDS:
        _row[f"scenario_{_n}"] = ROB_POINT[(_m, f"scenario_{_n}")]
        _row[f"scenario_{_n}_ci_low"], _row[f"scenario_{_n}_ci_high"], _ = percentile_ci(ROB_BOOT[(_m, f"scenario_{_n}")])
    # Cancer sensitivity and specificity with metadata as recorded (0%) and with all metadata missing (100%).
    _pred_all = ROB[_m][PRIMARY_REPEAT].argmax(axis=2)
    for _tag, _ci in (("0", _cond_rate_groups[RATES[0]][0]), ("100", _cond_rate_groups[RATES[-1]][0])):
        _pc = np.isin(_pred_all[_ci], CANCER_IDX)
        for _name, _event, _within in ((f"cancer_sens_{_tag}", _pc, IS_CANCER), (f"cancer_spec_{_tag}", ~_pc, ~IS_CANCER)):
            _row[_name] = float(weighted_rate(_event, _within, ONES_W)[0])
            _row[f"{_name}_ci_low"], _row[f"{_name}_ci_high"], _ = percentile_ci(weighted_rate(_event, _within, BOOT_W))
    _sum_rows.append(_row)
ROBUST_SUMMARY = pd.DataFrame(_sum_rows)
ROBUST_SUMMARY.to_csv(DIRS["logs"] / "rq2_summary.csv", index=False)
MID_RATE_LABEL = _mid.split("_")[1]

_md_rows = []
for _m in [m for m in MODELS if m.endswith("_md") and m[:-3] in MODELS]:
    for _g, _label in ((_rate_keys[0], "0% missing"), (_rate_keys[-1], "100% missing")):
        _res = paired_bootstrap(ROB_BOOT[(_m, _g)], ROB_BOOT[(_m[:-3], _g)], ROB_POINT[(_m, _g)], ROB_POINT[(_m[:-3], _g)])
        _md_rows.append(dict(model_md=_m, model_standard=_m[:-3], condition=_label, **_res))
ROBUST_MD_EFFECT = pd.DataFrame(_md_rows)
if len(ROBUST_MD_EFFECT):
    ROBUST_MD_EFFECT["p_holm"] = holm_adjust(ROBUST_MD_EFFECT["p"].to_numpy())
ROBUST_MD_EFFECT.to_csv(DIRS["logs"] / "rq2_md_effect.csv", index=False)
print("Balanced accuracy vs. random metadata missingness (seed-averaged; pooled out-of-fold):")
display(ROBUST_CURVES[ROBUST_CURVES["kind"] == "mcar"].pivot(index="model", columns="rate", values="balanced_accuracy").reindex(MODELS).round(3))
print("Modality dropout minus standard training (paired patient-level bootstrap):")
display(ROBUST_MD_EFFECT.round(4))

## Part 11. RQ3 — calibration and selective prediction

**Purpose:** Measure calibration (ECE, Brier score, negative log-likelihood, reliability diagrams) before and after temperature scaling, and selective prediction (risk–coverage curves, AURC, and performance at a validation-chosen abstention threshold).

**Why:** A model used for triage must give confidences that mean what they say, and must be able to defer uncertain cases. Discrimination metrics do not show this.

**Method choice:**
- **ECE:** top-label expected calibration error with **15 equal-width bins** on the maximum predicted probability (Naeini et al., *AAAI*, 2015; Guo et al., *ICML*, 2017): Σ_b (n_b / n) · |accuracy_b − mean confidence_b|. Equal-width ECE depends on the binning and is biased for small samples, so an **equal-mass** (15 quantile bins) variant is reported alongside it.
- **Brier score:** multi-class form, the mean over images of the squared distance between the probability vector and the one-hot label (Brier, *Monthly Weather Review*, 1950); range 0–2, lower is better.
- **Temperature scaling** uses the per-run temperatures fitted on validation data in Part 7; "after" numbers are test-fold predictions rescaled with their own run's temperature.
- **Selective prediction:** confidence is the maximum probability; the risk–coverage curve gives the error rate among the most confident fraction of cases, and AURC is its mean over all coverages (Geifman & El-Yaniv, *NeurIPS*, 2017; Geifman et al., *ICLR*, 2019). AURC mixes ranking quality and base error, so the error at full coverage is shown next to it.
- **Operating point:** for each run the confidence threshold that retains `SELECTIVE_TARGET_COVERAGE` of the **validation** images is applied unchanged to the test fold; the realised test coverage and the selective accuracy and balanced accuracy are reported. The threshold is never tuned on test data. Calibrated probabilities are used for this operating point.
- **Cancer threshold:** in addition to the arg-max rule of RQ1, a threshold on the calibrated cancer probability is chosen per run on validation data by maximising Youden's J (Youden, *Cancer*, 1950) and applied to the test fold.

**Output:** `CALIB`, `RELIABILITY`, `RISK_COVERAGE`; files `logs/rq3_calibration.csv`, `logs/rq3_reliability_bins.csv`, `logs/rq3_selective_thresholds.csv`.

In [ ]:
def ece_equal_width(conf: np.ndarray, correct: np.ndarray, n_bins: int) -> float:
    """Top-label ECE with equal-width bins (lower edge exclusive, upper edge inclusive)."""
    b = np.clip(np.ceil(conf * n_bins).astype(int) - 1, 0, n_bins - 1)
    cnt = np.bincount(b, minlength=n_bins).astype(float)
    s_conf = np.bincount(b, weights=conf, minlength=n_bins)
    s_acc = np.bincount(b, weights=correct, minlength=n_bins)
    return float(np.abs(s_acc - s_conf).sum() / len(conf))


def ece_equal_mass(conf: np.ndarray, correct: np.ndarray, n_bins: int) -> float:
    """Top-label ECE with equal-mass (quantile) bins."""
    order = np.argsort(conf, kind="stable")
    parts = np.array_split(order, n_bins)
    return float(sum(abs(correct[p].mean() - conf[p].mean()) * len(p) for p in parts if len(p)) / len(conf))


def brier_multiclass(y: np.ndarray, P: np.ndarray) -> float:
    """Mean squared distance between probability vectors and one-hot labels (range 0-2)."""
    onehot = np.eye(P.shape[1])[y]
    return float(((P - onehot) ** 2).sum(axis=1).mean())


def nll_multiclass(y: np.ndarray, P: np.ndarray) -> float:
    """Mean negative log-likelihood of the true class."""
    return float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1.0)).mean())


def risk_coverage(conf: np.ndarray, correct: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Coverage and selective risk (error rate) when cases are accepted in order of decreasing confidence."""
    order = np.argsort(-conf, kind="stable")
    err = 1.0 - correct[order]
    n = len(conf)
    return np.arange(1, n + 1) / n, np.cumsum(err) / np.arange(1, n + 1)


def aurc(conf: np.ndarray, correct: np.ndarray) -> float:
    """Area under the risk-coverage curve: mean selective risk over all coverages."""
    return float(risk_coverage(conf, correct)[1].mean())


def reliability_bins(conf: np.ndarray, correct: np.ndarray, n_bins: int) -> pd.DataFrame:
    """Per-bin count, mean confidence and accuracy (equal-width bins) for reliability diagrams."""
    b = np.clip(np.ceil(conf * n_bins).astype(int) - 1, 0, n_bins - 1)
    cnt = np.bincount(b, minlength=n_bins)
    with np.errstate(divide="ignore", invalid="ignore"):
        mc = np.bincount(b, weights=conf, minlength=n_bins) / cnt
        ac = np.bincount(b, weights=correct, minlength=n_bins) / cnt
    return pd.DataFrame({"bin": np.arange(n_bins), "bin_low": np.arange(n_bins) / n_bins, "bin_high": (np.arange(n_bins) + 1) / n_bins,
                         "count": cnt, "mean_confidence": mc, "accuracy": ac})


_cal_rows, _rel_rows, _thr_rows = [], [], []
RISK_COVERAGE: Dict[Tuple[str, str], Tuple[np.ndarray, np.ndarray]] = {}
_r = PRIMARY_REPEAT
for _m in MODELS:
    _row: Dict[str, Any] = dict(model=_m)
    for _tag, _P in (("raw", PROBS[("grouped", _m)][_r]), ("ts", PROBS_TS[("grouped", _m)][_r])):
        _conf, _pred = _P.max(axis=1), _P.argmax(axis=1)
        _corr = (_pred == Y).astype(float)
        _point = dict(ece=ece_equal_width(_conf, _corr, CFG.ECE_BINS), ece_mass=ece_equal_mass(_conf, _corr, CFG.ECE_BINS),
                      brier=brier_multiclass(Y, _P), nll=nll_multiclass(Y, _P), aurc=aurc(_conf, _corr))
        _boot = {k: np.empty(len(BOOT_IDX)) for k in ("ece", "brier", "aurc")}
        for _b, _ix in enumerate(BOOT_IDX):
            _boot["ece"][_b] = ece_equal_width(_conf[_ix], _corr[_ix], CFG.ECE_BINS)
            _boot["brier"][_b] = brier_multiclass(Y[_ix], _P[_ix])
            _boot["aurc"][_b] = aurc(_conf[_ix], _corr[_ix])
        for _k, _v in _point.items():
            _row[f"{_k}_{_tag}"] = _v
        for _k, _v in _boot.items():
            _row[f"{_k}_{_tag}_ci_low"], _row[f"{_k}_{_tag}_ci_high"], _ = percentile_ci(_v)
            _row[f"_boot_{_k}_{_tag}"] = _v
        RISK_COVERAGE[(_m, _tag)] = risk_coverage(_conf, _corr)
        _rb = reliability_bins(_conf, _corr, CFG.ECE_BINS)
        _rb.insert(0, "calibration", _tag)
        _rb.insert(0, "model", _m)
        _rel_rows.append(_rb)
    for _k in ("ece", "brier", "aurc"):      # paired change after - before temperature scaling
        _d = _row.pop(f"_boot_{_k}_ts") - _row.pop(f"_boot_{_k}_raw")
        _row[f"{_k}_change"] = _row[f"{_k}_ts"] - _row[f"{_k}_raw"]
        _row[f"{_k}_change_ci_low"], _row[f"{_k}_change_ci_high"], _ = percentile_ci(_d)
    _row["error_full_coverage"] = 1.0 - POOLED[("grouped", _m)]["accuracy"]
    _row["temperature_mean"] = float(np.mean([TEMPS[("grouped", _m, _r, f)] for f in range(CFG.N_FOLDS)]))

    # ---- operating points chosen on validation data, applied to the test folds
    _accept = np.zeros(N, dtype=bool)
    _pred_cancer_thr = np.zeros(N, dtype=bool)
    for _f in range(CFG.N_FOLDS):
        _va, _lv = VALS[("grouped", _m, _r, _f)]
        _temp = TEMPS[("grouped", _m, _r, _f)]
        _pv = softmax_np(_lv / _temp)
        _tau = float(np.quantile(_pv.max(axis=1), 1.0 - CFG.SELECTIVE_TARGET_COVERAGE))
        _te = np.flatnonzero(TEST_FOLD["grouped"][_r] == _f)
        _pt = PROBS_TS[("grouped", _m)][_r, _te]
        _accept[_te] = _pt.max(axis=1) >= _tau
        _sv, _yv = _pv[:, CANCER_IDX].sum(axis=1), Y_CANCER[_va]
        _fpr, _tpr, _thr = skm.roc_curve(_yv, _sv)
        _finite = np.isfinite(_thr)
        _cthr = float(_thr[_finite][np.argmax((_tpr - _fpr)[_finite])])
        _pred_cancer_thr[_te] = _pt[:, CANCER_IDX].sum(axis=1) >= _cthr
        _thr_rows.append(dict(model=_m, repeat=_r, fold=_f, confidence_threshold=_tau, cancer_probability_threshold=_cthr, n_val=len(_va)))
    _pts = PROBS_TS[("grouped", _m)][_r]
    _corr_ts = (_pts.argmax(axis=1) == Y)
    _row["selective_coverage"] = float(_accept.mean())
    _row["selective_n"] = int(_accept.sum())
    _row["selective_accuracy"] = float(_corr_ts[_accept].mean()) if _accept.any() else np.nan
    _row["selective_balanced_accuracy"] = balanced_accuracy_fast(Y[_accept], _pts.argmax(axis=1)[_accept], K) if _accept.any() else np.nan
    _row["selective_n_classes_present"] = int(len(np.unique(Y[_accept])))
    _yc = Y_CANCER.astype(bool)
    _row["cancer_sens_youden"] = float((_pred_cancer_thr & _yc).sum() / _yc.sum())
    _row["cancer_spec_youden"] = float((~_pred_cancer_thr & ~_yc).sum() / (~_yc).sum())
    _cal_rows.append(_row)
CALIB = pd.DataFrame(_cal_rows)
RELIABILITY = pd.concat(_rel_rows, ignore_index=True)
CALIB.to_csv(DIRS["logs"] / "rq3_calibration.csv", index=False)
RELIABILITY.to_csv(DIRS["logs"] / "rq3_reliability_bins.csv", index=False)
pd.DataFrame(_thr_rows).to_csv(DIRS["logs"] / "rq3_selective_thresholds.csv", index=False)
display(CALIB[["model", "ece_raw", "ece_ts", "ece_mass_raw", "ece_mass_ts", "brier_raw", "brier_ts", "aurc_raw", "aurc_ts",
               "error_full_coverage", "selective_coverage", "selective_accuracy", "temperature_mean"]].round(4))

## Part 12. RQ4 — fairness across Fitzpatrick skin types and other subgroups

**Purpose:** Report performance per Fitzpatrick skin type, age band and sex, with sample sizes and patient-level bootstrap intervals, and the gap between the best and worst sufficiently large subgroup.

**Why:** An average metric can hide systematic under-performance for a subgroup. The dataset documentation indicates that darker skin types are under-represented, so intervals and sample sizes must be shown and small subgroups must not be over-interpreted.

**Method choice:** For each subgroup: number of images and patients, accuracy, balanced accuracy over the classes present in that subgroup (with the number of classes present, because a subgroup that lacks a class is not comparable on this metric), and cancer sensitivity and specificity. Intervals reuse the patient-level bootstrap, restricted to the subgroup. Subgroups with fewer than `MIN_SUBGROUP_N` images are **flagged**; their numbers are listed for transparency but excluded from the gap and must not be interpreted. Images with a missing subgroup value are reported as a separate "Missing" group and excluded from the gap; in this dataset missing Fitzpatrick values are not random with respect to diagnosis (see the missingness-by-class table in Part 3), so that group is not a neutral reference. The gap (maximum minus minimum balanced accuracy over eligible subgroups) is reported with a bootstrap interval; as a maximum of noisy differences it is biased upwards, which the table notes state. No fairness-mitigation method is applied: RQ4 is an audit.

**Output:** `FAIR`, `FAIR_GAPS`; files `logs/rq4_subgroups.csv`, `logs/rq4_gaps.csv`.

In [ ]:
SUBGROUPS: Dict[str, pd.Series] = {}
if ROLES["fitzpatrick"] is not None:
    SUBGROUPS["Fitzpatrick type"] = pd.Series(pd.Categorical(DF["_fitz"], categories=EDA["fitz_levels"], ordered=True))
if ROLES["age"] is not None:
    _age = pd.to_numeric(DF[ROLES["age"]].astype(str).str.strip().where(~is_missing_token(DF[ROLES["age"]])), errors="coerce")
    _edges = list(CFG.AGE_BIN_EDGES)
    _labels = [f"<{_edges[1]}"] + [f"{_edges[i]}–{_edges[i + 1] - 1}" for i in range(1, len(_edges) - 2)] + [f"≥{_edges[-2]}"]
    _band = pd.cut(_age, bins=_edges, right=False, labels=_labels).astype(object)
    _band = _band.where(_age.notna(), "Missing")
    if _band.isna().any():
        raise ValueError(f"Some ages fall outside CFG.AGE_BIN_EDGES = {_edges}; widen the outer edges.")
    SUBGROUPS["Age band (years)"] = pd.Series(pd.Categorical(_band, categories=_labels + (["Missing"] if (_band == "Missing").any() else []), ordered=True))
if ROLES["sex"] is not None:
    _sx = DF[ROLES["sex"]].astype(str).str.strip().str.upper()
    _sx = _sx.where(~is_missing_token(_sx), "Missing")
    _lv = sorted([v for v in _sx.unique() if v != "Missing"]) + (["Missing"] if (_sx == "Missing").any() else [])
    SUBGROUPS["Sex"] = pd.Series(pd.Categorical(_sx, categories=_lv, ordered=True))
if not SUBGROUPS:
    print("[warning] no subgroup variable is available in this file; RQ4 outputs are n/a.")

_yc = Y_CANCER.astype(np.float32)
_fair_rows, _gap_rows = [], []
for _attr, _series in SUBGROUPS.items():
    _codes = _series.to_numpy()
    for _m in MODELS:
        _P = PROBS[("grouped", _m)][PRIMARY_REPEAT]
        _pred = _P.argmax(axis=1)
        _correct = (_pred == Y).astype(np.float32)
        _pc = np.isin(_pred, CANCER_IDX).astype(np.float32)
        _elig_boot, _elig_point, _elig_names = [], [], []
        for _level in _series.cat.categories:
            _g = (_codes == _level)
            _n = int(_g.sum())
            if _n == 0:
                continue
            _gw = BOOT_W * _g[None, :].astype(np.float32)
            _nb = _gw.sum(axis=1)
            with np.errstate(divide="ignore", invalid="ignore"):
                _acc_b = np.where(_nb > 0, (_gw @ _correct) / _nb, np.nan)
                _sens_b = np.where((_gw @ _yc) > 0, (_gw @ (_yc * _pc)) / (_gw @ _yc), np.nan)
                _spec_b = np.where((_gw @ (1 - _yc)) > 0, (_gw @ ((1 - _yc) * (1 - _pc))) / (_gw @ (1 - _yc)), np.nan)
            _ba_b = weighted_ba(_correct, _gw)
            _n_cancer = int((_g & (Y_CANCER == 1)).sum())
            _n_non = _n - _n_cancer
            _row = dict(attribute=_attr, subgroup=str(_level), model=_m, n_images=_n, n_patients=int(len(set(PATIENTS[_g]))),
                        n_classes_present=int(len(np.unique(Y[_g]))), n_cancer=_n_cancer, n_non_cancer=_n_non,
                        flag_small=bool(_n < CFG.MIN_SUBGROUP_N),
                        accuracy=float(_correct[_g].mean()), balanced_accuracy=balanced_accuracy_fast(Y[_g], _pred[_g], K),
                        cancer_sensitivity=float(_pc[_g & (Y_CANCER == 1)].mean()) if _n_cancer else np.nan,
                        cancer_specificity=float(1 - _pc[_g & (Y_CANCER == 0)].mean()) if _n_non else np.nan)
            for _name, _b in (("accuracy", _acc_b), ("balanced_accuracy", _ba_b), ("cancer_sensitivity", _sens_b), ("cancer_specificity", _spec_b)):
                _row[f"{_name}_ci_low"], _row[f"{_name}_ci_high"], _ = percentile_ci(_b)
            _fair_rows.append(_row)
            if _n >= CFG.MIN_SUBGROUP_N and str(_level) != "Missing":
                _elig_boot.append(_ba_b)
                _elig_point.append(_row["balanced_accuracy"])
                _elig_names.append(str(_level))
        if len(_elig_names) >= 2:
            _eb = np.stack(_elig_boot, axis=1)
            _gb = np.nanmax(_eb, axis=1) - np.nanmin(_eb, axis=1)
            _lo, _hi, _ = percentile_ci(_gb)
            _gap_rows.append(dict(attribute=_attr, model=_m, n_eligible_subgroups=len(_elig_names), eligible_subgroups=", ".join(_elig_names),
                                  best_subgroup=_elig_names[int(np.argmax(_elig_point))], worst_subgroup=_elig_names[int(np.argmin(_elig_point))],
                                  gap=float(max(_elig_point) - min(_elig_point)), gap_ci_low=_lo, gap_ci_high=_hi))
        else:
            _gap_rows.append(dict(attribute=_attr, model=_m, n_eligible_subgroups=len(_elig_names), eligible_subgroups=", ".join(_elig_names),
                                  best_subgroup="", worst_subgroup="", gap=np.nan, gap_ci_low=np.nan, gap_ci_high=np.nan))
FAIR = pd.DataFrame(_fair_rows)
FAIR_GAPS = pd.DataFrame(_gap_rows)
FAIR.to_csv(DIRS["logs"] / "rq4_subgroups.csv", index=False)
FAIR_GAPS.to_csv(DIRS["logs"] / "rq4_gaps.csv", index=False)
if len(FAIR):
    _mshow = SELECTED.get("best_fusion", "image_only")
    print(f"Subgroup results for '{_mshow}' (all models are in logs/rq4_subgroups.csv):")
    display(FAIR[FAIR["model"] == _mshow][["attribute", "subgroup", "n_images", "n_patients", "n_classes_present", "flag_small",
                                           "balanced_accuracy", "balanced_accuracy_ci_low", "balanced_accuracy_ci_high",
                                           "cancer_sensitivity", "cancer_specificity"]].round(3))
    display(FAIR_GAPS[FAIR_GAPS["model"] == _mshow].round(3))

## Part 13. Leakage analysis (optional)

**Purpose:** Quantify how much an image-level split, which lets images of the same patient appear in training and test data, inflates the estimated performance compared with the patient-grouped protocol.

**Why:** Part of the published PAD-UFES-20 literature does not state whether splits were grouped by patient. Measuring the inflation on the same model and data shows how far results from the two protocols can be compared, which is needed to interpret the literature table (T8).

**Method choice:** The identical model, settings and seeds are trained under `StratifiedKFold` on images (inner validation split also on images). Because both schemes yield one out-of-fold prediction for every image, the difference is evaluated with the paired patient-level bootstrap. The metadata model is included because metadata is constant within a patient, so image-level splitting lets a tabular model partly recognise patients. Controlled by `CFG.RUN_LEAKAGE`.

**Output:** `LEAKAGE` table; file `logs/leakage_comparison.csv`.

In [ ]:
_leak_rows = []
if CFG.RUN_LEAKAGE:
    for _m in SCHEME_MODELS["image_level"]:
        for _met in ("balanced_accuracy", "macro_f1", "auroc_macro_ovr"):
            _res = paired_bootstrap(BOOT[("image_level", _m)][_met].to_numpy(), BOOT[("grouped", _m)][_met].to_numpy(),
                                    POOLED[("image_level", _m)][_met], POOLED[("grouped", _m)][_met])
            _row = dict(model=_m, metric=_met, inflation=_res["diff"], inflation_ci_low=_res["ci_low"],
                        inflation_ci_high=_res["ci_high"], p_boot=_res["p"])
            for _s in ("grouped", "image_level"):
                _q = RQ1[(RQ1["scheme"] == _s) & (RQ1["model"] == _m) & (RQ1["metric"] == _met)].iloc[0]
                _row.update({f"{_s}_pooled": _q["pooled"], f"{_s}_ci_low": _q["ci_low"], f"{_s}_ci_high": _q["ci_high"],
                             f"{_s}_fold_mean": _q["fold_mean"], f"{_s}_fold_sd": _q["fold_sd"]})
            _leak_rows.append(_row)
LEAKAGE = pd.DataFrame(_leak_rows)
LEAKAGE.to_csv(DIRS["logs"] / "leakage_comparison.csv", index=False)
if len(LEAKAGE):
    display(LEAKAGE[["model", "metric", "grouped_pooled", "image_level_pooled", "inflation", "inflation_ci_low", "inflation_ci_high", "p_boot"]].round(4))
else:
    print("Leakage analysis is disabled (CFG.RUN_LEAKAGE = False); Table T10 and Figure F9 will be skipped.")

### Metadata-missingness audit (extension of Part 13)

**Purpose:** Test whether the *pattern of missing metadata*, rather than the clinical content of the metadata, reveals the diagnosis, and show how every model performs separately on images with recorded metadata and on images whose metadata is mostly missing.

**Why:** The missingness-by-class table in Part 3 shows that a block of fields is recorded for some diagnoses and largely absent for others. If "this block is missing" is by itself predictive of the label, then (i) a metadata or fusion model can score well by detecting missingness, which would not transfer to a clinic where the fields are collected for every patient, (ii) the missingness indicators that this pipeline adds make the shortcut easy to learn, and (iii) the robustness experiment (RQ2) and the "Missing" subgroups (RQ4) are affected. This is the same kind of label leakage as the biopsy flag, only indirect, and a trustworthiness paper has to quantify it rather than report the pooled numbers alone.

**Method choice:** Three analyses, none of which changes the main models. (1) *Association:* a 2 × 2 table of "metadata mostly missing" (at least `AUDIT_ROW_MISSING_SHARE` of an image's fields missing) against cancer vs non-cancer, with the sensitivity and specificity of the trivial rule "mostly missing ⇒ non-cancer". (2) *Ablated metadata-only models,* trained on the same folds with the same settings as the metadata-only baseline: one that sees **only the missingness pattern** (one 0/1 indicator per field, no values), and one that sees only the fields with little missingness (at most `AUDIT_LOW_MISSING_MAX`), whose availability does not depend on the diagnosis. If the missingness-only model approaches the full metadata model, most of the metadata signal is the shortcut. (3) *Stratified evaluation* of all models from their stored out-of-fold predictions, separately for images with recorded metadata and images with mostly missing metadata. The stratum with recorded metadata is the fairer estimate of what metadata adds, because the shortcut cannot separate cases inside it. Intervals are patient-level bootstrap intervals. This audit describes the problem; it does not remove it. Removing it needs a sensitivity analysis in which all models are retrained without access to missingness information (see the notes delivered with the notebook).

**Output:** `AUDIT_ASSOC`, `AUDIT_MODELS`, `AUDIT_STRATA`; files `logs/audit_missingness_association.csv`, `logs/audit_metadata_variants.csv`, `logs/audit_strata.csv`.

In [ ]:
AUDIT_ASSOC: Optional[Dict[str, Any]] = None
AUDIT_MODELS = pd.DataFrame()
AUDIT_STRATA = pd.DataFrame()
AUDIT_SKIP_REASON = ""
ROW_MISSING_SHARE = MISSING_MATRIX.mean(axis=1)
MOSTLY_MISSING = ROW_MISSING_SHARE >= CFG.AUDIT_ROW_MISSING_SHARE

if not CFG.RUN_SHORTCUT_AUDIT:
    AUDIT_SKIP_REASON = "CFG.RUN_SHORTCUT_AUDIT is False"
elif MOSTLY_MISSING.sum() == 0 or (~MOSTLY_MISSING).sum() == 0:
    AUDIT_SKIP_REASON = (f"no image (or every image) has at least {100 * CFG.AUDIT_ROW_MISSING_SHARE:.0f}% of its metadata fields missing, "
                         "so there is no missingness pattern to audit")
else:
    # ---- (1) association between the missingness pattern and the cancer / non-cancer label
    _n_cm, _n_cr = int((IS_CANCER & MOSTLY_MISSING).sum()), int((IS_CANCER & ~MOSTLY_MISSING).sum())
    _n_bm, _n_br = int((~IS_CANCER & MOSTLY_MISSING).sum()), int((~IS_CANCER & ~MOSTLY_MISSING).sum())
    AUDIT_ASSOC = dict(n_cancer_missing=_n_cm, n_cancer_recorded=_n_cr, n_noncancer_missing=_n_bm, n_noncancer_recorded=_n_br,
                       rule_sensitivity=float(weighted_rate(~MOSTLY_MISSING, IS_CANCER, ONES_W)[0]),
                       rule_specificity=float(weighted_rate(MOSTLY_MISSING, ~IS_CANCER, ONES_W)[0]))
    for _name, _event, _within in (("rule_sensitivity", ~MOSTLY_MISSING, IS_CANCER), ("rule_specificity", MOSTLY_MISSING, ~IS_CANCER)):
        AUDIT_ASSOC[f"{_name}_ci_low"], AUDIT_ASSOC[f"{_name}_ci_high"], _ = percentile_ci(weighted_rate(_event, _within, BOOT_W))
    AUDIT_ASSOC["missing_by_class"] = {c: float(MOSTLY_MISSING[Y == k].mean()) for k, c in enumerate(CLASS_NAMES)}
    pd.DataFrame([{k: v for k, v in AUDIT_ASSOC.items() if k != "missing_by_class"}]).to_csv(DIRS["logs"] / "audit_missingness_association.csv", index=False)
    print(f"Images with at least {100 * CFG.AUDIT_ROW_MISSING_SHARE:.0f}% of metadata fields missing: {int(MOSTLY_MISSING.sum())} of {N}")
    display(pd.DataFrame({"metadata mostly missing": [_n_cm, _n_bm], "metadata recorded": [_n_cr, _n_br]}, index=["cancer", "non-cancer"]))
    print("Share of images with mostly missing metadata, by class: " + ", ".join(f"{c} {v:.3f}" for c, v in AUDIT_ASSOC["missing_by_class"].items()))
    print(f"Trivial rule 'mostly missing => non-cancer': sensitivity {AUDIT_ASSOC['rule_sensitivity']:.3f}, specificity {AUDIT_ASSOC['rule_specificity']:.3f}\n")

    # ---- (2) ablated metadata-only models on the same folds (out-of-fold predictions, repeat 0)
    def _audit_oof(feature_fn: Callable[[Dict[str, Any]], np.ndarray]) -> np.ndarray:
        """Out-of-fold probabilities of a LightGBM model trained on the features returned by feature_fn(prep)."""
        P = np.full((N, K), np.nan)
        for f in range(CFG.N_FOLDS):
            prep = PREP[("grouped", PRIMARY_REPEAT, f)]
            tr, va, te = get_split("grouped", PRIMARY_REPEAT, f)
            X = feature_fn(prep)
            clf = fit_lgbm(X[tr], Y[tr], X[va], Y[va], prep["class_weights"], run_seed(PRIMARY_REPEAT, f))
            P[te] = clf.predict_proba(X[te])
        assert np.isfinite(P).all()
        return P

    _variants: Dict[str, np.ndarray] = {}
    if "meta_gbm" in MODELS:
        _variants["All fields with missingness indicators (metadata-only baseline)"] = PROBS[("grouped", "meta_gbm")][PRIMARY_REPEAT]
    _variants["Missingness pattern only (one indicator per field, no values)"] = _audit_oof(lambda prep: MISSING_MATRIX.astype(np.float32))
    if LOW_MISSING_FIELDS:
        _variants[f"Low-missingness fields only ({', '.join(LOW_MISSING_FIELDS)})"] = (
            PROBS[("grouped", "meta_gbm_r")][PRIMARY_REPEAT] if "meta_gbm_r" in SENS_MODELS else
            _audit_oof(lambda prep: prep["X"][:, np.concatenate([prep["groups"][j] for j in LOW_MISSING_IDX])]))
    _rows = []
    for _name, _P in _variants.items():
        _pt, _bt = compute_metrics(Y, _P), bootstrap_metrics(Y, _P, BOOT_IDX)
        _row: Dict[str, Any] = dict(variant=_name)
        for _met in ("balanced_accuracy", "auroc_macro_ovr", "cancer_sensitivity", "cancer_specificity"):
            _row[_met] = _pt[_met]
            _row[f"{_met}_ci_low"], _row[f"{_met}_ci_high"], _ = percentile_ci(_bt[_met].to_numpy())
        _rows.append(_row)
    AUDIT_MODELS = pd.DataFrame(_rows)
    AUDIT_MODELS.to_csv(DIRS["logs"] / "audit_metadata_variants.csv", index=False)
    print("Metadata-only variants (pooled out-of-fold, same folds and settings):")
    display(AUDIT_MODELS[["variant", "balanced_accuracy", "auroc_macro_ovr", "cancer_sensitivity", "cancer_specificity"]].round(3))

    # ---- (3) every model, separately for images with recorded and with mostly missing metadata
    _rows = []
    for _m in MODELS:
        _pred = PROBS[("grouped", _m)][PRIMARY_REPEAT].argmax(axis=1)
        _correct = (_pred == Y).astype(np.float32)
        _pc = np.isin(_pred, CANCER_IDX)
        for _sname, _g in (("recorded", ~MOSTLY_MISSING), ("mostly missing", MOSTLY_MISSING)):
            _gw = BOOT_W * _g[None, :].astype(np.float32)
            _row = dict(model=_m, stratum=_sname, n_images=int(_g.sum()), n_cancer=int((_g & IS_CANCER).sum()),
                        n_non_cancer=int((_g & ~IS_CANCER).sum()), n_classes_present=int(len(np.unique(Y[_g]))),
                        balanced_accuracy=balanced_accuracy_fast(Y[_g], _pred[_g], K),
                        cancer_sensitivity=float(weighted_rate(_pc, IS_CANCER & _g, ONES_W)[0]),
                        cancer_specificity=float(weighted_rate(~_pc, ~IS_CANCER & _g, ONES_W)[0]))
            _row["balanced_accuracy_ci_low"], _row["balanced_accuracy_ci_high"], _ = percentile_ci(weighted_ba(_correct, _gw))
            _row["cancer_sensitivity_ci_low"], _row["cancer_sensitivity_ci_high"], _ = percentile_ci(weighted_rate(_pc, IS_CANCER & _g, BOOT_W))
            _row["cancer_specificity_ci_low"], _row["cancer_specificity_ci_high"], _ = percentile_ci(weighted_rate(~_pc, ~IS_CANCER & _g, BOOT_W))
            _rows.append(_row)
    AUDIT_STRATA = pd.DataFrame(_rows)
    AUDIT_STRATA.to_csv(DIRS["logs"] / "audit_strata.csv", index=False)
    print("All models by metadata availability (pooled out-of-fold):")
    display(AUDIT_STRATA.pivot(index="model", columns="stratum", values=["balanced_accuracy", "cancer_sensitivity", "cancer_specificity"]).reindex(MODELS).round(3))
if AUDIT_SKIP_REASON:
    print(f"[info] metadata-missingness audit skipped: {AUDIT_SKIP_REASON}. Tables T11 and T12 will be reported as skipped.")

### Sensitivity analysis: models restricted to low-missingness metadata (extension of Part 13)

**Purpose:** Estimate how much of the benefit of metadata and of fusion remains when the models cannot exploit which fields are missing.

**Why:** The audit above shows that the pattern of missing metadata is associated with the diagnosis. The main models (Table T2) can use that pattern, so their advantage over the image-only model mixes two things: clinical information and a recording artefact that would not exist in a clinic where the fields are collected for every patient. The audit measures the artefact; this analysis removes it and re-estimates the comparison.

**Method choice:** The metadata-only model and the early-concatenation, late-fusion, cross-attention and MetaBlock-style models are retrained with identical architectures, folds, seeds and settings, but their metadata input is restricted to the fields that are missing in at most `AUDIT_LOW_MISSING_MAX` of the images (`LOW_MISSING_FIELDS`). Restricting the field set is preferred over two alternatives: imputing the missing block without indicators leaves the block detectable (all its fields take the imputation value together), and analysing only the patients with recorded metadata changes the case mix (it removes two thirds of the benign lesions). The cost of the restriction is that informative fields are also dropped, so this is a **conservative lower bound** on what clinical metadata can add, not an estimate of the value of complete metadata. The models are marked "(R)". Reported: pooled out-of-fold metrics with patient-level bootstrap intervals; the paired difference in balanced accuracy to the image-only model and to the same model with all metadata (Holm-adjusted over the comparisons in this table); and cancer sensitivity and specificity among the images with recorded metadata.

**Output:** `SENS`; file `logs/sensitivity_restricted_metadata.csv`.

In [ ]:
SENS = pd.DataFrame()
SENS_SKIP_REASON = ""
if not CFG.RUN_RESTRICTED_SENSITIVITY:
    SENS_SKIP_REASON = "CFG.RUN_RESTRICTED_SENSITIVITY is False"
elif not SENS_MODELS:
    SENS_SKIP_REASON = "no restricted-metadata model could be defined (needs meta_gbm and at least one low-missingness field)"
else:
    _rec_mask = ~MOSTLY_MISSING
    _rows = []
    for _m in ["image_only"] + SENS_MODELS:
        _P = PROBS[("grouped", _m)][PRIMARY_REPEAT]
        _pc = np.isin(_P.argmax(axis=1), CANCER_IDX)
        _row: Dict[str, Any] = dict(model=_m, full_metadata_model=RESTRICTED_MODELS.get(_m, ""))
        for _met in ("balanced_accuracy", "auroc_macro_ovr", "cancer_sensitivity", "cancer_specificity"):
            _q = RQ1[(RQ1["scheme"] == "grouped") & (RQ1["model"] == _m) & (RQ1["metric"] == _met)].iloc[0]
            _row[_met], _row[f"{_met}_ci_low"], _row[f"{_met}_ci_high"] = _q["pooled"], _q["ci_low"], _q["ci_high"]
        for _name, _event, _within in (("recorded_cancer_sensitivity", _pc, IS_CANCER & _rec_mask), ("recorded_cancer_specificity", ~_pc, ~IS_CANCER & _rec_mask)):
            _row[_name] = float(weighted_rate(_event, _within, ONES_W)[0])
            _row[f"{_name}_ci_low"], _row[f"{_name}_ci_high"], _ = percentile_ci(weighted_rate(_event, _within, BOOT_W))
        _row["n_recorded_cancer"], _row["n_recorded_non_cancer"] = int((IS_CANCER & _rec_mask).sum()), int((~IS_CANCER & _rec_mask).sum())
        for _tag, _ref in (("vs_image_only", "image_only"), ("vs_full_metadata", RESTRICTED_MODELS.get(_m))):
            if _m == "image_only" or _ref is None:
                _res = dict(diff=np.nan, ci_low=np.nan, ci_high=np.nan, p=np.nan)
            else:
                _res = paired_bootstrap(BOOT[("grouped", _m)]["balanced_accuracy"].to_numpy(), BOOT[("grouped", _ref)]["balanced_accuracy"].to_numpy(),
                                        POOLED[("grouped", _m)]["balanced_accuracy"], POOLED[("grouped", _ref)]["balanced_accuracy"])
            _row.update({f"ba_diff_{_tag}": _res["diff"], f"ba_diff_{_tag}_ci_low": _res["ci_low"], f"ba_diff_{_tag}_ci_high": _res["ci_high"], f"p_{_tag}": _res["p"]})
        _rows.append(_row)
    SENS = pd.DataFrame(_rows)
    _all_p = holm_adjust(np.concatenate([SENS["p_vs_image_only"].to_numpy(), SENS["p_vs_full_metadata"].to_numpy()]))
    SENS["p_holm_vs_image_only"], SENS["p_holm_vs_full_metadata"] = _all_p[:len(SENS)], _all_p[len(SENS):]
    SENS.to_csv(DIRS["logs"] / "sensitivity_restricted_metadata.csv", index=False)
    print(f"Restricted metadata fields: {LOW_MISSING_FIELDS}")
    display(SENS[["model", "balanced_accuracy", "auroc_macro_ovr", "cancer_sensitivity", "cancer_specificity", "ba_diff_vs_image_only",
                  "p_holm_vs_image_only", "ba_diff_vs_full_metadata", "p_holm_vs_full_metadata",
                  "recorded_cancer_sensitivity", "recorded_cancer_specificity"]].round(3))
if SENS_SKIP_REASON:
    print(f"[info] restricted-metadata sensitivity analysis skipped: {SENS_SKIP_REASON}. Table T13 will be reported as skipped.")

## Part 14. Explainability (supporting evidence only)

**Purpose:** Produce Grad-CAM maps for correctly and incorrectly classified test images of every class, and aggregate the SHAP contributions of the metadata fields.

**Why:** Saliency maps and attributions are a plausibility check: they can reveal obvious shortcuts (for example, attention on rulers, ink marks or image borders) and show which clinical fields the metadata model relies on. **They do not establish clinical validity.** A map that looks reasonable does not show that the model uses clinically correct reasoning, saliency methods can be insensitive to the model's parameters, and attributions describe the model, not the disease. They are reported as supporting material only.

**Method choice:**
- **Grad-CAM** (Selvaraju et al., *ICCV*, 2017) on the last feature map of the backbone, for the class the model predicted. Each image is explained by the fold model for which it was a **test** image.
- **Example selection is rule-based, not hand-picked:** for every true class, among the test images the model classified correctly, the image with the *median* confidence is taken (ties broken by image id); the same rule is applied to the incorrectly classified images. The median is used so that examples are typical rather than the most flattering or most extreme.
- **SHAP:** exact TreeSHAP contributions of the metadata-only model, computed on each fold's test images in Part 6, summed over the encoded columns of each field (contributions are additive, so the sum is the field's contribution), and summarised as the mean absolute contribution over images and classes. The spread across folds is reported. Fields that are correlated share credit in ways that are not unique, and a high value means the model uses the field, not that the field is causal.

**Output:** `GRADCAM_PANEL` (images, maps, labels), `SHAP_IMPORTANCE`; files `logs/gradcam_selection.csv`, `logs/shap_importance.csv`.

In [ ]:
def gradcam_map(model: FusionNet, img_uint8: torch.Tensor, meta: torch.Tensor, target: int) -> np.ndarray:
    """Grad-CAM heat-map in [0, 1] at IMG_SIZE resolution for one image (evaluation view) and target class."""
    model.eval()
    with torch.no_grad():
        tokens = model.backbone(img_uint8)
    tokens = tokens.detach().float().requires_grad_(True)
    logits = model.forward_from_tokens(tokens, meta)
    model.zero_grad(set_to_none=True)
    logits[0, target].backward()
    n_prefix = model.backbone.n_prefix
    act, grad = tokens[0, n_prefix:].detach(), tokens.grad[0, n_prefix:]
    cam = torch.relu((act * grad.mean(dim=0, keepdim=True)).sum(dim=1))
    side = int(round(math.sqrt(cam.numel())))
    assert side * side == cam.numel(), "The token grid is not square; Grad-CAM cannot be reshaped."
    cam = F.interpolate(cam.view(1, 1, side, side), size=(CFG.IMG_SIZE, CFG.IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
    cam = cam / cam.max() if float(cam.max()) > 0 else cam
    return cam.cpu().numpy()


GRADCAM_PANEL: Optional[Dict[str, Any]] = None
GRADCAM_SKIP_REASON = ""
if not CFG.RUN_EXPLAIN:
    GRADCAM_SKIP_REASON = "CFG.RUN_EXPLAIN is False"
elif CFG.GRADCAM_MODEL not in DEEP_MODELS:
    GRADCAM_SKIP_REASON = f"CFG.GRADCAM_MODEL = {CFG.GRADCAM_MODEL} is not a trained network"
else:
    _gm = CFG.GRADCAM_MODEL
    _P = PROBS[("grouped", _gm)][PRIMARY_REPEAT]
    _pred, _conf = _P.argmax(axis=1), _P.max(axis=1)
    _picks = []
    for _k, _c in enumerate(CLASS_NAMES):
        for _outcome, _mask in (("correct", (Y == _k) & (_pred == _k)), ("incorrect", (Y == _k) & (_pred != _k))):
            _cand = np.flatnonzero(_mask)
            if len(_cand) == 0:
                _picks.append(dict(true_class=_c, outcome=_outcome, n_candidates=0, row=-1))
                continue
            _order = sorted(_cand, key=lambda i: (_conf[i], IMG_IDS[i]))
            _picks.append(dict(true_class=_c, outcome=_outcome, n_candidates=len(_cand), row=int(_order[(len(_order) - 1) // 2])))
    _needed_folds = sorted({int(TEST_FOLD["grouped"][PRIMARY_REPEAT][p["row"]]) for p in _picks if p["row"] >= 0})
    _missing_w = [f for f in _needed_folds if not run_files("grouped", _gm, PRIMARY_REPEAT, f)["best"].is_file()]
    if _missing_w:
        GRADCAM_SKIP_REASON = (f"best weights of {_gm} are not on disk for folds {_missing_w} "
                               "(they were not kept when it was trained: set KEEP_BEST_WEIGHTS to 'gradcam_model' or 'all', delete that model's "
                               "_DONE.json files for these folds and rerun Part 6)")
    else:
        _eval_tf = T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE), antialias=True)
        for _f in _needed_folds:
            _prep = PREP[("grouped", PRIMARY_REPEAT, _f)]
            _net = build_model(MODEL_KIND[_gm][0], _prep).to(DEVICE)
            _net.load_state_dict(torch.load(run_files("grouped", _gm, PRIMARY_REPEAT, _f)["best"], map_location=DEVICE, weights_only=False)["model"])
            for _p in _picks:
                if _p["row"] < 0 or int(TEST_FOLD["grouped"][PRIMARY_REPEAT][_p["row"]]) != _f:
                    continue
                _i = _p["row"]
                _img = _eval_tf(torch.from_numpy(np.array(IMAGES[_i], copy=True)).permute(2, 0, 1)).unsqueeze(0).to(DEVICE)
                _meta = torch.from_numpy(_prep["X"][_i:_i + 1]).float().to(DEVICE)
                _p.update(img_id=IMG_IDS[_i], fold=_f, pred_class=CLASS_NAMES[_pred[_i]], confidence=float(_conf[_i]),
                          image=_img[0].permute(1, 2, 0).cpu().numpy(), cam=gradcam_map(_net, _img, _meta, int(_pred[_i])))
            del _net
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        GRADCAM_PANEL = dict(model=_gm, picks=_picks)
        pd.DataFrame([{k: v for k, v in p.items() if k not in ("image", "cam")} for p in _picks]).to_csv(DIRS["logs"] / "gradcam_selection.csv", index=False)
        print(f"Grad-CAM computed for {sum(p['row'] >= 0 for p in _picks)} rule-selected test images of '{_gm}'.")
if GRADCAM_SKIP_REASON:
    print(f"[info] Grad-CAM skipped: {GRADCAM_SKIP_REASON}. Figure F10 will be reported as skipped.")

SHAP_IMPORTANCE: Optional[pd.DataFrame] = None
SHAP_SKIP_REASON = ""
if not CFG.RUN_EXPLAIN:
    SHAP_SKIP_REASON = "CFG.RUN_EXPLAIN is False"
elif "meta_gbm" not in MODELS:
    SHAP_SKIP_REASON = "meta_gbm is not in MODELS_TO_RUN"
else:
    _per_fold = []
    for _f in range(CFG.N_FOLDS):
        _z = np.load(run_files("grouped", "meta_gbm", PRIMARY_REPEAT, _f)["shap"], allow_pickle=False)
        assert list(_z["fields"]) == FIELD_NAMES, "Stored SHAP fields differ from the current metadata fields."
        _per_fold.append(np.abs(_z["shap"]).mean(axis=(0, 1)))          # mean |contribution| over images and classes
    _pf = np.stack(_per_fold)                                           # (folds, fields)
    SHAP_IMPORTANCE = pd.DataFrame({"field": FIELD_NAMES, "mean_abs_shap": _pf.mean(axis=0), "sd_across_folds": _pf.std(axis=0, ddof=1),
                                    "n_folds": _pf.shape[0]}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    SHAP_IMPORTANCE.to_csv(DIRS["logs"] / "shap_importance.csv", index=False)
    print("Metadata field importance (mean |SHAP|, log-odds units, metadata-only model, test folds):")
    display(SHAP_IMPORTANCE.round(4))
if SHAP_SKIP_REASON:
    print(f"[info] SHAP summary skipped: {SHAP_SKIP_REASON}. Figure F11 will be reported as skipped.")

## Part 15. Export of tables and figures

**Purpose:** Define the single plotting style used by every figure, the fixed colour / marker / line-style assignment of every model, and the function that saves a figure in both journal widths and all required formats.

**Why:** Figures must be inserted into Word at their final physical size without rescaling, stay legible in greyscale, and be identical in style across the paper.

**Method choice:**
- **Size:** each figure is created at exactly 90 mm or 190 mm width and saved without `bbox_inches="tight"` (which would change the physical size); layout is handled by constrained layout. Height is set per figure and asserted to be at most 230 mm.
- **Layout deviation (stated):** the specification asks for a horizontal layout. Multi-panel figures keep panels side by side in the 190 mm version. In the 90 mm version, panels are stacked vertically, because two or more panels side by side in 90 mm cannot keep tick labels at 7 pt or larger. Some single-column versions are therefore taller than wide.
- **Formats:** PNG (RGB, white background, 600 dpi metadata) for Word; TIFF with LZW compression for submission; PDF with embedded TrueType fonts (`pdf.fonttype = 42`) and SVG. SVG text is converted to outlines by default so that it renders identically on machines without the font (`SVG_TEXT_AS_PATHS`).
- **Text and colour:** Arial if installed (otherwise Liberation Sans, which has the same metrics, then DejaVu Sans, with a printed warning); black text and axes; no transparency anywhere. Okabe–Ito colour-blind-safe palette (Okabe & Ito, 2008, *Color Universal Design*; Wong, *Nature Methods*, 2011). Each model family has one colour; the standard and modality-dropout variants share the colour and differ in line style (solid / dashed) and marker, so every series is identifiable without colour.
- **Enforcement:** `save_figure` audits every artist (font size, text colour, alpha) and refuses to save a figure that violates the standards; the audit result is stored for the QC cell.

**Output:** `MODEL_STYLE`, `new_figure()`, `save_figure()`, `FIG_REGISTRY`; the font actually used is printed and recorded.

In [ ]:
import matplotlib.font_manager as font_manager
from matplotlib.collections import Collection
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

for _pattern in ("/usr/share/fonts/truetype/msttcorefonts/*.ttf", "/usr/share/fonts/truetype/liberation*/*.ttf"):
    for _ttf in glob.glob(_pattern):
        font_manager.fontManager.addfont(_ttf)        # register fonts installed after matplotlib built its cache
_available_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT_USED = next((f for f in ("Arial", "Liberation Sans", "DejaVu Sans") if f in _available_fonts), "DejaVu Sans")
if FONT_USED != "Arial":
    print(f"[warning] Arial is not available in this runtime; figures use '{FONT_USED}'. "
          "For the final submission figures, rerun this part in a runtime where the Arial installation in Part 1 succeeds.")
ENV_INFO["figure_font"] = FONT_USED

OKABE_ITO = dict(black="#000000", orange="#E69F00", sky_blue="#56B4E9", bluish_green="#009E73",
                 yellow="#F0E442", blue="#0072B2", vermillion="#D55E00", reddish_purple="#CC79A7")
_FAMILY_STYLE = {  # family -> (colour, marker standard, marker modality-dropout, label)
    "image_only": (OKABE_ITO["black"], "o", "o", "Image only"),
    "meta_gbm": (OKABE_ITO["orange"], "s", "p", "Metadata only (GBM)"),
    "early_concat": (OKABE_ITO["blue"], "^", "<", "Early concatenation"),
    "late_fusion": (OKABE_ITO["bluish_green"], "D", "d", "Late fusion"),
    "cross_attn": (OKABE_ITO["vermillion"], "v", ">", "Cross-attention"),
    "metablock": (OKABE_ITO["reddish_purple"], "P", "X", "MetaBlock-style"),
}
MODEL_STYLE: Dict[str, Dict[str, Any]] = {}
for _m, (_fam, _md) in MODEL_KIND.items():
    _col, _mk, _mk_md, _lab = _FAMILY_STYLE[_fam]
    MODEL_STYLE[_m] = dict(color=_col, marker=_mk_md if _md else _mk, linestyle="--" if _md else "-",
                           label=_lab + (" + MD" if _md else "") + (" (R)" if _m in RESTRICTED_MODELS else ""))
MODEL_LABEL = {m: s["label"] for m, s in MODEL_STYLE.items()}
MD_NOTE = "MD, modality-dropout training (for the gradient-boosting model: training with missingness-augmented copies)"
_class_colors = [OKABE_ITO[c] for c in ("black", "orange", "blue", "bluish_green", "vermillion", "reddish_purple", "sky_blue")]
_class_markers = ["o", "s", "^", "D", "v", "P", "X"]
_class_lines = ["-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (1, 1))]
CLASS_STYLE = {c: dict(color=_class_colors[i % 7], marker=_class_markers[i % 7], linestyle=_class_lines[i % 7]) for i, c in enumerate(CLASS_NAMES)}
HEAT_CMAP = LinearSegmentedColormap.from_list("white_skyblue", ["#FFFFFF", OKABE_ITO["sky_blue"]])   # black text readable on every cell

plt.rcdefaults()
matplotlib.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [FONT_USED, "DejaVu Sans"], "font.size": CFG.FONT_AXIS_PT, "font.weight": "normal",
    "axes.labelsize": CFG.FONT_AXIS_PT, "axes.titlesize": CFG.FONT_PANEL_PT, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.fontsize": CFG.FONT_AXIS_PT, "xtick.labelsize": CFG.FONT_TICK_PT, "ytick.labelsize": CFG.FONT_TICK_PT,
    "text.color": "#000000", "axes.labelcolor": "#000000", "axes.edgecolor": "#000000", "xtick.color": "#000000", "ytick.color": "#000000",
    "axes.linewidth": 0.8, "xtick.major.width": 0.8, "ytick.major.width": 0.8, "xtick.major.size": 3.0, "ytick.major.size": 3.0,
    "lines.linewidth": 1.6, "lines.markersize": 5.0, "lines.markeredgecolor": "#000000", "lines.markeredgewidth": 0.6,
    "errorbar.capsize": 2.5, "hatch.linewidth": 0.6, "axes.grid": False,
    "legend.framealpha": 1.0, "legend.edgecolor": "#000000", "legend.fancybox": False, "legend.frameon": True,
    "figure.facecolor": "#FFFFFF", "axes.facecolor": "#FFFFFF", "savefig.facecolor": "#FFFFFF", "savefig.transparent": False,
    "figure.dpi": 100, "savefig.dpi": CFG.FIG_DPI, "pdf.fonttype": 42, "ps.fonttype": 42,
    "svg.fonttype": "path" if CFG.SVG_TEXT_AS_PATHS else "none", "figure.constrained_layout.use": True,
})
ERR_KW = dict(elinewidth=1.0, capsize=2.5, capthick=1.0, ecolor="#000000")
FIG_FORMATS = ("png", "tiff", "pdf", "svg")
FIG_REGISTRY: Dict[str, Dict[str, Any]] = {}
FIG_SKIPPED: Dict[str, str] = {}
CAPTIONS: Dict[str, str] = {}


def mm_to_px(mm: float) -> int:
    """Pixel count of a physical length at the export resolution."""
    return int(round(mm / 25.4 * CFG.FIG_DPI))


def new_figure(width_key: str, height_mm: float, nrows: int = 1, ncols: int = 1, **kwargs: Any):
    """Create a figure at the exact final physical size (width from CFG.FIG_WIDTHS_MM, given height in mm)."""
    assert height_mm <= CFG.FIG_MAX_HEIGHT_MM + 1e-9, f"Figure height {height_mm} mm exceeds the maximum {CFG.FIG_MAX_HEIGHT_MM} mm."
    # + 0.01 px guards against truncation when matplotlib converts inches to whole pixels
    w_in = (mm_to_px(CFG.FIG_WIDTHS_MM[width_key]) + 0.01) / CFG.FIG_DPI
    h_in = (mm_to_px(height_mm) + 0.01) / CFG.FIG_DPI
    fig, axes = plt.subplots(nrows, ncols, figsize=(w_in, h_in), **kwargs)
    return fig, axes


def audit_figure(fig) -> Dict[str, Any]:
    """Inspect every visible artist: smallest font size, smallest alpha, and any non-black text."""
    min_font, min_alpha, problems = float("inf"), 1.0, []

    def _alpha_of(color: Any) -> Optional[float]:
        try:
            return float(to_rgba(color)[3])
        except (ValueError, TypeError):
            return None

    for art in fig.findobj():
        if not art.get_visible():
            continue
        a = art.get_alpha()
        if a is not None and np.ndim(a) == 0 and float(a) < 1.0:
            min_alpha = min(min_alpha, float(a))
        if isinstance(art, matplotlib.text.Text):
            if art.get_text().strip():
                min_font = min(min_font, float(art.get_fontsize()))
                if tuple(np.round(to_rgba(art.get_color())[:3], 3)) != (0.0, 0.0, 0.0):
                    problems.append(f"non-black text '{art.get_text()[:20]}'")
        elif isinstance(art, Line2D):
            for col in (art.get_color(), art.get_markerfacecolor(), art.get_markeredgecolor()):
                al = _alpha_of(col)
                if al is not None and 0.0 < al < 1.0:
                    min_alpha = min(min_alpha, al)
        elif isinstance(art, Patch):
            for col in (art.get_facecolor(), art.get_edgecolor()):
                al = _alpha_of(col)
                if al is not None and 0.0 < al < 1.0:
                    min_alpha = min(min_alpha, al)
        elif isinstance(art, Collection):
            for arr in (art.get_facecolor(), art.get_edgecolor()):
                arr = np.atleast_2d(np.asarray(arr, dtype=float))
                if arr.size and arr.shape[1] == 4:
                    partial = arr[(arr[:, 3] > 0.0) & (arr[:, 3] < 1.0), 3]
                    if partial.size:
                        min_alpha = min(min_alpha, float(partial.min()))
    return dict(min_font_pt=min_font if np.isfinite(min_font) else None, min_alpha=min_alpha, problems=problems)


def save_figure(fig, fig_id: str, width_key: str) -> None:
    """Audit a figure and save it as PNG, TIFF (LZW), PDF and SVG at its final size; record everything for QC."""
    w_mm = CFG.FIG_WIDTHS_MM[width_key]
    fig.canvas.draw()                                   # finalise constrained layout before auditing
    audit = audit_figure(fig)
    if audit["min_font_pt"] is not None and audit["min_font_pt"] < CFG.MIN_FONT_PT - 1e-6:
        raise AssertionError(f"{fig_id} ({width_key}): smallest font {audit['min_font_pt']} pt is below {CFG.MIN_FONT_PT} pt.")
    if audit["min_alpha"] < 1.0:
        raise AssertionError(f"{fig_id} ({width_key}): an artist has alpha {audit['min_alpha']} < 1 (transparency is not allowed).")
    if audit["problems"]:
        raise AssertionError(f"{fig_id} ({width_key}): {audit['problems'][:5]}")
    base = DIRS[f"fig_{width_key}"] / f"{fig_id}_{width_key}"
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=CFG.FIG_DPI)
    buf.seek(0)
    rgba = Image.open(buf).convert("RGBA")
    rgb = Image.new("RGB", rgba.size, (255, 255, 255))
    rgb.paste(rgba, mask=rgba.split()[3])               # flatten onto white: no alpha channel in the output
    assert rgb.width == mm_to_px(w_mm), f"{fig_id}: rendered width {rgb.width} px, expected {mm_to_px(w_mm)} px."
    rgb.save(str(base) + ".png", dpi=(CFG.FIG_DPI, CFG.FIG_DPI))
    rgb.save(str(base) + ".tiff", compression="tiff_lzw", dpi=(CFG.FIG_DPI, CFG.FIG_DPI))
    fig.savefig(str(base) + ".pdf", dpi=CFG.FIG_DPI)
    fig.savefig(str(base) + ".svg", dpi=CFG.FIG_DPI)
    FIG_REGISTRY.setdefault(fig_id, {})[width_key] = dict(
        files={ext: str(base) + f".{ext}" for ext in FIG_FORMATS}, width_mm=w_mm, width_px=rgb.width, height_px=rgb.height,
        height_mm=round(rgb.height / CFG.FIG_DPI * 25.4, 2), min_font_pt=audit["min_font_pt"], min_alpha=audit["min_alpha"])


def render_both_widths(fig_id: str, builder: Callable[[str], Any], caption: str) -> None:
    """Build, save and display a figure in both widths, and store its factual draft caption."""
    for width_key in CFG.FIG_WIDTHS_MM:
        fig = builder(width_key)
        save_figure(fig, fig_id, width_key)
        if width_key == "double_column":
            plt.show()
        plt.close(fig)
    CAPTIONS[fig_id] = caption
    (DIRS["captions"] / f"{fig_id}_caption.txt").write_text(caption + "\n", encoding="utf-8")
    print(f"[saved] {fig_id}: " + "; ".join(f"{k} {v['width_px']}x{v['height_px']} px" for k, v in FIG_REGISTRY[fig_id].items()))


def skip_figure(fig_id: str, reason: str) -> None:
    """Record that a figure was not produced and why (reported by the QC cell as SKIP, never silently)."""
    FIG_SKIPPED[fig_id] = reason
    print(f"[skipped] {fig_id}: {reason}")


def panel_label(ax, letter: str, text: str = "") -> None:
    """Bold panel label '(a)' at the top-left of an axes, optionally followed by a short identifier."""
    ax.set_title(f"({letter})" + (f" {text}" if text else ""), loc="left", fontsize=CFG.FONT_PANEL_PT, fontweight="bold", color="#000000")


def model_handle(m: str) -> Line2D:
    """Legend handle with the fixed colour, marker and line style of a model."""
    s = MODEL_STYLE[m]
    return Line2D([], [], color=s["color"], marker=s["marker"], linestyle=s["linestyle"], label=s["label"])


CI_PCT = int(round(100 * CFG.CI_LEVEL))
CI_TEXT = (f"{CI_PCT}% CI from patient-level bootstrap ({CFG.N_BOOT} resamples of {N_CV_GROUPS} "
           + ("patients)" if GROUPS_ARE_PATIENTS else "patient groups; patients sharing an identical image form one group)"))
OOF_TEXT = f"pooled out-of-fold predictions of {CFG.N_FOLDS}-fold patient-grouped cross-validation (n = {N} images)"
print(f"Figure font: {FONT_USED}. Widths: {CFG.FIG_WIDTHS_MM} mm = "
      f"{ {k: mm_to_px(v) for k, v in CFG.FIG_WIDTHS_MM.items()} } px at {CFG.FIG_DPI} dpi.")

**Purpose:** Figures F1–F4: dataset overview, model comparison with confidence intervals, confusion matrices, and one-vs-rest ROC curves.

**Why:** These figures carry the dataset description and the RQ1 evidence.

**Method choice:** F2 is a dot-and-interval (forest-style) plot rather than a bar chart, because the quantities are estimates with intervals and bars would hide the uncertainty and force a zero baseline. Confusion matrices are row-normalised (each row sums to 1, so the diagonal is the per-class recall) with the value printed in every cell on a white-to-blue scale that keeps black text readable at every value. ROC curves are computed from the pooled out-of-fold probabilities; the AUROC shown in each legend entry is the pooled value from Part 8. All captions are generated from the computed quantities and describe only what is shown.

**Output:** `results/figures/{single_column,double_column}/F1…F4_*.{png,tiff,pdf,svg}` and `results/captions/F1…F4_caption.txt`.

In [ ]:
def _annotate_heatmap(ax, values: np.ndarray, rotate: bool = False) -> None:
    """Print every cell value (two decimals) in black."""
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center", fontsize=CFG.FONT_TICK_PT, color="#000000",
                    rotation=90 if rotate else 0)


def build_F1(width_key: str):
    """Dataset overview: (a) images per class, (b) images per patient, (c) metadata missingness by class."""
    double = width_key == "double_column"
    miss = EDA["missing_by_class"].to_numpy()
    ipp = EDA["images_per_patient"]
    if double:
        h = min(CFG.FIG_MAX_HEIGHT_MM, 70 + 12 + 7.0 * K + 30)
        fig = plt.figure(figsize=((mm_to_px(190.0) + 0.01) / CFG.FIG_DPI, (mm_to_px(h) + 0.01) / CFG.FIG_DPI))
        gs = fig.add_gridspec(2, 2, height_ratios=[1.0, max(0.6, 0.11 * K + 0.25)])
        ax_a, ax_b, ax_c = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])
    else:
        h = min(CFG.FIG_MAX_HEIGHT_MM, 50 + 50 + 5.0 * N_FIELDS + 25)
        fig = plt.figure(figsize=((mm_to_px(90.0) + 0.01) / CFG.FIG_DPI, (mm_to_px(h) + 0.01) / CFG.FIG_DPI))
        gs = fig.add_gridspec(3, 1, height_ratios=[1.0, 1.0, max(1.2, 0.1 * N_FIELDS + 0.3)])
        ax_a, ax_b, ax_c = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
    counts = EDA["class_counts"]["n_images"].to_numpy()
    bars = ax_a.bar(CLASS_NAMES, counts, color=OKABE_ITO["blue"], edgecolor="#000000", linewidth=0.8)
    ax_a.bar_label(bars, labels=[str(int(c)) for c in counts], fontsize=CFG.FONT_TICK_PT, color="#000000", padding=1.5)
    ax_a.set_ylim(0, counts.max() * 1.15)
    ax_a.set_xlabel("Diagnostic class")
    ax_a.set_ylabel("Number of images")
    panel_label(ax_a, "a")
    vals, freq = np.unique(ipp, return_counts=True)
    bars = ax_b.bar([str(int(v)) for v in vals], freq, color=OKABE_ITO["orange"], edgecolor="#000000", linewidth=0.8)
    ax_b.bar_label(bars, labels=[str(int(f)) for f in freq], fontsize=CFG.FONT_TICK_PT, color="#000000", padding=1.5)
    ax_b.set_ylim(0, freq.max() * 1.15)
    ax_b.set_xlabel("Images per patient")
    ax_b.set_ylabel("Number of patients")
    panel_label(ax_b, "b")
    if double:
        ax_c.imshow(miss, cmap=HEAT_CMAP, vmin=0.0, vmax=1.0, aspect="auto", interpolation="nearest")
        _annotate_heatmap(ax_c, miss, rotate=N_FIELDS > 24)
        ax_c.set_xticks(range(N_FIELDS), FIELD_NAMES, rotation=90)
        ax_c.set_yticks(range(K), CLASS_NAMES)
        ax_c.set_xlabel("Metadata field")
        ax_c.set_ylabel("Diagnostic class")
    else:
        ax_c.imshow(miss.T, cmap=HEAT_CMAP, vmin=0.0, vmax=1.0, aspect="auto", interpolation="nearest")
        _annotate_heatmap(ax_c, miss.T)
        ax_c.set_yticks(range(N_FIELDS), FIELD_NAMES)
        ax_c.set_xticks(range(K), CLASS_NAMES)
        ax_c.set_ylabel("Metadata field")
        ax_c.set_xlabel("Diagnostic class")
    panel_label(ax_c, "c")
    return fig


render_both_widths("F1", build_F1, (
    f"Figure F1. Overview of the dataset as loaded (n = {N} images, {OBSERVED['n_patients']} patients, {N_LESION_PAIRS} lesions, {K} classes). "
    f"(a) Number of images per diagnostic class. (b) Number of patients by number of images per patient. "
    f"(c) Fraction of images with a missing value for each of the {N_FIELDS} metadata fields used as model inputs, by diagnostic class; "
    f"the value is printed in each cell (0 = never missing, 1 = always missing)."))

_F2_METRICS = ["balanced_accuracy", "auroc_macro_ovr"]


def build_F2(width_key: str):
    """Per-model discrimination with patient-level bootstrap intervals (pooled out-of-fold estimates)."""
    double = width_key == "double_column"
    h = 22 + 6.0 * len(MODELS) if double else 2 * (22 + 6.0 * len(MODELS))
    fig, axes = new_figure(width_key, h, 1 if double else 2, 2 if double else 1, sharey=True)
    ypos = np.arange(len(MODELS))[::-1]
    for ax, met, letter in zip(np.atleast_1d(axes), _F2_METRICS, "ab"):
        sub = RQ1[(RQ1["scheme"] == "grouped") & (RQ1["metric"] == met)].set_index("model")
        for y, m in zip(ypos, MODELS):
            r, s = sub.loc[m], MODEL_STYLE[m]
            xerr = None if not np.isfinite(r["ci_low"]) else [[r["pooled"] - r["ci_low"]], [r["ci_high"] - r["pooled"]]]
            ax.errorbar(r["pooled"], y, xerr=xerr, fmt=s["marker"], color=s["color"], markerfacecolor=s["color"], **ERR_KW)
        ax.set_yticks(ypos, [MODEL_LABEL[m] for m in MODELS])
        ax.set_xlabel(METRIC_LABELS[met])
        ax.set_ylim(-0.7, len(MODELS) - 0.3)
        panel_label(ax, letter)
    return fig


render_both_widths("F2", build_F2, (
    f"Figure F2. Discrimination of the {len(MODELS)} models on {OOF_TEXT}. (a) Balanced accuracy. (b) Macro-averaged one-vs-rest AUROC. "
    f"Markers show the pooled estimate; error bars show the {CI_TEXT}. {MD_NOTE}."))

_F3_MODELS = [SELECTED[k] for k in ("image_only", "best_fusion") if k in SELECTED]


def build_F3(width_key: str):
    """Row-normalised confusion matrices of the image-only model and the validation-selected fusion model."""
    double = width_key == "double_column"
    n = len(_F3_MODELS)
    fig, axes = new_figure(width_key, 92 if double else min(CFG.FIG_MAX_HEIGHT_MM, 88 * n), 1 if double else n, n if double else 1)
    for ax, m, letter in zip(np.atleast_1d(axes), _F3_MODELS, "ab"):
        cm = skm.confusion_matrix(Y, PROBS[("grouped", m)][PRIMARY_REPEAT].argmax(axis=1), labels=list(range(K))).astype(float)
        cmn = cm / cm.sum(axis=1, keepdims=True)
        ax.imshow(cmn, cmap=HEAT_CMAP, vmin=0.0, vmax=1.0, interpolation="nearest")
        _annotate_heatmap(ax, cmn)
        ax.set_xticks(range(K), CLASS_NAMES)
        ax.set_yticks(range(K), CLASS_NAMES)
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        panel_label(ax, letter, MODEL_LABEL[m])
    return fig


render_both_widths("F3", build_F3, (
    "Figure F3. Row-normalised confusion matrices on " + OOF_TEXT + ". "
    + " ".join(f"({l}) {MODEL_LABEL[m]}." for l, m in zip("ab", _F3_MODELS))
    + " Each row sums to 1; diagonal cells are per-class recall. The fusion model shown is the one with the highest mean validation balanced accuracy"
      " (selected without using test folds)."))


def build_F4(width_key: str):
    """One-vs-rest ROC curves per class for the same two models."""
    double = width_key == "double_column"
    n = len(_F3_MODELS)
    # Each panel has its own legend (the AUROC values differ), placed in a separate axes below the plot so
    # that it can neither overlap the curves nor be cut off at the figure edge.
    if double:
        fig, axes = new_figure(width_key, 118, 2, n, squeeze=False, gridspec_kw=dict(height_ratios=[5.0, 1.0]))
        pairs = [(axes[0, i], axes[1, i]) for i in range(n)]
    else:
        fig, axes = new_figure(width_key, min(CFG.FIG_MAX_HEIGHT_MM, 114 * n), 2 * n, 1, squeeze=False,
                               gridspec_kw=dict(height_ratios=[5.0, 1.0] * n))
        pairs = [(axes[2 * i, 0], axes[2 * i + 1, 0]) for i in range(n)]
    grid = np.linspace(0.05, 0.95, 7)
    for (ax, lax), m, letter in zip(pairs, _F3_MODELS, "ab"):
        P = PROBS[("grouped", m)][PRIMARY_REPEAT]
        handles = []
        for k, c in enumerate(CLASS_NAMES):
            fpr, tpr, _ = skm.roc_curve(Y == k, P[:, k])
            st = CLASS_STYLE[c]
            ax.plot(fpr, tpr, color=st["color"], linestyle=st["linestyle"])
            ax.plot(grid, np.interp(grid, fpr, tpr), linestyle="none", marker=st["marker"], color=st["color"], markerfacecolor=st["color"])
            handles.append(Line2D([], [], color=st["color"], linestyle=st["linestyle"], marker=st["marker"],
                                  label=f"{c} ({POOLED[('grouped', m)][f'auroc_{c}']:.3f})"))
        ax.plot([0, 1], [0, 1], color="#000000", linestyle=":", linewidth=1.0)
        ax.set_xlim(-0.01, 1.01)
        ax.set_ylim(-0.01, 1.01)
        ax.set_aspect("equal")
        ax.set_xlabel("False-positive rate")
        ax.set_ylabel("True-positive rate")
        lax.axis("off")
        lax.legend(handles=handles, loc="center", ncol=3, title="Class (AUROC)",
                   title_fontsize=CFG.FONT_AXIS_PT, handlelength=2.6, columnspacing=1.0)
        panel_label(ax, letter, MODEL_LABEL[m])
    return fig


render_both_widths("F4", build_F4, (
    "Figure F4. One-vs-rest ROC curves for each diagnostic class on " + OOF_TEXT + ". "
    + " ".join(f"({l}) {MODEL_LABEL[m]}." for l, m in zip("ab", _F3_MODELS))
    + " Legends give the pooled one-vs-rest AUROC per class. The dotted diagonal is the chance line. Markers are placed at fixed false-positive rates"
      " to distinguish the curves without colour."))

**Purpose:** Figures F5–F9: reliability diagrams before and after temperature scaling, risk–coverage curves, robustness curves, the fairness forest plot and the leakage comparison.

**Why:** These figures carry the evidence for RQ3, RQ2 and RQ4 and for the leakage analysis.

**Method choice:** Reliability diagrams use lines through the occupied bins (accuracy against mean confidence) rather than translucent bars, so that "before" and "after" can be overlaid without transparency. Robustness panels share one y-axis range so that the fusion families can be compared directly, and each repeats the image-only and metadata-only curves as fixed references. Uncertainty is drawn as capped error bars rather than shaded bands, which satisfies the no-transparency rule. In the fairness plot, subgroups below the minimum size are drawn with open markers and marked with a dagger so that they are visibly not for interpretation.

**Output:** `results/figures/.../F5…F9_*` and captions; figures that depend on a disabled option are recorded as skipped.

In [ ]:
_F5_MODELS = [SELECTED[k] for k in ("image_only", "meta_only", "best_fusion") if k in SELECTED]


def build_F5(width_key: str):
    """Reliability diagrams (equal-width bins) before and after temperature scaling."""
    double = width_key == "double_column"
    n = len(_F5_MODELS)
    fig, axes = new_figure(width_key, 84 if double else min(CFG.FIG_MAX_HEIGHT_MM, 72 * n + 12), 1 if double else n, n if double else 1)
    for ax, m, letter in zip(np.atleast_1d(axes), _F5_MODELS, "abc"):
        for tag, col, mk, ls in (("raw", OKABE_ITO["black"], "o", "-"), ("ts", OKABE_ITO["vermillion"], "s", "--")):
            rb = RELIABILITY[(RELIABILITY["model"] == m) & (RELIABILITY["calibration"] == tag) & (RELIABILITY["count"] > 0)]
            ax.plot(rb["mean_confidence"], rb["accuracy"], color=col, marker=mk, linestyle=ls, markerfacecolor=col)
        ax.plot([0, 1], [0, 1], color="#000000", linestyle=":", linewidth=1.0)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal")
        ax.set_xlabel("Mean confidence in bin")
        ax.set_ylabel("Accuracy in bin")
        panel_label(ax, letter, MODEL_LABEL[m])
    fig.legend(handles=[Line2D([], [], color=OKABE_ITO["black"], marker="o", linestyle="-", label="Before temperature scaling"),
                        Line2D([], [], color=OKABE_ITO["vermillion"], marker="s", linestyle="--", label="After temperature scaling"),
                        Line2D([], [], color="#000000", linestyle=":", linewidth=1.0, label="Perfect calibration")],
               loc="outside lower center", ncol=3 if double else 1)
    return fig


render_both_widths("F5", build_F5, (
    f"Figure F5. Reliability diagrams on {OOF_TEXT}, using {CFG.ECE_BINS} equal-width bins of the top-label confidence; only non-empty bins are drawn. "
    + " ".join(f"({l}) {MODEL_LABEL[m]}." for l, m in zip("abc", _F5_MODELS))
    + " Solid line with circles: before temperature scaling. Dashed line with squares: after temperature scaling, with one temperature per fold fitted"
      " on that fold's validation split. Dotted diagonal: perfect calibration."))


def build_F6(width_key: str):
    """Risk-coverage curves of all models (confidence = maximum probability, before temperature scaling)."""
    double = width_key == "double_column"
    fig, ax = new_figure(width_key, 100 if double else 150)
    for i, m in enumerate(MODELS):
        cov, risk = RISK_COVERAGE[(m, "raw")]
        keep = cov >= 0.1
        sel = np.unique(np.linspace(0, keep.sum() - 1, 300).astype(int))
        s = MODEL_STYLE[m]
        ax.plot(cov[keep][sel], risk[keep][sel], color=s["color"], linestyle=s["linestyle"], marker=s["marker"],
                markerfacecolor=s["color"], markevery=(int(5 + 2 * i), 30))
    ax.set_xlim(0.1, 1.0)
    ax.set_ylim(bottom=0.0)
    ax.set_xlabel("Coverage (fraction of images retained)")
    ax.set_ylabel("Selective risk (error rate among retained images)")
    fig.legend(handles=[model_handle(m) for m in MODELS], loc="outside right center" if double else "outside lower center",
               ncol=1 if double else 2, handlelength=3.0)
    return fig


render_both_widths("F6", build_F6, (
    f"Figure F6. Risk–coverage curves on {OOF_TEXT}. Images are retained in order of decreasing confidence (maximum predicted probability, before"
    f" temperature scaling); selective risk is the error rate among retained images. Coverage from 0.1 to 1.0 is shown. {MD_NOTE}."))

_F7_FAMILIES = [f for f in ("early_concat", "late_fusion", "cross_attn", "metablock") if f in MODELS]
_F7_REFS = [m for m in ("image_only", "meta_gbm") if m in MODELS]


def build_F7(width_key: str):
    """Balanced accuracy against the rate of randomly missing metadata, one panel per fusion family."""
    double = width_key == "double_column"
    n = len(_F7_FAMILIES)
    ncols = 2 if (double and n > 1) else 1
    nrows = math.ceil(n / ncols)
    fig, axes = new_figure(width_key, min(CFG.FIG_MAX_HEIGHT_MM, (70 if double else 50) * nrows + 22), nrows, ncols, sharex=True, sharey=True, squeeze=False)
    curves = ROBUST_CURVES[ROBUST_CURVES["kind"] == "mcar"]
    shown = sorted({m for f in _F7_FAMILIES for m in (f, f + "_md") if m in MODELS} | set(_F7_REFS), key=MODELS.index)
    lo = np.nanmin(curves[curves["model"].isin(shown)]["ci_low"].to_numpy())
    hi = np.nanmax(curves[curves["model"].isin(shown)]["ci_high"].to_numpy())
    pad = 0.04 * (hi - lo) if hi > lo else 0.02
    flat = axes.ravel()
    for ax, fam, letter in zip(flat, _F7_FAMILIES, "abcd"):
        members = _F7_REFS + [m for m in (fam, fam + "_md") if m in MODELS]
        for j, m in enumerate(members):
            c = curves[curves["model"] == m].sort_values("rate")
            s = MODEL_STYLE[m]
            x = 100 * c["rate"].to_numpy() + (j - (len(members) - 1) / 2) * 1.2        # small offset so error bars do not overlap
            ax.errorbar(x, c["balanced_accuracy"], yerr=[c["balanced_accuracy"] - c["ci_low"], c["ci_high"] - c["balanced_accuracy"]],
                        color=s["color"], linestyle=s["linestyle"], marker=s["marker"], markerfacecolor=s["color"], **ERR_KW)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_xticks([0, 20, 40, 60, 80, 100])
        panel_label(ax, letter, _FAMILY_STYLE[fam][3])
    for ax in flat[n:]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("Balanced accuracy")
    for ax in flat[:n][-ncols:]:
        ax.set_xlabel("Metadata fields set to missing at test time (%)")
    for ax in flat[:n]:
        ax.tick_params(labelbottom=True)
    fig.legend(handles=[model_handle(m) for m in shown], loc="outside lower center", ncol=3 if double else 2, handlelength=3.0)
    return fig


if _F7_FAMILIES:
    render_both_widths("F7", build_F7, (
        f"Figure F7. Balanced accuracy on {OOF_TEXT} when each metadata field of each test image is independently set to missing with the probability"
        f" shown on the x-axis, in addition to the missingness present in the data. "
        + " ".join(f"({l}) {_FAMILY_STYLE[f][3]}." for l, f in zip("abcd", _F7_FAMILIES))
        + f" Each panel shows the standard (solid line) and modality-dropout (dashed line) variant of one fusion strategy, with the image-only and"
          f" metadata-only models as references. Points are means over {CFG.N_MASK_SEEDS} random masks (one mask at 0% and 100%); error bars show the"
          f" {CI_TEXT} of that mean. Points are offset horizontally by up to 2 percentage points for legibility. All panels share the y-axis range. {MD_NOTE}."))
else:
    skip_figure("F7", "no fusion model is in MODELS_TO_RUN")

_F8_MODELS = [SELECTED[k] for k in ("image_only", "best_fusion") if k in SELECTED]


def build_F8(width_key: str):
    """Forest plot of balanced accuracy by Fitzpatrick skin type with sample sizes."""
    double = width_key == "double_column"
    sub = FAIR[(FAIR["attribute"] == "Fitzpatrick type") & FAIR["model"].isin(_F8_MODELS)]
    levels = [l for l in EDA["fitz_levels"] if l in set(sub["subgroup"])]
    fig, ax = new_figure(width_key, 26 + (9.0 if double else 11.0) * len(levels) + (0 if double else 14))
    ypos = np.arange(len(levels))[::-1].astype(float)
    labels = []
    for y, lv in zip(ypos, levels):
        first = sub[sub["subgroup"] == lv].iloc[0]
        labels.append(f"{'Type ' if lv != 'Missing' else ''}{lv}{' †' if first['flag_small'] else ''} (n = {int(first['n_images'])})")
        for j, m in enumerate(_F8_MODELS):
            r = sub[(sub["subgroup"] == lv) & (sub["model"] == m)].iloc[0]
            s = MODEL_STYLE[m]
            yy = y + (0.5 * (len(_F8_MODELS) - 1) - j) * 0.3
            xerr = None if not np.isfinite(r["balanced_accuracy_ci_low"]) else [[r["balanced_accuracy"] - r["balanced_accuracy_ci_low"]],
                                                                               [r["balanced_accuracy_ci_high"] - r["balanced_accuracy"]]]
            ax.errorbar(r["balanced_accuracy"], yy, xerr=xerr, fmt=s["marker"], color=s["color"],
                        markerfacecolor="#FFFFFF" if r["flag_small"] else s["color"], **ERR_KW)
    ax.set_yticks(ypos, labels)
    ax.set_ylim(-0.6, len(levels) - 0.4)
    ax.set_xlabel("Balanced accuracy over the classes present in the subgroup")
    ax.set_ylabel("Fitzpatrick skin type")
    fig.legend(handles=[Line2D([], [], color=MODEL_STYLE[m]["color"], marker=MODEL_STYLE[m]["marker"], linestyle="none", label=MODEL_LABEL[m]) for m in _F8_MODELS],
               loc="outside lower center", ncol=len(_F8_MODELS) if double else 1)
    return fig


if "Fitzpatrick type" in SUBGROUPS:
    _f8 = FAIR[(FAIR["attribute"] == "Fitzpatrick type") & (FAIR["model"] == _F8_MODELS[0])]
    _small = _f8[_f8["flag_small"]]["subgroup"].tolist()
    render_both_widths("F8", build_F8, (
        f"Figure F8. Balanced accuracy by Fitzpatrick skin type on {OOF_TEXT} for " + " and ".join(MODEL_LABEL[m] for m in _F8_MODELS)
        + f". Balanced accuracy is computed over the diagnostic classes present in each subgroup. The number of images (n) is given for each subgroup;"
          f" 'Missing' denotes images without a recorded skin type. Error bars show the {CI_TEXT}. "
        + (f"Open markers and a dagger (†) mark subgroups with fewer than {CFG.MIN_SUBGROUP_N} images ({', '.join(_small)}), which are shown for completeness only."
           if _small else f"No subgroup has fewer than {CFG.MIN_SUBGROUP_N} images.")))
else:
    skip_figure("F8", "no Fitzpatrick column was found in the metadata")


def build_F9(width_key: str):
    """Balanced accuracy under patient-grouped and image-level splitting for the same models."""
    models = list(LEAKAGE["model"].unique())
    fig, ax = new_figure(width_key, 82)
    sub = LEAKAGE[LEAKAGE["metric"] == "balanced_accuracy"].set_index("model")
    styles = {"grouped": (OKABE_ITO["black"], "o"), "image_level": (OKABE_ITO["vermillion"], "s")}
    for i, m in enumerate(models):
        for j, sc in enumerate(("grouped", "image_level")):
            r = sub.loc[m]
            col, mk = styles[sc]
            yerr = None if not np.isfinite(r[f"{sc}_ci_low"]) else [[r[f"{sc}_pooled"] - r[f"{sc}_ci_low"]], [r[f"{sc}_ci_high"] - r[f"{sc}_pooled"]]]
            ax.errorbar(i + (j - 0.5) * 0.3, r[f"{sc}_pooled"], yerr=yerr, fmt=mk, color=col, markerfacecolor=col, **ERR_KW)
    ax.set_xticks(range(len(models)), [MODEL_LABEL[m] for m in models])
    ax.set_xlim(-0.6, len(models) - 0.4)
    ax.set_ylabel("Balanced accuracy")
    fig.legend(handles=[Line2D([], [], color=styles["grouped"][0], marker="o", linestyle="none", label="Patient-grouped split"),
                        Line2D([], [], color=styles["image_level"][0], marker="s", linestyle="none", label="Image-level split")],
               loc="outside lower center", ncol=2 if width_key == "double_column" else 1)
    return fig


if CFG.RUN_LEAKAGE and len(LEAKAGE):
    render_both_widths("F9", build_F9, (
        f"Figure F9. Balanced accuracy of the same models under {CFG.N_FOLDS}-fold patient-grouped cross-validation (circles) and {CFG.N_FOLDS}-fold"
        f" image-level stratified cross-validation that ignores patient identity (squares); pooled out-of-fold predictions, n = {N} images."
        f" Error bars show the {CI_TEXT}. Models: " + ", ".join(MODEL_LABEL[m] for m in LEAKAGE["model"].unique()) + "."))
else:
    skip_figure("F9", "leakage analysis disabled (CFG.RUN_LEAKAGE = False)")

**Purpose:** Figures F10 (Grad-CAM panel) and F11 (SHAP importance of metadata fields).

**Why:** Supporting, qualitative evidence for the explainability section. These figures do not establish clinical validity (see Part 14).

**Method choice:** The Grad-CAM overlay is blended numerically into an opaque RGB image (half photograph, half colour-mapped heat-map), so the saved figure contains no transparent artist. The viridis colour map is used because it is perceptually uniform and readable with colour-vision deficiency. The examples are those chosen by the fixed median-confidence rule in Part 14. F11 shows the mean absolute SHAP contribution per field with the standard deviation across folds.

**Output:** `results/figures/.../F10_*`, `F11_*` and captions, or a recorded skip with its reason.

In [ ]:
def _overlay(image_uint8: np.ndarray, cam: np.ndarray) -> np.ndarray:
    """Opaque blend of a photograph and its Grad-CAM heat-map (no alpha channel)."""
    heat = matplotlib.colormaps["viridis"](cam)[..., :3]
    return np.clip(0.5 * image_uint8.astype(float) / 255.0 + 0.5 * heat, 0.0, 1.0)


def _show_image(ax, arr: Optional[np.ndarray], n_candidates: int) -> None:
    """Draw an image, or the text 'n/a (n = k)' when the selection rule found no example."""
    ax.set_xticks([])
    ax.set_yticks([])
    if arr is None:
        ax.set_box_aspect(1.0)
        ax.text(0.5, 0.5, f"n/a (n = {n_candidates})", ha="center", va="center", fontsize=CFG.FONT_TICK_PT, color="#000000", transform=ax.transAxes)
    else:
        ax.imshow(arr, interpolation="nearest")


def build_F10(width_key: str):
    """Grad-CAM panel: for every true class a correct and an incorrect test example with its heat-map overlay."""
    double = width_key == "double_column"
    picks = {(p["true_class"], p["outcome"]): p for p in GRADCAM_PANEL["picks"]}
    kinds = [("correct", "photo"), ("correct", "cam"), ("incorrect", "photo"), ("incorrect", "cam")]
    names = ["Correct", "Correct: Grad-CAM", "Incorrect", "Incorrect: Grad-CAM"]
    if double:
        cell = 190.0 / K
        fig, axes = new_figure(width_key, min(CFG.FIG_MAX_HEIGHT_MM, 4 * (cell + 5.5) + 8), 4, K, squeeze=False)
    else:
        fig, axes = new_figure(width_key, min(CFG.FIG_MAX_HEIGHT_MM, K * (90.0 / 4 + 5.5) + 10), K, 4, squeeze=False)
    for ci, c in enumerate(CLASS_NAMES):
        for ki, (outcome, what) in enumerate(kinds):
            ax = axes[ki, ci] if double else axes[ci, ki]
            p = picks[(c, outcome)]
            has = p["row"] >= 0
            arr = None if not has else (p["image"] if what == "photo" else _overlay(p["image"], p["cam"]))
            _show_image(ax, arr, p["n_candidates"])
            if what == "cam" and has:
                ax.set_xlabel(f"{p['pred_class']}, {p['confidence']:.2f}", fontsize=CFG.FONT_TICK_PT)
            if double:
                if ki == 0:
                    ax.set_title(c, loc="center", fontsize=CFG.FONT_AXIS_PT, fontweight="bold")
                if ci == 0:
                    ax.set_ylabel(names[ki], fontsize=CFG.FONT_TICK_PT)
            else:
                if ci == 0:
                    ax.set_title(["Correct", "Grad-CAM", "Incorrect", "Grad-CAM"][ki], loc="center", fontsize=CFG.FONT_TICK_PT, fontweight="bold")
                if ki == 0:
                    ax.set_ylabel(c, fontsize=CFG.FONT_AXIS_PT)
    return fig


if GRADCAM_PANEL is not None:
    _n_missing = sum(p["row"] < 0 for p in GRADCAM_PANEL["picks"])
    render_both_widths("F10", build_F10, (
        f"Figure F10. Grad-CAM maps of the {MODEL_LABEL[GRADCAM_PANEL['model']]} model for test images of each true class ({', '.join(CLASS_NAMES)});"
        f" each image is explained by the fold model for which it was a test image. For each class, one correctly and one incorrectly classified image"
        f" is shown with its Grad-CAM overlay for the predicted class (yellow = highest activation); the text under each overlay gives the predicted"
        f" class and its probability. Examples were selected by a fixed rule: the image with the median confidence among the correctly (respectively"
        f" incorrectly) classified test images of the class, ties broken by image identifier."
        + (f" For {_n_missing} class/outcome combinations no image met the rule; these cells are marked n/a with the number of candidates." if _n_missing else "")
        + " Photographs are shown at the network input resolution. The maps are a qualitative plausibility check and do not establish clinical validity."))
else:
    skip_figure("F10", GRADCAM_SKIP_REASON)


def build_F11(width_key: str):
    """Mean absolute SHAP contribution of each metadata field (metadata-only model)."""
    top = SHAP_IMPORTANCE.head(20).iloc[::-1]
    fig, ax = new_figure(width_key, min(CFG.FIG_MAX_HEIGHT_MM, 22 + 5.5 * len(top)))
    ax.barh(top["field"], top["mean_abs_shap"], xerr=top["sd_across_folds"], color=OKABE_ITO["blue"], edgecolor="#000000", linewidth=0.8,
            error_kw=dict(elinewidth=1.0, capsize=2.5, capthick=1.0, ecolor="#000000"))
    ax.set_xlim(left=0.0)
    ax.set_xlabel("Mean absolute SHAP value (log-odds)")
    ax.set_ylabel("Metadata field")
    return fig


if SHAP_IMPORTANCE is not None:
    _n_show = min(20, len(SHAP_IMPORTANCE))
    render_both_widths("F11", build_F11, (
        f"Figure F11. Importance of metadata fields in the metadata-only gradient-boosting model: mean absolute SHAP value (TreeSHAP, log-odds units),"
        f" averaged over test images and the {K} classes, with the contributions of the encoded columns of a field summed."
        f" Bars show the mean over {CFG.N_FOLDS} test folds and error bars the standard deviation across folds."
        f" The {_n_show} fields with the largest values are shown" + (f" (of {len(SHAP_IMPORTANCE)})" if _n_show < len(SHAP_IMPORTANCE) else "")
        + ". SHAP values describe the model's use of a field and do not establish causal or clinical relevance."))
else:
    skip_figure("F11", SHAP_SKIP_REASON)
save_json(dict(figures=FIG_REGISTRY, skipped=FIG_SKIPPED), DIRS["qc"] / "figure_registry.json")

**Purpose:** Table export utilities: number formatting, automatic bolding of the best value, and writers for CSV, Word (`python-docx`) and LaTeX (`booktabs`).

**Why:** Tables must be consistent, reproducible and ready to paste, and no value may be formatted or highlighted by hand.

**Method choice:** Each table is held twice: a *raw* table with full-precision numeric values (exported as `<id>_raw.csv`) and a *display* table of formatted strings (exported as `<id>_formatted.csv`, `.docx`, `.tex`). Word tables use the three-line layout (top rule, rule under the header, bottom rule, no vertical lines), set through the cell-border XML because `python-docx` has no high-level border API. Numeric columns are centred so that the fixed-decimal entries align. Bold marks the best value in a column and is computed from the raw numbers. A quantity that cannot be computed is written as `n/a (n = k)`, never left blank or guessed.

**Output:** `export_table()`, `TABLE_REGISTRY`.

In [ ]:
TABLE_REGISTRY: Dict[str, Dict[str, Any]] = {}
TABLE_SKIPPED: Dict[str, str] = {}
DEC = CFG.DECIMALS


def na(n: Any) -> str:
    """Standard text for a value that cannot be computed, with the sample size that explains why."""
    return f"n/a (n = {n})"


def fnum(x: Any, n_for_na: Any = 0, d: int = DEC) -> str:
    """Fixed-decimal number, or 'n/a (n = k)' if not finite."""
    return f"{x:.{d}f}" if (x is not None and np.isfinite(x)) else na(n_for_na)


def fci(lo: Any, hi: Any, n_for_na: Any = 0, d: int = DEC) -> str:
    """Confidence interval '[lo–hi]' or 'n/a (n = k)'."""
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return na(n_for_na)
    sep = " to " if (lo < 0 or hi < 0) else "–"          # an en dash next to a minus sign is unreadable
    return f"[{lo:.{d}f}{sep}{hi:.{d}f}]"


def fest(x: Any, lo: Any, hi: Any, n_for_na: Any = 0, d: int = DEC) -> str:
    """Estimate with interval: '0.812 [0.790–0.830]'."""
    if not np.isfinite(x):
        return na(n_for_na)
    return f"{x:.{d}f} {fci(lo, hi, n_for_na, d)}" if (np.isfinite(lo) and np.isfinite(hi)) else f"{x:.{d}f} [CI {na(n_for_na)}]"


def fpm(pooled: Any, lo: Any, hi: Any, mean: Any, sd: Any, n_for_na: Any = 0, d: int = DEC) -> str:
    """Pooled estimate with bootstrap interval, then fold mean ± SD: '0.812 [0.790–0.830]; 0.810 ± 0.021'."""
    if not np.isfinite(pooled):
        return na(n_for_na)
    folds = f"{mean:.{d}f} ± {sd:.{d}f}" if (np.isfinite(mean) and np.isfinite(sd)) else f"folds {na(n_for_na)}"
    return f"{fest(pooled, lo, hi, n_for_na, d)}; {folds}"


def fp(p: Any, n_for_na: Any = 0) -> str:
    """p-value with three decimals, '<0.001' below that."""
    if p is None or not np.isfinite(p):
        return na(n_for_na)
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def best_mask(values: Sequence[float], mode: str) -> np.ndarray:
    """Boolean mask of the best finite value(s) in a column ('max' or 'min'); ties are all marked."""
    v = np.asarray(values, dtype=float)
    if not np.isfinite(v).any():
        return np.zeros(len(v), dtype=bool)
    target = np.nanmax(v) if mode == "max" else np.nanmin(v)
    return np.isfinite(v) & np.isclose(v, target, rtol=0, atol=0.5 * 10 ** (-DEC))


_TEX_MAP = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
            "~": r"\textasciitilde{}", "^": r"\textasciicircum{}", "±": r"$\pm$", "≥": r"$\geq$", "≤": r"$\leq$", "–": "--", "−": "$-$",
            "Δ": r"$\Delta$", "†": r"$\dagger$", "×": r"$\times$", "σ": r"$\sigma$", "<": r"$<$", ">": r"$>$", "|": r"$|$", "→": r"$\rightarrow$"}


def tex_escape(text: str) -> str:
    """Escape LaTeX special characters and map the Unicode symbols used in the tables."""
    return "".join(_TEX_MAP.get(ch, ch) for ch in str(text))


def _set_cell_border(cell, **edges: Dict[str, Any]) -> None:
    """Set individual borders of a Word table cell, e.g. top=dict(sz=12) (size in eighths of a point)."""
    tc_pr = cell._tc.get_or_add_tcPr()
    borders = tc_pr.find(qn("w:tcBorders"))
    if borders is None:
        borders = OxmlElement("w:tcBorders")
        tc_pr.append(borders)
    for edge, spec in edges.items():
        el = borders.find(qn(f"w:{edge}"))
        if el is None:
            el = OxmlElement(f"w:{edge}")
            borders.append(el)
        el.set(qn("w:val"), "single")
        el.set(qn("w:sz"), str(spec.get("sz", 8)))
        el.set(qn("w:space"), "0")
        el.set(qn("w:color"), "000000")


def _docx_run(paragraph, text: str, bold: bool = False, size: Optional[float] = None, italic: bool = False) -> None:
    """Add a run in the table font (also for East-Asian text, so Word does not substitute another font)."""
    run = paragraph.add_run(text)
    run.bold, run.italic = bold, italic
    run.font.name = CFG.TABLE_FONT
    run.font.size = Pt(size or CFG.TABLE_FONT_PT)
    run.font.color.rgb = RGBColor(0, 0, 0)
    r_pr = run._element.get_or_add_rPr()
    r_fonts = r_pr.find(qn("w:rFonts"))
    if r_fonts is None:
        r_fonts = OxmlElement("w:rFonts")
        r_pr.append(r_fonts)
    for attr in ("w:ascii", "w:hAnsi", "w:eastAsia", "w:cs"):
        r_fonts.set(qn(attr), CFG.TABLE_FONT)


def export_table(table_id: str, title: str, display_df: pd.DataFrame, raw_df: pd.DataFrame, notes: Sequence[str],
                 bold: Optional[pd.DataFrame] = None, landscape: bool = False) -> None:
    """Write one table as raw CSV, formatted CSV, Word (.docx) and LaTeX (.tex) and register it for QC."""
    assert len(display_df) > 0, f"{table_id}: the table has no rows."
    display_df = display_df.reset_index(drop=True).astype(str)
    if bold is None:
        bold = pd.DataFrame(False, index=display_df.index, columns=display_df.columns)
    bold = bold.reset_index(drop=True).reindex(columns=display_df.columns, fill_value=False)
    base = DIRS["tables"] / table_id
    raw_df.to_csv(str(base) + "_raw.csv", index=False)
    display_df.to_csv(str(base) + "_formatted.csv", index=False)
    caption = f"Table {table_id[1:]}. {title}" if table_id[1:].isdigit() else f"Table {table_id}. {title}"

    # ---------------- Word
    document = docx.Document()
    section = document.sections[0]
    if landscape:
        section.orientation = WD_ORIENT.LANDSCAPE
        section.page_width, section.page_height = section.page_height, section.page_width
    for side in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
        setattr(section, side, Mm(20))
    cap = document.add_paragraph()
    _docx_run(cap, caption.split(". ", 1)[0] + ". ", bold=True)
    _docx_run(cap, caption.split(". ", 1)[1])
    n_rows, n_cols = display_df.shape
    table = document.add_table(rows=n_rows + 1, cols=n_cols)
    table.autofit = True
    for j, col in enumerate(display_df.columns):
        cell = table.cell(0, j)
        par = cell.paragraphs[0]
        par.alignment = WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.CENTER
        _docx_run(par, str(col), bold=True)
        _set_cell_border(cell, top=dict(sz=12), bottom=dict(sz=6))
    for i in range(n_rows):
        for j in range(n_cols):
            cell = table.cell(i + 1, j)
            par = cell.paragraphs[0]
            par.alignment = WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.CENTER
            _docx_run(par, display_df.iat[i, j], bold=bool(bold.iat[i, j]))
            if i == n_rows - 1:
                _set_cell_border(cell, bottom=dict(sz=12))
    for note in notes:
        _docx_run(document.add_paragraph(), note, size=CFG.TABLE_FONT_PT - 1)
    document.save(str(base) + ".docx")

    # ---------------- LaTeX (booktabs)
    lines = [r"% requires \usepackage{booktabs}", r"\begin{table}[htbp]", r"\centering", r"\small",
             rf"\caption{{{tex_escape(title)}}}", rf"\label{{tab:{table_id.lower()}}}",
             r"\begin{tabular}{l" + "c" * (n_cols - 1) + "}", r"\toprule",
             " & ".join(rf"\textbf{{{tex_escape(c)}}}" for c in display_df.columns) + r" \\", r"\midrule"]
    for i in range(n_rows):
        cells = [rf"\textbf{{{tex_escape(display_df.iat[i, j])}}}" if bold.iat[i, j] else tex_escape(display_df.iat[i, j]) for j in range(n_cols)]
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    if notes:
        lines += [r"\par\smallskip", r"\begin{minipage}{0.95\linewidth}\footnotesize " + " ".join(tex_escape(n) for n in notes) + r"\end{minipage}"]
    lines.append(r"\end{table}")
    Path(str(base) + ".tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
    TABLE_REGISTRY[table_id] = dict(title=title, n_rows=n_rows, n_cols=n_cols, notes=list(notes),
                                    files={k: str(base) + s for k, s in (("raw", "_raw.csv"), ("formatted", "_formatted.csv"), ("docx", ".docx"), ("tex", ".tex"))})
    print(f"[saved] {table_id}: {n_rows} rows x {n_cols} columns -> csv, docx, tex")


def skip_table(table_id: str, reason: str) -> None:
    """Record that a table was not produced and why."""
    TABLE_SKIPPED[table_id] = reason
    print(f"[skipped] {table_id}: {reason}")

**Purpose:** Build and export Tables T1–T12 (T1 dataset, T2 main results, T3 statistical comparisons, T4 calibration, T5 robustness, T6 fairness, T7 settings, T8 literature, T9 software and hardware, T10 leakage, T11 and T12 metadata-missingness audit, T13 restricted-metadata sensitivity analysis) and the supplementary per-class tables TS1 and TS2.

**Why:** These are the tables of the manuscript; each is generated from the analysis objects of Parts 3–14.

**Method choice:**
- **T2** cells read "pooled estimate [95% CI]; fold mean ± SD". **Deviation from the requested cell format (`0.812 ± 0.021 [0.79–0.83]`), stated deliberately:** the bootstrap interval belongs to the pooled out-of-fold estimate, not to the fold mean, and the two can differ noticeably (for AUROC the fold mean can even lie outside the pooled interval, because pooling ranks scores across five different fold models). Printing a fold mean next to an interval of a different quantity would misreport the uncertainty, so each interval is printed next to the estimate it belongs to. The best pooled estimate per column is bold. Bold marks the numerically best value only; whether a difference is supported is answered by T3.
- **T8 (literature)** is built **only** from `literature_baselines.csv` in the project folder. If the file does not exist, an empty template with the required columns is created and T8 is skipped until you fill it from the original papers. No published number is typed into this notebook. Every row whose `split_protocol` text differs from this study's protocol label is marked "not directly comparable"; because published splits are never identical to ours, a matching label still does not make numbers strictly comparable, and the table note says so.
- **T6** lists the image-only model and the validation-selected fusion model; all models are in `logs/rq4_subgroups.csv`.

**Output:** `results/tables/T*.{docx,tex}`, `T*_raw.csv`, `T*_formatted.csv`; `literature_baselines.csv` template in the project folder if absent.

In [ ]:
# ------------------------------------------------------------------ T1 dataset characteristics
_t1 = [("Images", str(N), N), ("Patients", str(OBSERVED["n_patients"]), OBSERVED["n_patients"]),
       ("Lesions (distinct patient–lesion identifier pairs)", str(N_LESION_PAIRS), N_LESION_PAIRS),
       ("Distinct lesion-identifier values", str(N_LESION_IDS), N_LESION_IDS),
       ("Lesion-identifier values occurring under more than one patient", str(N_SHARED_LESION_IDS), N_SHARED_LESION_IDS),
       ("Diagnostic classes", str(K), K),
       ("Exact duplicate images: images (groups)", f"{DUPLICATES['n_images']} ({DUPLICATES['n_groups']})", DUPLICATES["n_images"]),
       ("Duplicate groups spanning more than one patient", str(DUPLICATES["n_groups_cross_patient"]), DUPLICATES["n_groups_cross_patient"])]
for _, _r in EDA["class_counts"].iterrows():
    _t1.append((f"Class {_r['class']}: images, n (%)", f"{int(_r['n_images'])} ({_r['pct_images']:.1f})", _r["n_images"]))
for _name, _lab in (("images_per_patient", "Images per patient"), ("images_per_lesion", "Images per lesion"), ("lesions_per_patient", "Lesions per patient")):
    _a = EDA[_name]
    _t1.append((f"{_lab}: median (min–max)", f"{np.median(_a):g} ({_a.min()}–{_a.max()})", float(np.median(_a))))
    _t1.append((f"{_lab}: more than one, n (%)", f"{int((_a > 1).sum())} ({100.0 * (_a > 1).mean():.1f})", int((_a > 1).sum())))
_t1.append(("Metadata fields used as model inputs (numeric / categorical)", f"{N_FIELDS} ({len(NUM_COLS)} / {len(CAT_COLS)})", N_FIELDS))
_t1.append(("Metadata fields with at least one missing value", str(int((EDA["missing_overall"] > 0).sum())), int((EDA["missing_overall"] > 0).sum())))
_t1.append(("Missing values over all input fields, %", f"{100.0 * MISSING_MATRIX.mean():.1f}", 100.0 * MISSING_MATRIX.mean()))
_mm = MISSING_MATRIX.mean(axis=1) >= CFG.AUDIT_ROW_MISSING_SHARE
_t1.append((f"Images with at least {100 * CFG.AUDIT_ROW_MISSING_SHARE:.0f}% of input fields missing, n (%)", f"{int(_mm.sum())} ({100.0 * _mm.mean():.1f})", int(_mm.sum())))
for _lab, _sel in (("cancer", Y_CANCER == 1), ("non-cancer", Y_CANCER == 0)):
    _t1.append((f"  of which {_lab} images, n (% of {_lab} images)", f"{int((_mm & _sel).sum())} ({100.0 * (_mm & _sel).sum() / max(1, int(_sel.sum())):.1f})", int((_mm & _sel).sum())))
if EDA["fitz_counts"] is not None:
    for _, _r in EDA["fitz_counts"].iterrows():
        _lab = "Fitzpatrick type missing" if _r["fitzpatrick"] == "Missing" else f"Fitzpatrick type {_r['fitzpatrick']}"
        _t1.append((f"{_lab}: images, n (%)", f"{int(_r['n_images'])} ({100.0 * _r['n_images'] / N:.1f})", _r["n_images"]))
if EDA["image_size"] is not None:
    _sz = EDA["image_size"]
    _t1.append(("Original image width, pixels: median (min–max)", f"{_sz.loc['50%', 'orig_width']:.0f} ({_sz.loc['min', 'orig_width']:.0f}–{_sz.loc['max', 'orig_width']:.0f})", _sz.loc["50%", "orig_width"]))
    _t1.append(("Original image height, pixels: median (min–max)", f"{_sz.loc['50%', 'orig_height']:.0f} ({_sz.loc['min', 'orig_height']:.0f}–{_sz.loc['max', 'orig_height']:.0f})", _sz.loc["50%", "orig_height"]))
export_table("T1", "Characteristics of the dataset as loaded and verified by the pipeline.",
             pd.DataFrame({"Characteristic": [r[0] for r in _t1], "Value": [r[1] for r in _t1]}),
             pd.DataFrame({"characteristic": [r[0] for r in _t1], "value_text": [r[1] for r in _t1], "value_numeric": [r[2] for r in _t1]}),
             ["All values are computed from the metadata file and image files; percentages refer to all images unless stated."])

# ------------------------------------------------------------------ T2 main results (RQ1) and TS1 per-class results
def _rq1_table(metrics: List[str], labels: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Display table, raw table and bold mask for a list of metrics (grouped scheme)."""
    sub = RQ1[(RQ1["scheme"] == "grouped") & RQ1["model"].isin(MODELS)]
    disp, bold = {"Model": [MODEL_LABEL[m] for m in MODELS]}, {"Model": [False] * len(MODELS)}
    raw = [sub[sub["metric"].isin(metrics)].assign(model_label=lambda d: d["model"].map(MODEL_LABEL))]
    for met, lab in zip(metrics, labels):
        rows = sub[sub["metric"] == met].set_index("model").loc[MODELS]
        disp[lab] = [fpm(r.pooled, r.ci_low, r.ci_high, r.fold_mean, r.fold_sd, int(r.n_boot_valid)) for r in rows.itertuples()]
        bold[lab] = best_mask(rows["pooled"].to_numpy(), "max")
    return pd.DataFrame(disp), pd.concat(raw), pd.DataFrame(bold)


_fold_note = (f"Each cell gives the pooled out-of-fold estimate with its {CI_PCT}% confidence interval from a patient-level bootstrap ({CFG.N_BOOT} resamples),"
              f" followed after the semicolon by the mean ± SD over {CFG.N_REPEATS * CFG.N_FOLDS} test folds ({CFG.N_REPEATS} × {CFG.N_FOLDS}-fold patient-grouped"
              f" cross-validation). The two can differ: pooled AUROC ranks scores across the fold models, whereas the fold mean averages within-fold values.")
_d, _raw, _b = _rq1_table(PRIMARY_METRICS, ["Balanced accuracy", "Macro-F1", "Macro AUROC", "Cancer sensitivity", "Cancer specificity"])
export_table("T2", "Discrimination of all models under patient-grouped cross-validation (RQ1).", _d, _raw, [
    _fold_note, "Bold: highest pooled estimate in the column (see Table 3 for statistical comparisons).",
    f"Macro AUROC: macro-averaged one-vs-rest area under the ROC curve. Cancer sensitivity and specificity: {', '.join(CFG.CANCER_CLASSES)} versus "
    f"{', '.join(c for c in CLASS_NAMES if c not in CFG.CANCER_CLASSES)}, from the arg-max class. GBM, gradient boosting. {MD_NOTE}."], bold=_b, landscape=True)
for _kind, _tid, _title in (("recall", "TS1", "Per-class recall of all models under patient-grouped cross-validation."),
                            ("auroc", "TS2", "Per-class one-vs-rest AUROC of all models under patient-grouped cross-validation.")):
    _d, _raw, _b = _rq1_table([f"{_kind}_{c}" for c in CLASS_NAMES], list(CLASS_NAMES))
    export_table(_tid, _title, _d, _raw, [_fold_note, "Bold: highest pooled estimate in the column.",
                 "Images per class: " + ", ".join(f"{c} {int(n)}" for c, n in zip(CLASS_NAMES, EDA['class_counts']['n_images'])) + f". {MD_NOTE}."], bold=_b, landscape=True)

# ------------------------------------------------------------------ T3 pairwise statistical comparisons
_d = pd.DataFrame({
    "Comparison (A vs B)": [f"{MODEL_LABEL[r.model_a]} vs {MODEL_LABEL[r.model_b]}" for r in STATS.itertuples()],
    "Δ balanced accuracy [CI]": [fest(r.balanced_accuracy_diff, r.balanced_accuracy_ci_low, r.balanced_accuracy_ci_high, CFG.N_BOOT) for r in STATS.itertuples()],
    "p (Holm)": [f"{fp(r.balanced_accuracy_p_boot, CFG.N_BOOT)} ({fp(r.balanced_accuracy_p_boot_holm, CFG.N_BOOT)})" for r in STATS.itertuples()],
    "Δ macro AUROC [CI]": [fest(r.auroc_macro_ovr_diff, r.auroc_macro_ovr_ci_low, r.auroc_macro_ovr_ci_high, CFG.N_BOOT) for r in STATS.itertuples()],
    "p (Holm) ": [f"{fp(r.auroc_macro_ovr_p_boot, CFG.N_BOOT)} ({fp(r.auroc_macro_ovr_p_boot_holm, CFG.N_BOOT)})" for r in STATS.itertuples()],
    "Δ cancer AUROC, DeLong z": [f"{fnum(r.cancer_auroc_diff, N)}, z = {fnum(r.delong_z, N, 2)}" for r in STATS.itertuples()],
    "DeLong p (Holm)": [f"{fp(r.delong_p, N)} ({fp(r.delong_p_holm, N)})" for r in STATS.itertuples()],
    "Wilcoxon p (Holm)": [f"{fp(r.wilcoxon_p, f'{int(r.wilcoxon_n)} folds')} ({fp(r.wilcoxon_p_holm, f'{int(r.wilcoxon_n)} folds')})" if np.isfinite(r.wilcoxon_p)
                          else na(f"{int(r.wilcoxon_n)} folds") for r in STATS.itertuples()],
})
export_table("T3", "Pairwise statistical comparisons between models on pooled out-of-fold predictions.", _d, STATS, [
    f"Δ = A − B. Differences in balanced accuracy and macro AUROC: paired patient-level bootstrap ({CFG.N_BOOT} resamples) with {CI_PCT}% percentile"
    f" confidence interval and two-sided bootstrap p-value (smallest attainable value {2.0 / (CFG.N_BOOT + 1):.4f}).",
    "Cancer AUROC: DeLong test for correlated ROC curves on the summed cancer-class probability; it assumes independent observations, whereas images are"
    " clustered within patients and pooled scores come from different fold models, so it is a secondary analysis.",
    f"Holm-adjusted p-values (in parentheses) are adjusted over the {len(STATS)} comparisons in the table, separately for each test. " + WILCOXON_NOTE,
    f"{MD_NOTE}."], landscape=True)

# ------------------------------------------------------------------ T4 calibration and selective prediction (RQ3)
_c = CALIB.set_index("model").loc[MODELS]
_d = pd.DataFrame({
    "Model": [MODEL_LABEL[m] for m in MODELS],
    "ECE before": [fest(r.ece_raw, r.ece_raw_ci_low, r.ece_raw_ci_high, N) for r in _c.itertuples()],
    "ECE after TS": [fest(r.ece_ts, r.ece_ts_ci_low, r.ece_ts_ci_high, N) for r in _c.itertuples()],
    "Brier before": [fest(r.brier_raw, r.brier_raw_ci_low, r.brier_raw_ci_high, N) for r in _c.itertuples()],
    "Brier after TS": [fest(r.brier_ts, r.brier_ts_ci_low, r.brier_ts_ci_high, N) for r in _c.itertuples()],
    "AURC": [fest(r.aurc_raw, r.aurc_raw_ci_low, r.aurc_raw_ci_high, N) for r in _c.itertuples()],
    "Error at full coverage": [fnum(r.error_full_coverage, N) for r in _c.itertuples()],
    "Selective accuracy (coverage)": [f"{fnum(r.selective_accuracy, int(r.selective_n))} ({fnum(r.selective_coverage, N)})" for r in _c.itertuples()],
})
_b = pd.DataFrame({c: False for c in _d.columns}, index=_d.index)
for _col, _src in (("ECE before", "ece_raw"), ("ECE after TS", "ece_ts"), ("Brier before", "brier_raw"), ("Brier after TS", "brier_ts"), ("AURC", "aurc_raw")):
    _b[_col] = best_mask(_c[_src].to_numpy(), "min")
export_table("T4", "Calibration and selective prediction on pooled out-of-fold predictions (RQ3).", _d, CALIB, [
    f"ECE: top-label expected calibration error with {CFG.ECE_BINS} equal-width confidence bins. Brier: multi-class Brier score (range 0–2)."
    f" TS: temperature scaling with one temperature per fold fitted on that fold's validation split. AURC: area under the risk–coverage curve"
    f" (confidence = maximum probability, before TS). Lower is better for ECE, Brier, AURC and error; bold marks the lowest value in a column.",
    f"Brackets: {CI_PCT}% patient-level bootstrap confidence intervals ({CFG.N_BOOT} resamples). Selective accuracy: accuracy on the test images whose calibrated"
    f" confidence is at least the threshold that retained {100 * CFG.SELECTIVE_TARGET_COVERAGE:.0f}% of the validation images of the same fold; the realised test"
    f" coverage is in parentheses. Equal-mass ECE, negative log-likelihood and validation-thresholded cancer sensitivity/specificity are in the raw CSV. {MD_NOTE}."],
    bold=_b, landscape=True)

# ------------------------------------------------------------------ T5 robustness (RQ2)
_s = ROBUST_SUMMARY.set_index("model").loc[MODELS]
_d = pd.DataFrame({"Model": [MODEL_LABEL[m] for m in MODELS]})
_b = pd.DataFrame({"Model": [False] * len(MODELS)})
for _col, _key, _mode in (("BA, 0% missing", "ba_0", "max"), (f"BA, {100 * float(MID_RATE_LABEL):.0f}% missing", "ba_mid", "max"), ("BA, 100% missing", "ba_100", "max"),
                          ("Drop 0%→100%", "drop_0_to_100", "min"), ("Mean BA over rates", "aurc_robust", "max")):
    _d[_col] = [fest(r[_key], r[f"{_key}_ci_low"], r[f"{_key}_ci_high"], N) for _, r in _s.iterrows()]
    _b[_col] = best_mask(_s[_key].to_numpy(), _mode)
for _col, _key in (("Cancer sens., 100% missing", "cancer_sens_100"), ("Cancer spec., 100% missing", "cancer_spec_100")):
    _d[_col] = [fest(r[_key], r[f"{_key}_ci_low"], r[f"{_key}_ci_high"], N) for _, r in _s.iterrows()]
    _b[_col] = best_mask(_s[_key].to_numpy(), "max")
for _n in SCENARIO_FIELDS:
    _col = f"BA, scenario: {_n.replace('_', ' ')}"
    _d[_col] = [fest(r[f"scenario_{_n}"], r[f"scenario_{_n}_ci_low"], r[f"scenario_{_n}_ci_high"], N) for _, r in _s.iterrows()]
    _b[_col] = best_mask(_s[f"scenario_{_n}"].to_numpy(), "max")
_notes = [f"BA: balanced accuracy on pooled out-of-fold predictions when each metadata field of each test image is set to missing with the stated probability"
          f" (mean over {CFG.N_MASK_SEEDS} random masks for intermediate rates), in addition to missingness present in the data. Brackets: {CI_PCT}% patient-level"
          f" bootstrap confidence intervals. Mean BA over rates: trapezoidal mean over the rates {', '.join(f'{100 * r:.0f}%' for r in RATES)}.",
          "Bold: best value in the column (highest BA, smallest drop). The image-only model does not use metadata, so its values are constant by construction.",
          f"Cancer sens. / spec.: sensitivity and specificity for {', '.join(CFG.CANCER_CLASSES)} versus the other classes when all metadata fields are missing; the values"
          " with metadata as recorded are in Table 2. In this dataset missing metadata is itself associated with non-cancer diagnoses (Table 11), which must be"
          " considered when reading this table."]
if SCENARIO_FIELDS:
    _notes.append("Scenarios (fields set to missing): " + "; ".join(f"{n.replace('_', ' ')}: {', '.join(f)}" for n, f in SCENARIO_FIELDS.items()) + ".")
if len(ROBUST_MD_EFFECT):
    _notes.append("Modality dropout minus standard training, Δ BA [CI], Holm-adjusted p: " + "; ".join(
        f"{MODEL_LABEL[r.model_standard]} at {r.condition}: {fest(r['diff'], r.ci_low, r.ci_high, N)}, p = {fp(r.p_holm, N)}" for _, r in ROBUST_MD_EFFECT.iterrows()) + ".")
_notes.append(f"{MD_NOTE}.")
export_table("T5", "Robustness to missing metadata at test time (RQ2).", _d, ROBUST_SUMMARY, _notes, bold=_b, landscape=True)

# ------------------------------------------------------------------ T6 fairness (RQ4)
if len(FAIR):
    _fm = [SELECTED[k] for k in ("image_only", "best_fusion") if k in SELECTED]
    _f = FAIR[FAIR["model"].isin(_fm)]
    _rows = []
    for (_attr, _sg), _g in _f.groupby(["attribute", "subgroup"], sort=False):
        _first = _g.iloc[0]
        _row = {"Attribute": _attr, "Subgroup": f"{_sg}{' †' if _first['flag_small'] else ''}", "Images / patients": f"{int(_first['n_images'])} / {int(_first['n_patients'])}",
                "Classes present": str(int(_first["n_classes_present"]))}
        for _m in _fm:
            _r = _g[_g["model"] == _m].iloc[0]
            _row[f"BA, {MODEL_LABEL[_m]}"] = fest(_r["balanced_accuracy"], _r["balanced_accuracy_ci_low"], _r["balanced_accuracy_ci_high"], int(_r["n_images"]))
            _row[f"Cancer sens., {MODEL_LABEL[_m]}"] = fest(_r["cancer_sensitivity"], _r["cancer_sensitivity_ci_low"], _r["cancer_sensitivity_ci_high"], int(_r["n_cancer"]))
        _rows.append(_row)
    _gap_txt = "; ".join(
        f"{r.attribute}, {MODEL_LABEL[r.model]}: " + (f"{fest(r.gap, r.gap_ci_low, r.gap_ci_high, r.n_eligible_subgroups)} ({r.worst_subgroup} lowest, {r.best_subgroup} highest)"
                                                      if np.isfinite(r.gap) else na(f"{r.n_eligible_subgroups} eligible subgroups"))
        for r in FAIR_GAPS[FAIR_GAPS["model"].isin(_fm)].itertuples())
    export_table("T6", "Performance by Fitzpatrick skin type, age band and sex on pooled out-of-fold predictions (RQ4).", pd.DataFrame(_rows), FAIR, [
        f"BA: balanced accuracy over the diagnostic classes present in the subgroup (number given in 'Classes present'); values from subgroups with different"
        f" classes present are not directly comparable. Cancer sens.: sensitivity for {', '.join(CFG.CANCER_CLASSES)}; n/a (n = k) means no or too few cancer images (k)."
        f" Brackets: {CI_PCT}% patient-level bootstrap confidence intervals ({CFG.N_BOOT} resamples).",
        f"† Subgroup with fewer than {CFG.MIN_SUBGROUP_N} images: reported for completeness, not interpreted, and excluded from the gap. 'Missing': attribute not recorded.",
        "Gap (highest minus lowest BA over eligible subgroups, excluding 'Missing') [CI]: " + _gap_txt + ". The gap is a maximum over noisy differences and is biased upwards.",
        "The raw CSV contains all models and also accuracy and cancer specificity."], landscape=True)
else:
    skip_table("T6", "no subgroup variable (Fitzpatrick, age, sex) was found in the metadata")

# ------------------------------------------------------------------ T7 hyperparameters and training settings
_n_params = (RUN_SUMMARY[RUN_SUMMARY["model"].isin(DEEP_MODELS)].groupby("model")["n_parameters"].first() if "n_parameters" in RUN_SUMMARY.columns else pd.Series(dtype=float))
_epochs = RUN_SUMMARY[(RUN_SUMMARY["scheme"] == "grouped") & RUN_SUMMARY["model"].isin(DEEP_MODELS)] if "best_epoch" in RUN_SUMMARY.columns else pd.DataFrame()
_t7 = [
    ("Cross-validation", f"{CFG.N_REPEATS} × {CFG.N_FOLDS}-fold StratifiedGroupKFold grouped by patient; inner validation split 1/{CFG.INNER_VAL_SPLITS} of each training fold, grouped by patient"),
    ("Random seed (base)", str(CFG.SEED)),
    ("Backbone (timm identifier)", f"{CFG.BACKBONE_REGISTRY[CFG.BACKBONE]}; pretrained = {CFG.PRETRAINED}"),
    ("Image input", f"cache {CFG.CACHE_SIZE} × {CFG.CACHE_SIZE} px ({CFG.CACHE_RESIZE_MODE}); network input {CFG.IMG_SIZE} × {CFG.IMG_SIZE} px"),
    ("Training augmentation", "random resized crop (scale 0.6–1.0, ratio 0.8–1.25), horizontal and vertical flip, colour jitter (brightness 0.1, contrast 0.1, saturation 0.1, hue 0.02)"),
    ("Evaluation view", "resize to network input, no test-time augmentation"),
    ("Optimiser", f"AdamW; learning rate {CFG.LR_BACKBONE:g} (backbone), {CFG.LR_HEAD:g} (new layers); weight decay {CFG.WEIGHT_DECAY:g} (not on biases and normalisation parameters)"),
    ("Schedule", f"{CFG.WARMUP_EPOCHS} warm-up epochs (linear), then cosine decay; at most {CFG.EPOCHS} epochs"),
    ("Batch size", f"{CFG.BATCH_SIZE} × {CFG.GRAD_ACCUM_STEPS} accumulation steps = {CFG.BATCH_SIZE * CFG.GRAD_ACCUM_STEPS} effective"),
    ("Regularisation", f"dropout {CFG.DROPOUT:g}; gradient-norm clipping {CFG.GRAD_CLIP_NORM:g}; label smoothing {CFG.LABEL_SMOOTHING:g}"),
    ("Mixed precision", f"{'torch.amp autocast and gradient scaling' if _amp_enabled() else 'not used'}"),
    ("Class imbalance", "class-weighted cross-entropy, weight = n / (K · n_k) from the training part of each fold (also used as sample weights for gradient boosting)"),
    ("Model selection", f"highest validation balanced accuracy; early stopping after {CFG.PATIENCE} epochs without improvement"),
    ("Fusion head sizes", f"metadata MLP {CFG.META_HIDDEN}; fusion dimension {CFG.FUSION_DIM}; cross-attention heads {CFG.XATTN_HEADS}"),
    ("Modality dropout", f"all fields dropped with probability {CFG.MD_P_FULL:g}; otherwise, with probability {CFG.MD_P_PARTIAL:g}, each field dropped with a per-sample rate from U(0, 1)"),
    ("Metadata preprocessing", "fitted on the training part of each fold: median imputation + missingness indicators + standardisation (numeric); explicit missing category + one-hot (categorical)"),
    ("Metadata input fields", f"{N_FIELDS}: " + ", ".join(FIELD_NAMES)),
    ("Gradient boosting (LightGBM)", ", ".join(f"{k} = {v}" for k, v in CFG.GBM_PARAMS.items()) + f"; early stopping {CFG.GBM_EARLY_STOPPING_ROUNDS} rounds on validation log-loss; "
     f"{CFG.GBM_MD_N_AUG} masked copies for the missingness-augmented variant"),
    ("Late fusion", "log-linear pooling of image-only and metadata-model probabilities; image weight chosen on validation data from {0, 0.05, …, 1}"),
    ("Calibration", "temperature scaling, one temperature per fold fitted on validation data"),
    ("Sensitivity models (R)", (f"same architectures and settings with the metadata input restricted to {len(LOW_MISSING_FIELDS)} low-missingness fields: " + ", ".join(LOW_MISSING_FIELDS))
     if SENS_MODELS else "not run"),
    ("Bootstrap", f"patient-level cluster bootstrap, {CFG.N_BOOT} resamples, {CI_PCT}% percentile intervals"),
]
for _m, _v in _n_params.items():
    _t7.append((f"Trainable parameters: {MODEL_LABEL[_m]}", f"{int(_v):,}"))
if len(_epochs):
    _t7.append(("Selected epoch across runs: median (min–max)", f"{_epochs['best_epoch'].median():g} ({int(_epochs['best_epoch'].min())}–{int(_epochs['best_epoch'].max())})"))
_t7df = pd.DataFrame(_t7, columns=["Setting", "Value"])
export_table("T7", "Hyperparameters and training settings.", _t7df, _t7df.rename(columns={"Setting": "setting", "Value": "value"}),
             [f"All values are read from the configuration used for the runs (configuration hash {CFG_HASH}). No hyperparameter search was performed; the same settings were used for all folds."])

# ------------------------------------------------------------------ T8 literature comparison (from the user's file only)
LIT_COLUMNS = ["method", "metric", "value", "split_protocol", "citation", "page_or_table"]
LIT_PATH = DIRS["project"] / "literature_baselines.csv"
if not LIT_PATH.is_file():
    pd.DataFrame(columns=LIT_COLUMNS).to_csv(LIT_PATH, index=False)
    print(f"[info] empty literature template created: {LIT_PATH}")
LIT = pd.read_csv(LIT_PATH, dtype=str, keep_default_na=False)
_missing_cols = [c for c in LIT_COLUMNS if c not in LIT.columns]
if _missing_cols:
    raise ValueError(f"{LIT_PATH} lacks the columns {_missing_cols}. Required columns: {LIT_COLUMNS}.")
LIT = LIT[LIT_COLUMNS].apply(lambda s: s.str.strip())
LIT = LIT[(LIT != "").any(axis=1)].reset_index(drop=True)
if len(LIT) == 0:
    skip_table("T8", f"the literature file {LIT_PATH.name} has no rows yet; fill it from the original papers and rerun Parts 15-17")
else:
    _num = pd.to_numeric(LIT["value"], errors="coerce")
    if _num.isna().any():
        raise ValueError(f"Non-numeric 'value' entries in {LIT_PATH.name}, rows {list(np.flatnonzero(_num.isna()) + 2)} (spreadsheet row numbers).")
    _ours = CFG.SPLIT_PROTOCOL_LABEL.strip().lower()
    _comp = ["same protocol label" if p.strip().lower() == _ours else "not directly comparable" for p in LIT["split_protocol"]]
    _d = pd.DataFrame({"Method": LIT["method"], "Metric": LIT["metric"], "Reported value": LIT["value"], "Split protocol": LIT["split_protocol"],
                       "Source": [f"{c}, {p}" if p else c for c, p in zip(LIT["citation"], LIT["page_or_table"])], "Comparability": _comp})
    export_table("T8", "Published results on PAD-UFES-20 as reported by their authors.", _d, LIT.assign(comparability=_comp), [
        f"All rows are transcribed by the authors of this study from the cited sources into {LIT_PATH.name}; none is generated by the analysis code.",
        f"Protocol of this study: {CFG.SPLIT_PROTOCOL_LABEL}. Rows with a different split protocol are marked 'not directly comparable'. Even with the same protocol"
        " label, published results use different splits, preprocessing and model selection, so no row is strictly comparable with Table 2."], landscape=True)

# ------------------------------------------------------------------ T9 software versions and hardware
_t9 = [(k.replace("_", " "), str(v)) for k, v in ENV_INFO.items()]
_t9df = pd.DataFrame(_t9, columns=["Item", "Value"])
export_table("T9", "Software versions and hardware used for the reported runs.", _t9df, _t9df.rename(columns={"Item": "item", "Value": "value"}), [
    "Recorded automatically in the session that produced the tables and figures; the complete package list is in logs/requirements_lock.txt."
    " Deterministic algorithms were requested; mixed-precision arithmetic and some CUDA kernels remain non-deterministic at the level of the last decimals."])

# ------------------------------------------------------------------ T10 leakage comparison
if CFG.RUN_LEAKAGE and len(LEAKAGE):
    _d = pd.DataFrame({
        "Model": [MODEL_LABEL[m] for m in LEAKAGE["model"]],
        "Metric": [METRIC_LABELS[m] for m in LEAKAGE["metric"]],
        "Patient-grouped split": [fest(r.grouped_pooled, r.grouped_ci_low, r.grouped_ci_high, N) for r in LEAKAGE.itertuples()],
        "Image-level split": [fest(r.image_level_pooled, r.image_level_ci_low, r.image_level_ci_high, N) for r in LEAKAGE.itertuples()],
        "Difference (image-level − grouped)": [fest(r.inflation, r.inflation_ci_low, r.inflation_ci_high, N) for r in LEAKAGE.itertuples()],
        "p": [fp(r.p_boot, CFG.N_BOOT) for r in LEAKAGE.itertuples()],
    })
    export_table("T10", "Effect of the split protocol: patient-grouped versus image-level cross-validation for the same models.", _d, LEAKAGE, [
        f"Pooled out-of-fold estimates with {CI_PCT}% patient-level bootstrap confidence intervals ({CFG.N_BOOT} resamples). Image-level split: {CFG.N_FOLDS}-fold"
        " stratified cross-validation on images, ignoring patient identity (inner validation split also on images). Difference and p-value: paired patient-level bootstrap."])
else:
    skip_table("T10", "leakage analysis disabled (CFG.RUN_LEAKAGE = False)")
# ------------------------------------------------------------------ T11 / T12 metadata-missingness audit
if AUDIT_ASSOC is not None:
    _thr = f"{100 * CFG.AUDIT_ROW_MISSING_SHARE:.0f}%"
    _rule = dict(variant=f"Rule without any model: at least {_thr} of fields missing ⇒ non-cancer",
                 cancer_sensitivity=AUDIT_ASSOC["rule_sensitivity"], cancer_sensitivity_ci_low=AUDIT_ASSOC["rule_sensitivity_ci_low"],
                 cancer_sensitivity_ci_high=AUDIT_ASSOC["rule_sensitivity_ci_high"], cancer_specificity=AUDIT_ASSOC["rule_specificity"],
                 cancer_specificity_ci_low=AUDIT_ASSOC["rule_specificity_ci_low"], cancer_specificity_ci_high=AUDIT_ASSOC["rule_specificity_ci_high"])
    _a = pd.concat([pd.DataFrame([_rule]), AUDIT_MODELS], ignore_index=True)
    _not_def = "not defined (binary rule)"
    _d = pd.DataFrame({
        "Predictor": _a["variant"],
        "Balanced accuracy (6 classes)": [fest(r.balanced_accuracy, r.balanced_accuracy_ci_low, r.balanced_accuracy_ci_high, N) if pd.notna(r.balanced_accuracy) else _not_def for r in _a.itertuples()],
        "Macro AUROC": [fest(r.auroc_macro_ovr, r.auroc_macro_ovr_ci_low, r.auroc_macro_ovr_ci_high, N) if pd.notna(r.auroc_macro_ovr) else _not_def for r in _a.itertuples()],
        "Cancer sensitivity": [fest(r.cancer_sensitivity, r.cancer_sensitivity_ci_low, r.cancer_sensitivity_ci_high, N) for r in _a.itertuples()],
        "Cancer specificity": [fest(r.cancer_specificity, r.cancer_specificity_ci_low, r.cancer_specificity_ci_high, N) for r in _a.itertuples()],
    })
    export_table("T11", "Audit of the association between missing metadata and diagnosis: predictors that use the missingness pattern or restricted metadata.", _d, _a, [
        f"Images with at least {_thr} of the {N_FIELDS} metadata input fields missing: {int(MOSTLY_MISSING.sum())} of {N}"
        f" (cancer {AUDIT_ASSOC['n_cancer_missing']} of {AUDIT_ASSOC['n_cancer_missing'] + AUDIT_ASSOC['n_cancer_recorded']};"
        f" non-cancer {AUDIT_ASSOC['n_noncancer_missing']} of {AUDIT_ASSOC['n_noncancer_missing'] + AUDIT_ASSOC['n_noncancer_recorded']})."
        " Share by class: " + ", ".join(f"{c} {v:.3f}" for c, v in AUDIT_ASSOC["missing_by_class"].items()) + ".",
        f"Model rows: gradient-boosting models trained on the same patient-grouped folds and settings as the metadata-only baseline; pooled out-of-fold estimates"
        f" with {CI_PCT}% patient-level bootstrap confidence intervals ({CFG.N_BOOT} resamples). Low-missingness fields: fields missing in at most"
        f" {100 * CFG.AUDIT_LOW_MISSING_MAX:.0f}% of images.",
        f"Cancer: {', '.join(CFG.CANCER_CLASSES)}. No value is bolded: the table quantifies a property of the data, not a ranking of methods."], landscape=True)

    _st = AUDIT_STRATA.set_index(["model", "stratum"])
    _rec, _mis = _st.xs("recorded", level="stratum").loc[MODELS], _st.xs("mostly missing", level="stratum").loc[MODELS]
    _d = pd.DataFrame({
        "Model": [MODEL_LABEL[m] for m in MODELS],
        "BA, recorded": [fest(r.balanced_accuracy, r.balanced_accuracy_ci_low, r.balanced_accuracy_ci_high, int(r.n_images)) for r in _rec.itertuples()],
        "Cancer sens., recorded": [fest(r.cancer_sensitivity, r.cancer_sensitivity_ci_low, r.cancer_sensitivity_ci_high, int(r.n_cancer)) for r in _rec.itertuples()],
        "Cancer spec., recorded": [fest(r.cancer_specificity, r.cancer_specificity_ci_low, r.cancer_specificity_ci_high, int(r.n_non_cancer)) for r in _rec.itertuples()],
        "BA, mostly missing": [fest(r.balanced_accuracy, r.balanced_accuracy_ci_low, r.balanced_accuracy_ci_high, int(r.n_images)) for r in _mis.itertuples()],
        "Cancer sens., mostly missing": [fest(r.cancer_sensitivity, r.cancer_sensitivity_ci_low, r.cancer_sensitivity_ci_high, int(r.n_cancer)) for r in _mis.itertuples()],
        "Cancer spec., mostly missing": [fest(r.cancer_specificity, r.cancer_specificity_ci_low, r.cancer_specificity_ci_high, int(r.n_non_cancer)) for r in _mis.itertuples()],
    })
    _b = pd.DataFrame({c: False for c in _d.columns}, index=_d.index)
    for _col, _src, _frame in (("BA, recorded", "balanced_accuracy", _rec), ("Cancer sens., recorded", "cancer_sensitivity", _rec), ("Cancer spec., recorded", "cancer_specificity", _rec)):
        _b[_col] = best_mask(_frame[_src].to_numpy(), "max")
    _r0, _m0 = _rec.iloc[0], _mis.iloc[0]
    export_table("T12", "Performance of all models by availability of metadata, from pooled out-of-fold predictions.", _d, AUDIT_STRATA, [
        f"Recorded: images with fewer than {_thr} of metadata fields missing (n = {int(_r0.n_images)}; cancer {int(_r0.n_cancer)}, non-cancer {int(_r0.n_non_cancer)};"
        f" {int(_r0.n_classes_present)} classes present). Mostly missing: the remaining images (n = {int(_m0.n_images)}; cancer {int(_m0.n_cancer)}, non-cancer {int(_m0.n_non_cancer)};"
        f" {int(_m0.n_classes_present)} classes present).",
        f"BA: balanced accuracy over the classes present in the stratum; values of the two strata are not comparable with each other when different classes are present."
        f" n/a (n = k): not computable because the stratum contains k images of the required kind. Brackets: {CI_PCT}% patient-level bootstrap confidence intervals.",
        f"Bold: highest value among models within the stratum with recorded metadata. {MD_NOTE}."], bold=_b, landscape=True)
else:
    skip_table("T11", AUDIT_SKIP_REASON)
    skip_table("T12", AUDIT_SKIP_REASON)
# ------------------------------------------------------------------ T13 restricted-metadata sensitivity analysis
if len(SENS):
    _is_ref = SENS["model"] == "image_only"
    _ref_txt = "reference"
    _d = pd.DataFrame({
        "Model": [MODEL_LABEL[m] for m in SENS["model"]],
        "Balanced accuracy": [fest(r.balanced_accuracy, r.balanced_accuracy_ci_low, r.balanced_accuracy_ci_high, N) for r in SENS.itertuples()],
        "Macro AUROC": [fest(r.auroc_macro_ovr, r.auroc_macro_ovr_ci_low, r.auroc_macro_ovr_ci_high, N) for r in SENS.itertuples()],
        "Cancer sens. / spec.": [f"{fnum(r.cancer_sensitivity, N)} / {fnum(r.cancer_specificity, N)}" for r in SENS.itertuples()],
        "Δ BA vs image only [CI], p": [_ref_txt if r.model == "image_only" else
                                       f"{fest(r.ba_diff_vs_image_only, r.ba_diff_vs_image_only_ci_low, r.ba_diff_vs_image_only_ci_high, CFG.N_BOOT)}, {fp(r.p_holm_vs_image_only, CFG.N_BOOT)}"
                                       for r in SENS.itertuples()],
        "Δ BA vs same model, all metadata [CI], p": [_ref_txt if r.model == "image_only" else
                                                     f"{fest(r.ba_diff_vs_full_metadata, r.ba_diff_vs_full_metadata_ci_low, r.ba_diff_vs_full_metadata_ci_high, CFG.N_BOOT)}, {fp(r.p_holm_vs_full_metadata, CFG.N_BOOT)}"
                                                     for r in SENS.itertuples()],
        "Recorded metadata: cancer sens. / spec.": [f"{fnum(r.recorded_cancer_sensitivity, int(r.n_recorded_cancer))} / {fnum(r.recorded_cancer_specificity, int(r.n_recorded_non_cancer))}"
                                                    for r in SENS.itertuples()],
    })
    _b = pd.DataFrame({c: False for c in _d.columns}, index=_d.index)
    _b["Balanced accuracy"] = best_mask(SENS["balanced_accuracy"].to_numpy(), "max")
    _b["Macro AUROC"] = best_mask(SENS["auroc_macro_ovr"].to_numpy(), "max")
    export_table("T13", "Sensitivity analysis: metadata and fusion models restricted to metadata fields with little missingness.", _d, SENS, [
        f"(R): the model's metadata input is restricted to the fields missing in at most {100 * CFG.AUDIT_LOW_MISSING_MAX:.0f}% of images ({', '.join(LOW_MISSING_FIELDS)}),"
        " so that the pattern of missing fields cannot serve as a proxy for the diagnosis; architectures, folds, seeds and settings are those of Table 2.",
        f"Pooled out-of-fold estimates with {CI_PCT}% patient-level bootstrap confidence intervals ({CFG.N_BOOT} resamples). Δ BA: paired bootstrap difference in balanced accuracy;"
        f" p-values are Holm-adjusted over the {int(2 * (len(SENS) - 1))} comparisons of this table. 'Same model, all metadata' is the corresponding row of Table 2.",
        f"Recorded metadata: images with fewer than {100 * CFG.AUDIT_ROW_MISSING_SHARE:.0f}% of all metadata fields missing"
        f" (cancer {int(SENS['n_recorded_cancer'].iloc[0])}, non-cancer {int(SENS['n_recorded_non_cancer'].iloc[0])}).",
        "Bold: highest value in the column. Dropping fields also removes clinical information, so these results are a conservative lower bound on the value of metadata."],
        bold=_b, landscape=True)
else:
    skip_table("T13", SENS_SKIP_REASON)
save_json(dict(tables=TABLE_REGISTRY, skipped=TABLE_SKIPPED), DIRS["qc"] / "table_registry.json")

## Part 16. Automated quality-control report

**Purpose:** Verify the integrity of the experiment and of every exported file, print PASS / FAIL for each check, and stop if any check fails.

**Why:** A leakage bug, a metric bug or a mis-sized figure would invalidate the paper and is easy to miss by eye. The checks are executed against the files on disk, not against the variables that produced them, so they also catch stale or partially written outputs.

**Method choice:** The specification asks that QC pass "before export", but several checks inspect the exported files themselves. The order used here resolves this: Part 15 writes tables and figures into the results folder, this cell audits them, and the release step (results summary and zip archive, Part 17) runs only if every check passes. Four result states are used: **PASS**; **FAIL** (stops the notebook and prints how to fix it); **SKIP** (an optional output was switched off or an input is not yet supplied, with the reason); **WARN** (the run is valid as a test but not reportable, e.g. `FAST_MODE`). The check "no test-fold data used for early stopping, temperature scaling or thresholds" is a provenance audit: it verifies that the data used for those steps are exactly the validation parts and that these are disjoint from the test folds; together with the structure of the training function (Part 6), which only receives training and validation indices, this is the available evidence. Independent re-computation of table values uses scikit-learn and separately written reference implementations, not the functions that produced the tables.

**Output:** `qc_report/qc_report.csv`, `qc_report/qc_report.txt`; the variable `QC_PASSED`.

In [ ]:
QC: List[Dict[str, str]] = []


def qc_add(check: str, ok: Optional[bool], detail: str, fix: str = "", status: Optional[str] = None) -> None:
    """Record one QC result. `status` overrides the PASS/FAIL derived from `ok` (used for SKIP and WARN)."""
    QC.append(dict(check=check, status=status or ("PASS" if ok else "FAIL"), detail=detail, fix="" if (status or ok) else fix))


def _run_check(check: str, fn: Callable[[], Tuple[bool, str]], fix: str) -> None:
    """Run a check function; a failing assertion inside it is reported as FAIL with its message."""
    try:
        ok, detail = fn()
    except AssertionError as exc:
        ok, detail = False, f"assertion failed: {exc}"
    qc_add(check, ok, detail, fix)


# 1 ------------------------------------------------------------ patient overlap (from the saved split file)
def _qc_overlap() -> Tuple[bool, str]:
    saved = pd.read_csv(DIRS["splits"] / "fold_assignments.csv", dtype={"img_id": str})
    saved = saved[saved["scheme"] == "grouped"].merge(pd.DataFrame({"img_id": IMG_IDS, "patient": PATIENTS}), on="img_id", how="left")
    assert saved["patient"].notna().all(), "split file contains unknown image ids"
    n_bad, n_splits = 0, 0
    for (_, _), g in saved.groupby(["repeat", "fold"]):
        parts = {r: set(g.loc[g["role"] == r, "patient"]) for r in ("train", "val", "test")}
        n_bad += len(parts["train"] & parts["test"]) + len(parts["val"] & parts["test"]) + len(parts["train"] & parts["val"])
        n_splits += 1
    # identical images must not sit on both sides of a split either
    digest = pd.Series([hashlib.md5(IMAGES[i].tobytes()).hexdigest() for i in range(N)], index=IMG_IDS)
    n_dup_leak = 0
    for (_, _), g in saved.groupby(["repeat", "fold"]):
        d = {r: set(digest.loc[g.loc[g["role"] == r, "img_id"]]) for r in ("train", "val", "test")}
        n_dup_leak += len((d["train"] | d["val"]) & d["test"])
    return (n_bad == 0 and n_dup_leak == 0), (f"{n_splits} grouped splits checked; patients shared between train / validation / test: {n_bad}; "
                                              f"identical images shared between train/validation and test: {n_dup_leak}")


_run_check("Zero patient overlap between train, validation and test in every fold", _qc_overlap,
           "Delete splits/fold_assignments.csv and all predictions, then rerun: the saved splits are not patient-grouped.")


# 2 ------------------------------------------------------------ preprocessing fitted on training rows only
def _qc_preprocessing() -> Tuple[bool, str]:
    n_checked = 0
    for (s, r, f), prep in PREP.items():
        tr, va, te = get_split(s, r, f)
        base = DIRS["preprocessing"] / f"{s}_r{r}_f{f}"
        fit_ids = pd.read_csv(str(base) + "_fit_ids.csv", dtype=str)["img_id"].to_numpy()
        assert set(fit_ids) == set(IMG_IDS[tr]), f"{s} r{r} f{f}: saved fit ids are not exactly the training ids"
        assert not (set(fit_ids) & (set(IMG_IDS[va]) | set(IMG_IDS[te]))), f"{s} r{r} f{f}: fit ids overlap validation/test"
        saved = joblib.load(str(base) + ".joblib")
        if NUM_COLS:
            assert int(np.max(saved.named_transformers_["num"].named_steps["scale"].n_samples_seen_)) == len(tr), "scaler saw a different number of rows"
        refit = build_preprocessor().fit(META.iloc[tr])
        assert np.allclose(saved.transform(META), refit.transform(META), atol=1e-9), f"{s} r{r} f{f}: saved pipeline differs from a training-only refit"
        rng = np.random.default_rng([CFG.SEED, r, f, 5])
        rows = rng.choice(N, size=min(48, N), replace=False)
        fm = rng.random((len(rows), N_FIELDS)) < 0.5
        assert np.allclose(apply_field_mask(prep["X"][rows], fm, prep), saved.transform(mask_raw(META.iloc[rows], fm)), atol=1e-5), "field masking is not equivalent to raw masking"
        n_checked += 1
    return True, f"{n_checked} saved pipelines equal a refit on training rows only; fit ids disjoint from validation and test rows"


_run_check("Preprocessing objects fitted only on training indices", _qc_preprocessing,
           "Rerun Part 5; if it persists, the preprocessing was changed to use non-training rows.")


# 3 ------------------------------------------------------------ no test data in early stopping / temperature / thresholds
def _qc_no_test_tuning() -> Tuple[bool, str]:
    n_runs = 0
    for s, models in SCHEME_MODELS.items():
        for m in models:
            for r in range(CFG.N_REPEATS):
                for f in range(CFG.N_FOLDS):
                    fpths = run_files(s, m, r, f)
                    _, va, te = get_split(s, r, f)
                    val_tab, val_logits = read_logits(fpths["val"])
                    test_tab, _ = read_logits(fpths["test"])
                    val_ids, test_ids = set(val_tab["img_id"]), set(test_tab["img_id"])
                    assert val_ids == set(IMG_IDS[va]) and test_ids == set(IMG_IDS[te]), f"{s}/{m} r{r} f{f}: prediction ids differ from the split"
                    assert not (val_ids & test_ids), f"{s}/{m} r{r} f{f}: validation and test images overlap"
                    if s == "grouped":
                        assert not (set(val_tab["patient_id"]) & set(test_tab["patient_id"])), f"{s}/{m} r{r} f{f}: validation and test patients overlap"
                    assert load_json(fpths["done"])["val_ids_digest"] == ids_digest(IMG_IDS[va]), "model-selection fingerprint is not the validation part"
                    assert abs(fit_temperature(val_logits, Y[va]) - TEMPS[(s, m, r, f)]) < 1e-4, "temperature is not reproducible from the validation file"
                    n_runs += 1
    thr = pd.read_csv(DIRS["logs"] / "rq3_selective_thresholds.csv")
    for row in thr.itertuples():
        assert int(row.n_val) == len(get_split("grouped", int(row.repeat), int(row.fold))[1]), "threshold was not computed on the validation part"
    return True, (f"{n_runs} runs: model selection, temperature scaling and thresholds use exactly the validation parts, which are disjoint "
                  "from the test folds (provenance audit; temperatures re-fitted from the validation files match)")


_run_check("No test-fold data used for early stopping, temperature scaling or thresholds", _qc_no_test_tuning,
           "Rerun Parts 6-11 without modifying the split handling.")


# 4 ------------------------------------------------------------ metric functions against scikit-learn / reference code
def _qc_metrics() -> Tuple[bool, str]:
    n = 0
    for seed in (11, 12, 13):
        res = validate_metrics_against_sklearn(seed=seed, n=350)
        assert all(res.values()), f"mismatch with scikit-learn: {[k for k, v in res.items() if not v]}"
        n += len(res)
    rng = np.random.default_rng(3)
    conf, corr = rng.random(500), (rng.random(500) < 0.6).astype(float)
    ece_ref = 0.0
    for b in range(CFG.ECE_BINS):                                  # textbook loop implementation
        lo, hi = b / CFG.ECE_BINS, (b + 1) / CFG.ECE_BINS
        inb = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        if inb.any():
            ece_ref += inb.mean() * abs(corr[inb].mean() - conf[inb].mean())
    assert abs(ece_equal_width(conf, corr, CFG.ECE_BINS) - ece_ref) < 1e-12, "ECE differs from the reference implementation"
    order = np.argsort(-conf, kind="stable")
    aurc_ref = float(np.mean([1.0 - corr[order[:k]].mean() for k in range(1, len(conf) + 1)]))
    assert abs(aurc(conf, corr) - aurc_ref) < 1e-12, "AURC differs from the reference implementation"
    yb = rng.integers(0, 3, 200)
    Pb = rng.dirichlet(np.ones(3), 200)
    brier_ref = float(np.mean([sum((Pb[i, k] - (1.0 if yb[i] == k else 0.0)) ** 2 for k in range(3)) for i in range(200)]))
    assert abs(brier_multiclass(yb, Pb) - brier_ref) < 1e-12, "Brier score differs from the reference implementation"
    ybin, sa, sb = rng.integers(0, 2, 300), rng.random(300), rng.random(300)
    dl = delong_test(ybin, sa, sb)
    assert abs(dl["auc_a"] - skm.roc_auc_score(ybin, sa)) < 1e-9, "DeLong AUC differs from scikit-learn"
    assert np.allclose(holm_adjust([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06]), "Holm adjustment unit test failed"
    return True, f"{n} metric comparisons with scikit-learn agree (1e-9); ECE, AURC, Brier, DeLong AUC and Holm match reference implementations"


_run_check("All metric functions validated against scikit-learn on a small test case", _qc_metrics,
           "A metric function was edited incorrectly; restore the original function in Parts 8, 9 or 11.")


# 5-7 ---------------------------------------------------------- figure files, sizes, dpi, mode, fonts, alpha
EXPECTED_FIGS = [f"F{i}" for i in range(1, 12)]


def _qc_fig_files() -> Tuple[bool, str]:
    n = 0
    for fid in EXPECTED_FIGS:
        if fid in FIG_SKIPPED:
            continue
        assert fid in FIG_REGISTRY, f"{fid} was neither produced nor recorded as skipped"
        for wk, w_mm in CFG.FIG_WIDTHS_MM.items():
            assert wk in FIG_REGISTRY[fid], f"{fid}: width '{wk}' missing"
            for ext in FIG_FORMATS:
                p = Path(FIG_REGISTRY[fid][wk]["files"][ext])
                assert p.is_file() and p.stat().st_size > 0, f"missing or empty file {p.name}"
                assert p.parent == DIRS[f"fig_{wk}"], f"{p.name} is not in the {wk} folder"
            for ext in ("png", "tiff"):
                with Image.open(FIG_REGISTRY[fid][wk]["files"][ext]) as im:
                    assert im.width == mm_to_px(w_mm), f"{fid} {wk} .{ext}: width {im.width} px, expected {mm_to_px(w_mm)} px"
                    assert im.height <= mm_to_px(CFG.FIG_MAX_HEIGHT_MM) + 1, f"{fid} {wk}: height exceeds {CFG.FIG_MAX_HEIGHT_MM} mm"
            assert (DIRS["captions"] / f"{fid}_caption.txt").read_text(encoding="utf-8").strip(), f"{fid}: caption file is empty"
            n += 1
    return True, (f"{n} figure/width combinations: PNG, TIFF, PDF and SVG exist; raster widths are "
                  + " and ".join(f"{mm_to_px(v)} px ({v:g} mm)" for v in CFG.FIG_WIDTHS_MM.values()) + f" at {CFG.FIG_DPI} dpi (read back with PIL)")


def _qc_fig_png() -> Tuple[bool, str]:
    n = 0
    for fid, widths in FIG_REGISTRY.items():
        for wk, rec in widths.items():
            with Image.open(rec["files"]["png"]) as im:
                dpi = im.info.get("dpi", (0, 0))
                assert abs(dpi[0] - CFG.FIG_DPI) < 0.05 and abs(dpi[1] - CFG.FIG_DPI) < 0.05, f"{fid} {wk} PNG dpi = {dpi}"
                assert im.mode == "RGB", f"{fid} {wk} PNG mode = {im.mode} (must be RGB without alpha)"
            with Image.open(rec["files"]["tiff"]) as im:
                dpi = im.info.get("dpi", (0, 0))
                assert abs(float(dpi[0]) - CFG.FIG_DPI) < 0.05, f"{fid} {wk} TIFF dpi = {dpi}"
                assert im.mode == "RGB" and im.info.get("compression") == "tiff_lzw", f"{fid} {wk} TIFF mode/compression = {im.mode}/{im.info.get('compression')}"
            pdf_bytes = Path(rec["files"]["pdf"]).read_bytes()
            assert b"/Type3" not in pdf_bytes, f"{fid} {wk} PDF contains Type 3 (bitmap) fonts"
            n += 1
    return True, (f"{n} PNG files: dpi metadata {CFG.FIG_DPI} (stored as whole pixels per metre, i.e. within 0.01 dpi), mode RGB, no alpha channel; "
                  "TIFF files RGB with LZW compression; PDF files contain no Type 3 fonts")


def _qc_fig_style() -> Tuple[bool, str]:
    fonts = [rec["min_font_pt"] for widths in FIG_REGISTRY.values() for rec in widths.values() if rec["min_font_pt"] is not None]
    alphas = [rec["min_alpha"] for widths in FIG_REGISTRY.values() for rec in widths.values()]
    assert fonts and min(fonts) >= CFG.MIN_FONT_PT - 1e-6, f"smallest font is {min(fonts) if fonts else 'unknown'} pt"
    assert min(alphas) >= 1.0, f"an artist has alpha {min(alphas)}"
    return True, f"smallest font over all figures: {min(fonts):g} pt (limit {CFG.MIN_FONT_PT:g} pt); smallest artist alpha: {min(alphas):g}; font family: {FONT_USED}"


_fig_fix = "Rerun Part 15. If a figure function was edited, keep new_figure()/save_figure() and do not use bbox_inches='tight', alpha or font sizes below 7 pt."
_run_check("Every figure exists in both widths and all formats, with pixel widths matching 90 mm and 190 mm at 600 dpi", _qc_fig_files, _fig_fix)
_run_check("PNG dpi metadata equals 600 and mode is RGB with no alpha channel", _qc_fig_png, _fig_fix)
_run_check("Minimum font size >= 7 pt and no artist with alpha < 1", _qc_fig_style, _fig_fix)
for _fid, _why in FIG_SKIPPED.items():
    qc_add(f"Figure {_fid}", None, f"not produced: {_why}", status="SKIP")
if FONT_USED != "Arial":
    qc_add("Figure font is Arial", None, f"Arial is not installed in this runtime; '{FONT_USED}' was used. Rerun Part 15 where the Arial installation succeeds "
           "before submitting figures.", status="WARN")


# 8 ------------------------------------------------------------ table numbers recomputed from the saved OOF files
def _qc_tables_vs_oof() -> Tuple[bool, str]:
    t2 = pd.read_csv(TABLE_REGISTRY["T2"]["files"]["raw"])
    t4 = pd.read_csv(TABLE_REGISTRY["T4"]["files"]["raw"])
    n_vals = 0
    tol = 5e-5
    for m in MODELS:
        oof = pd.read_csv(DIRS["predictions"] / f"oof_grouped_{m}.csv", dtype={"img_id": str})
        assert set(oof["repeat"]) == set(range(CFG.N_REPEATS)) and (oof.groupby("repeat")["img_id"].nunique() == N).all(), f"{m}: OOF file incomplete"
        prim = oof[oof["repeat"] == PRIMARY_REPEAT]
        y = prim["y_true"].to_numpy()
        P = prim[[f"prob_{c}" for c in CLASS_NAMES]].to_numpy()
        Pts = prim[[f"prob_ts_{c}" for c in CLASS_NAMES]].to_numpy()
        pred = P.argmax(axis=1)
        yc, pc = np.isin(y, CANCER_IDX), np.isin(pred, CANCER_IDX)
        ref = {"balanced_accuracy": skm.balanced_accuracy_score(y, pred), "macro_f1": skm.f1_score(y, pred, average="macro"),
               "auroc_macro_ovr": skm.roc_auc_score(y, P / P.sum(axis=1, keepdims=True), multi_class="ovr", average="macro", labels=list(range(K))),
               "cancer_sensitivity": skm.recall_score(yc, pc), "cancer_specificity": skm.recall_score(~yc, ~pc)}
        fold_ba = [skm.balanced_accuracy_score(g["y_true"], g[[f"prob_{c}" for c in CLASS_NAMES]].to_numpy().argmax(axis=1))
                   for _, g in oof.groupby(["repeat", "fold"])]
        for met, val in ref.items():
            row = t2[(t2["model"] == m) & (t2["metric"] == met)].iloc[0]
            assert abs(row["pooled"] - val) < tol, f"T2 {m} {met}: table {row['pooled']:.6f} vs recomputed {val:.6f}"
            n_vals += 1
        row = t2[(t2["model"] == m) & (t2["metric"] == "balanced_accuracy")].iloc[0]
        assert abs(row["fold_mean"] - np.mean(fold_ba)) < tol and abs(row["fold_sd"] - np.std(fold_ba, ddof=1)) < tol, f"T2 {m}: fold mean/SD mismatch"
        n_vals += 2
        c4 = t4[t4["model"] == m].iloc[0]
        for tag, Pq in (("raw", P), ("ts", Pts)):
            onehot = np.eye(K)[y]
            brier_ref = float(np.mean(np.sum((Pq - onehot) ** 2, axis=1)))
            conf, corr = Pq.max(axis=1), (Pq.argmax(axis=1) == y).astype(float)
            edges = np.linspace(0.0, 1.0, CFG.ECE_BINS + 1)
            ece_ref = sum(((conf > lo) & (conf <= hi)).mean() * abs(corr[(conf > lo) & (conf <= hi)].mean() - conf[(conf > lo) & (conf <= hi)].mean())
                          for lo, hi in zip(edges[:-1], edges[1:]) if ((conf > lo) & (conf <= hi)).any())
            assert abs(c4[f"brier_{tag}"] - brier_ref) < tol, f"T4 {m} Brier ({tag}) mismatch"
            assert abs(c4[f"ece_{tag}"] - ece_ref) < 5e-4, f"T4 {m} ECE ({tag}): table {c4[f'ece_{tag}']:.6f} vs recomputed {ece_ref:.6f}"
            n_vals += 2
    if "T6" in TABLE_REGISTRY:
        t6 = pd.read_csv(TABLE_REGISTRY["T6"]["files"]["raw"])
        for attr, series in SUBGROUPS.items():
            counts = series.astype(str).value_counts()
            for row in t6[t6["attribute"] == attr].itertuples():
                assert int(row.n_images) == int(counts[str(row.subgroup)]), f"T6 {attr} {row.subgroup}: n mismatch"
                n_vals += 1
    if "T5" in TABLE_REGISTRY:
        t5 = pd.read_csv(TABLE_REGISTRY["T5"]["files"]["raw"])
        for m in MODELS:       # at 0% masking the robustness value must equal the main result
            a = float(t5[t5["model"] == m]["ba_0"].iloc[0])
            b = float(t2[(t2["model"] == m) & (t2["metric"] == "balanced_accuracy")]["pooled"].iloc[0])
            assert abs(a - b) < 2e-3, f"T5 {m}: balanced accuracy at 0% masking ({a:.4f}) differs from T2 ({b:.4f})"
            n_vals += 1
    return True, (f"{n_vals} table values recomputed from predictions/oof_grouped_*.csv with scikit-learn or reference code agree with the exported tables "
                  f"(T2 pooled estimates and fold mean/SD of balanced accuracy, T4 Brier and ECE before/after scaling, T5 at 0% masking, T6 subgroup sizes)")


_run_check("Table values match the saved out-of-fold prediction files when recomputed", _qc_tables_vs_oof,
           "Tables are stale relative to the prediction files: rerun Parts 7-15 in order.")


# 9 ------------------------------------------------------------ literature table
def _qc_literature() -> Tuple[bool, str]:
    src = pd.read_csv(LIT_PATH, dtype=str, keep_default_na=False)[LIT_COLUMNS].apply(lambda s: s.str.strip())
    src = src[(src != "").any(axis=1)].reset_index(drop=True)
    out = pd.read_csv(TABLE_REGISTRY["T8"]["files"]["raw"], dtype=str, keep_default_na=False)
    assert len(out) == len(src) and (out[LIT_COLUMNS].to_numpy() == src.to_numpy()).all(), "T8 rows are not identical to the rows of the literature file"
    empty = src[(src["citation"] == "") | (src["method"] == "") | (src["metric"] == "") | (src["split_protocol"] == "")]
    assert len(empty) == 0, f"rows without method, metric, split protocol or citation: spreadsheet rows {list(empty.index + 2)}"
    flagged = int((out["comparability"] == "not directly comparable").sum())
    return True, f"{len(out)} rows, all from {LIT_PATH.name}, each with a citation; {flagged} flagged 'not directly comparable'"


if "T8" in TABLE_REGISTRY:
    _run_check("Literature table contains only rows from the user's file, each with a citation", _qc_literature,
               f"Complete the citation, method, metric and split_protocol of every row in {LIT_PATH.name} and rerun Parts 15-17.")
for _tid, _why in TABLE_SKIPPED.items():
    qc_add(f"Table {_tid}", None, f"not produced: {_why}", status="SKIP")


# 10 ----------------------------------------------------------- no NaN / placeholder text in tables and captions
def _qc_no_placeholders() -> Tuple[bool, str]:
    import re
    # Running text: the token 'nan' and common placeholder markers. Table cells: additionally no empty cell
    # and no cell that consists only of a null-like word.
    pattern = re.compile(r"(?<![A-Za-z])nan(?![A-Za-z])|TODO|TBD|PLACEHOLDER|XXX|\?\?\?|lorem ipsum", re.IGNORECASE)
    null_cells = {"", "nan", "none", "null", "inf", "-inf", "na", "n/a"}
    hits, n_files = [], 0
    for tid, rec in TABLE_REGISTRY.items():
        texts = {"formatted.csv": Path(rec["files"]["formatted"]).read_text(encoding="utf-8"), "tex": Path(rec["files"]["tex"]).read_text(encoding="utf-8")}
        d = docx.Document(rec["files"]["docx"])
        texts["docx"] = "\n".join([p.text for p in d.paragraphs] + [c.text for t in d.tables for row in t.rows for c in row.cells])
        cells = [str(v) for v in pd.read_csv(rec["files"]["formatted"], dtype=str, keep_default_na=False).to_numpy().ravel()]
        cells += [c.text for t in d.tables for row in t.rows for c in row.cells]
        bad_cells = [c for c in cells if c.strip().lower() in null_cells]
        if bad_cells:
            hits.append(f"{tid}: {len(bad_cells)} empty or null-like cell(s), e.g. '{bad_cells[0]}'")
        for kind, text in texts.items():
            n_files += 1
            m = pattern.search(text)
            if m:
                hits.append(f"{tid} ({kind}): '{m.group(0)}'")
            assert text.strip(), f"{tid} ({kind}) is empty"
    for fid in FIG_REGISTRY:
        text = (DIRS["captions"] / f"{fid}_caption.txt").read_text(encoding="utf-8")
        n_files += 1
        m = pattern.search(text)
        if m:
            hits.append(f"caption {fid}: '{m.group(0)}'")
    assert not hits, "; ".join(hits[:8])
    return True, f"{n_files} exported table and caption files contain no NaN or placeholder text (uncomputable values are written as 'n/a (n = k)')"


_run_check("No NaN or placeholder text in any exported table or caption", _qc_no_placeholders,
           "Find the reported cell, trace it to its analysis part and rerun; an uncomputable value must be written with na(n).")


# 11 ----------------------------------------------------------- completeness and reportability
_required_tables = ["T1", "T2", "T3", "T4", "T5", "T7", "T9"]
_missing_tables = [t for t in _required_tables if t not in TABLE_REGISTRY]
qc_add("Required tables T1-T5, T7 and T9 were exported", not _missing_tables, f"missing: {_missing_tables}" if _missing_tables else "all present",
       "Rerun Part 15.")
if CFG.FAST_MODE:
    qc_add("Results are reportable", None, "FAST_MODE is on: this is a debugging run (2 epochs, reduced bootstrap). Outputs must not be reported.", status="WARN")
else:
    qc_add("Results are reportable", CFG.N_BOOT >= 1000, f"FAST_MODE off; {CFG.N_BOOT} bootstrap resamples; configuration hash {CFG_HASH}", "Set N_BOOT >= 1000.")

QC_REPORT = pd.DataFrame(QC)
QC_REPORT.to_csv(DIRS["qc"] / "qc_report.csv", index=False)
_lines = [f"Quality-control report  |  {dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}  |  configuration {CFG_HASH}", "=" * 100]
for _q in QC:
    _lines.append(f"[{_q['status']:<4s}] {_q['check']}")
    _lines.append(f"        {_q['detail']}")
    if _q["status"] == "FAIL":
        _lines.append(f"        HOW TO FIX: {_q['fix']}")
(DIRS["qc"] / "qc_report.txt").write_text("\n".join(_lines) + "\n", encoding="utf-8")
print("\n".join(_lines))
_n_fail = int((QC_REPORT["status"] == "FAIL").sum())
QC_PASSED = _n_fail == 0
print("=" * 100)
print(f"PASS {int((QC_REPORT['status'] == 'PASS').sum())} | FAIL {_n_fail} | SKIP {int((QC_REPORT['status'] == 'SKIP').sum())} | WARN {int((QC_REPORT['status'] == 'WARN').sum())}")
if not QC_PASSED:
    raise RuntimeError(f"{_n_fail} quality-control check(s) FAILED. The results are not released (no summary, no zip archive). "
                       "See 'HOW TO FIX' above and qc_report/qc_report.txt.")

## Part 17. Results summary and release

**Purpose:** Write a results summary in which RQ1–RQ4 are each answered by quoting the computed numbers, then zip the results folder for download.

**Why:** The summary is the bridge to the Results section of the manuscript. Generating it from the result tables removes transcription errors.

**Method choice:** Sentences are produced from templates filled with computed values. The templates state estimates, intervals and adjusted p-values and whether an interval includes zero; they do not attach interpretive words such as "better" or "significant", and they make no claim that a number does not directly show. Interpretation, limitations (single centre, small sample, clinical-consensus labels for part of the benign cases, small subgroups, no external validation) and discussion are for the authors to write. This cell runs only if the QC cell passed.

**Output:** `results/results_summary.md`; `results.zip` (or `results_FAST_DEBUG_NOT_FOR_PUBLICATION.zip`) in the output root; a cell-by-cell printout of where everything was saved.

In [ ]:
assert QC_PASSED, "The quality-control cell did not pass; nothing is released."


def _rq1(model: str, metric: str) -> pd.Series:
    """RQ1 summary row of one model and metric (grouped scheme)."""
    return RQ1[(RQ1["scheme"] == "grouped") & (RQ1["model"] == model) & (RQ1["metric"] == metric)].iloc[0]


def _zero_text(lo: float, hi: float) -> str:
    """Factual statement about whether a confidence interval contains zero."""
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return "interval not available"
    return "interval includes zero" if lo <= 0.0 <= hi else "interval excludes zero"


S: List[str] = []
S.append("# Results summary (generated automatically from the computed results)")
S.append("")
if CFG.FAST_MODE:
    S.append("**FAST_MODE debugging run: these numbers must not be reported.**")
    S.append("")
S.append(f"Data: {N} images, {OBSERVED['n_patients']} patients, {N_LESION_PAIRS} lesions ({N_LESION_IDS} distinct lesion-identifier values), {K} classes ({', '.join(CLASS_NAMES)}). "
         f"Protocol: {CFG.N_REPEATS} × {CFG.N_FOLDS}-fold patient-grouped stratified cross-validation; backbone {CFG.BACKBONE_REGISTRY[CFG.BACKBONE]}; "
         f"configuration {CFG_HASH}. Intervals are {CI_PCT}% patient-level bootstrap intervals ({CFG.N_BOOT} resamples) of pooled out-of-fold estimates; "
         f"'a [l–u]; m ± s' means pooled estimate a with interval l–u, and mean m ± SD s over the test folds. BA = balanced accuracy. {MD_NOTE}.")
S.append("")
S.append("## RQ1. Discrimination")
for _m in MODELS:
    _b, _a = _rq1(_m, "balanced_accuracy"), _rq1(_m, "auroc_macro_ovr")
    S.append(f"- {MODEL_LABEL[_m]}: BA {fpm(_b.pooled, _b.ci_low, _b.ci_high, _b.fold_mean, _b.fold_sd, CFG.N_BOOT)}; "
             f"macro AUROC {fpm(_a.pooled, _a.ci_low, _a.ci_high, _a.fold_mean, _a.fold_sd, CFG.N_BOOT)}; "
             f"cancer sensitivity {_rq1(_m, 'cancer_sensitivity').pooled:.3f}, specificity {_rq1(_m, 'cancer_specificity').pooled:.3f}.")
_ranked = sorted(MODELS, key=lambda m: -_rq1(m, "balanced_accuracy").pooled)
S.append(f"- Highest pooled BA: {MODEL_LABEL[_ranked[0]]} ({_rq1(_ranked[0], 'balanced_accuracy').pooled:.3f}); lowest: {MODEL_LABEL[_ranked[-1]]} "
         f"({_rq1(_ranked[-1], 'balanced_accuracy').pooled:.3f}).")
S.append("- Paired comparisons (Δ BA = A − B, Holm-adjusted bootstrap p):")
for _r in STATS.itertuples():
    S.append(f"  - {MODEL_LABEL[_r.model_a]} vs {MODEL_LABEL[_r.model_b]}: Δ = {fest(_r.balanced_accuracy_diff, _r.balanced_accuracy_ci_low, _r.balanced_accuracy_ci_high, CFG.N_BOOT)}, "
             f"p = {fp(_r.balanced_accuracy_p_boot_holm, CFG.N_BOOT)} ({_zero_text(_r.balanced_accuracy_ci_low, _r.balanced_accuracy_ci_high)}).")
S.append(f"- {WILCOXON_NOTE}")
S.append("")
S.append("## RQ2. Robustness to missing metadata")
for _r in ROBUST_SUMMARY.itertuples():
    S.append(f"- {MODEL_LABEL[_r.model]}: BA {_r.ba_0:.3f} at 0%, {_r.ba_mid:.3f} at {100 * float(MID_RATE_LABEL):.0f}% and {_r.ba_100:.3f} at 100% randomly missing fields; "
             f"drop 0%→100% = {fest(_r.drop_0_to_100, _r.drop_0_to_100_ci_low, _r.drop_0_to_100_ci_high, N)}.")
if "image_only" in MODELS:
    _img100 = float(ROBUST_SUMMARY.set_index("model").loc["image_only", "ba_100"])
    _below = [MODEL_LABEL[r.model] for r in ROBUST_SUMMARY.itertuples() if r.model in FUSION_MODELS and r.ba_100 < _img100]
    S.append(f"- Fusion models with BA at 100% missing metadata below the image-only model ({_img100:.3f}): " + (", ".join(_below) if _below else "none") + ".")
for _r in ROBUST_MD_EFFECT.itertuples():
    S.append(f"- Modality dropout minus standard, {MODEL_LABEL[_r.model_standard]}, {_r.condition}: Δ BA = {fest(_r.diff, _r.ci_low, _r.ci_high, N)}, "
             f"Holm-adjusted p = {fp(_r.p_holm, CFG.N_BOOT)} ({_zero_text(_r.ci_low, _r.ci_high)}).")
for _n, _fields in SCENARIO_FIELDS.items():
    S.append(f"- Scenario '{_n.replace('_', ' ')}' ({len(_fields)} fields missing): " + "; ".join(
        f"{MODEL_LABEL[r.model]} {getattr(r, 'scenario_' + _n):.3f}" for r in ROBUST_SUMMARY.itertuples() if USES_METADATA[r.model]) + ".")
S.append("")
S.append("## RQ3. Calibration and selective prediction")
for _r in CALIB.itertuples():
    S.append(f"- {MODEL_LABEL[_r.model]}: ECE {_r.ece_raw:.3f} before and {_r.ece_ts:.3f} after temperature scaling (change {fest(_r.ece_change, _r.ece_change_ci_low, _r.ece_change_ci_high, N)}); "
             f"Brier {_r.brier_raw:.3f} → {_r.brier_ts:.3f}; AURC {_r.aurc_raw:.3f}; error {_r.error_full_coverage:.3f} at full coverage and "
             f"{fnum(1.0 - _r.selective_accuracy, int(_r.selective_n))} at {100 * _r.selective_coverage:.1f}% coverage (threshold from validation data, target {100 * CFG.SELECTIVE_TARGET_COVERAGE:.0f}%).")
S.append("")
S.append("## RQ4. Fairness across subgroups")
if len(FAIR):
    for _r in FAIR_GAPS[FAIR_GAPS["model"].isin([SELECTED[k] for k in ("image_only", "best_fusion") if k in SELECTED])].itertuples():
        if np.isfinite(_r.gap):
            S.append(f"- {_r.attribute}, {MODEL_LABEL[_r.model]}: BA gap between subgroups with at least {CFG.MIN_SUBGROUP_N} images = "
                     f"{fest(_r.gap, _r.gap_ci_low, _r.gap_ci_high, _r.n_eligible_subgroups)} (lowest: {_r.worst_subgroup}; highest: {_r.best_subgroup}; "
                     f"eligible subgroups: {_r.eligible_subgroups}).")
        else:
            S.append(f"- {_r.attribute}, {MODEL_LABEL[_r.model]}: gap {na(str(_r.n_eligible_subgroups) + ' eligible subgroups')}.")
    _small = FAIR[FAIR["flag_small"]][["attribute", "subgroup", "n_images"]].drop_duplicates()
    S.append("- Subgroups below the minimum size (reported, not interpreted): "
             + ("; ".join(f"{r.attribute} {r.subgroup} (n = {int(r.n_images)})" for r in _small.itertuples()) if len(_small) else "none") + ".")
else:
    S.append(f"- {na(0)}: no subgroup variable was available.")
S.append("")
S.append("## Metadata-missingness audit")
if AUDIT_ASSOC is not None:
    _aa = AUDIT_ASSOC
    S.append(f"- {int(MOSTLY_MISSING.sum())} of {N} images have at least {100 * CFG.AUDIT_ROW_MISSING_SHARE:.0f}% of their metadata fields missing: "
             f"{_aa['n_cancer_missing']} of {_aa['n_cancer_missing'] + _aa['n_cancer_recorded']} cancer images and "
             f"{_aa['n_noncancer_missing']} of {_aa['n_noncancer_missing'] + _aa['n_noncancer_recorded']} non-cancer images.")
    S.append(f"- The rule 'mostly missing ⇒ non-cancer', which uses no clinical content, has cancer sensitivity "
             f"{fest(_aa['rule_sensitivity'], _aa['rule_sensitivity_ci_low'], _aa['rule_sensitivity_ci_high'], N)} and specificity "
             f"{fest(_aa['rule_specificity'], _aa['rule_specificity_ci_low'], _aa['rule_specificity_ci_high'], N)}.")
    for _r in AUDIT_MODELS.itertuples():
        S.append(f"- Metadata-only model, {_r.variant}: BA {fest(_r.balanced_accuracy, _r.balanced_accuracy_ci_low, _r.balanced_accuracy_ci_high, N)}; "
                 f"cancer sensitivity {_r.cancer_sensitivity:.3f}, specificity {_r.cancer_specificity:.3f}.")
    for _r in AUDIT_STRATA[AUDIT_STRATA["stratum"] == "recorded"].itertuples():
        S.append(f"- Images with recorded metadata (n = {_r.n_images}), {MODEL_LABEL[_r.model]}: BA "
                 f"{fest(_r.balanced_accuracy, _r.balanced_accuracy_ci_low, _r.balanced_accuracy_ci_high, _r.n_images)}; cancer sensitivity "
                 f"{fnum(_r.cancer_sensitivity, _r.n_cancer)}, specificity {fnum(_r.cancer_specificity, _r.n_non_cancer)}.")
else:
    S.append(f"- Not run: {AUDIT_SKIP_REASON}.")
S.append("")
S.append("## Sensitivity analysis: restricted metadata")
if len(SENS):
    S.append(f"- Models marked (R) use only the low-missingness fields ({', '.join(LOW_MISSING_FIELDS)}).")
    for _r in SENS.itertuples():
        _line = (f"- {MODEL_LABEL[_r.model]}: BA {fest(_r.balanced_accuracy, _r.balanced_accuracy_ci_low, _r.balanced_accuracy_ci_high, N)}; "
                 f"cancer sensitivity {_r.cancer_sensitivity:.3f}, specificity {_r.cancer_specificity:.3f}; among images with recorded metadata "
                 f"{fnum(_r.recorded_cancer_sensitivity, _r.n_recorded_cancer)} / {fnum(_r.recorded_cancer_specificity, _r.n_recorded_non_cancer)}")
        if _r.model != "image_only":
            _line += (f"; Δ BA vs image only {fest(_r.ba_diff_vs_image_only, _r.ba_diff_vs_image_only_ci_low, _r.ba_diff_vs_image_only_ci_high, CFG.N_BOOT)}, "
                      f"Holm-adjusted p = {fp(_r.p_holm_vs_image_only, CFG.N_BOOT)} ({_zero_text(_r.ba_diff_vs_image_only_ci_low, _r.ba_diff_vs_image_only_ci_high)})"
                      f"; Δ BA vs {MODEL_LABEL[_r.full_metadata_model]} {fest(_r.ba_diff_vs_full_metadata, _r.ba_diff_vs_full_metadata_ci_low, _r.ba_diff_vs_full_metadata_ci_high, CFG.N_BOOT)}, "
                      f"Holm-adjusted p = {fp(_r.p_holm_vs_full_metadata, CFG.N_BOOT)}")
        S.append(_line + ".")
else:
    S.append(f"- Not run: {SENS_SKIP_REASON}.")
S.append("")
S.append("## Leakage analysis")
if len(LEAKAGE):
    for _r in LEAKAGE[LEAKAGE["metric"] == "balanced_accuracy"].itertuples():
        S.append(f"- {MODEL_LABEL[_r.model]}: BA {_r.grouped_pooled:.3f} with patient-grouped splits and {_r.image_level_pooled:.3f} with image-level splits; "
                 f"difference {fest(_r.inflation, _r.inflation_ci_low, _r.inflation_ci_high, N)} ({_zero_text(_r.inflation_ci_low, _r.inflation_ci_high)}).")
else:
    S.append("- Not run (CFG.RUN_LEAKAGE = False).")
S.append("")
S.append("## Explainability (supporting material)")
S.append("- " + (f"Grad-CAM panel for {MODEL_LABEL[GRADCAM_PANEL['model']]}: Figure F10." if GRADCAM_PANEL is not None else f"Grad-CAM not produced: {GRADCAM_SKIP_REASON}."))
if SHAP_IMPORTANCE is not None:
    S.append("- Metadata fields with the largest mean |SHAP| in the metadata-only model: "
             + ", ".join(f"{r.field} ({r.mean_abs_shap:.3f})" for r in SHAP_IMPORTANCE.head(5).itertuples()) + ".")
S.append("- These analyses are plausibility checks and do not establish clinical validity.")
S.append("")
S.append("## Outputs not produced")
_skips = [f"Figure {k}: {v}" for k, v in FIG_SKIPPED.items()] + [f"Table {k}: {v}" for k, v in TABLE_SKIPPED.items()]
S.extend([f"- {s}" for s in _skips] if _skips else ["- None."])
SUMMARY_TEXT = "\n".join(S) + "\n"
(DIRS["results"] / "results_summary.md").write_text(SUMMARY_TEXT, encoding="utf-8")
display(Markdown(SUMMARY_TEXT))

shutil.copyfile(DIRS["qc"] / "qc_report.txt", DIRS["results"] / "qc_report.txt")
_zip_name = "results_FAST_DEBUG_NOT_FOR_PUBLICATION" if CFG.FAST_MODE else "results"
_zip_path = shutil.make_archive(str(DIRS["out_root"] / _zip_name), "zip", root_dir=str(DIRS["out_root"]), base_dir="results")
print(f"\nResults archive: {_zip_path} ({os.path.getsize(_zip_path) / 1024**2:.1f} MB)")
print("Folders:")
for _k in ("data", "splits", "checkpoints", "predictions", "tables", "fig_single_column", "fig_double_column", "captions", "logs", "qc"):
    print(f"  {_k:<18s} {DIRS[_k]}")
if IN_COLAB:
    print("\nTo download the archive, right-click it in the Colab file browser, or run:\n"
          "  from google.colab import files; files.download(r'" + _zip_path + "')")

## Cell-by-cell index

The notebook has **75 cells** (38 Markdown, 37 code) before this index. Use this list to confirm that nothing is missing.

| Cell | Type | Part | Content |
|---|---|---|---|
| 1 | Markdown | 0 | Trustworthy Multimodal Skin-Lesion Diagnosis on PAD-UFES-20 |
| 2 | Markdown | 1 | Part 1. Environment setup — Install the small number of packages that are not guaranteed to be present in Colab, at pinned versions, and try to install the Arial font. |
| 3 | Code | 1 | Code for the preceding description; defines: _installed_version, ensure_package, try_install_arial |
| 4 | Markdown | 1 | Import all libraries, check the GPU, mount Google Drive, define the seeding function and print the software and hardware versions. |
| 5 | Code | 1 | Code for the preceding description; defines: _version_tuple, seed_everything |
| 6 | Markdown | 2 | Part 2. Configuration — One cell that holds every path, hyperparameter, backbone choice, seed, fold count, image size, epoch budget and analysis setting. |
| 7 | Code | 2 | Code for the preceding description |
| 8 | Markdown | 2 | Validate the configuration, apply the `FAST_MODE` overrides, create the output folder tree, compute the configuration hash and save the configuration and environment records. |
| 9 | Code | 2 | Code for the preceding description; defines: finalise_config, save_json, load_json |
| 10 | Markdown | 3 | Part 3. Data loading, verification and exploratory analysis — Locate the dataset (Drive folder, Drive archive or browser upload), extract any archives (including archives nested inside the main archive), find the metadata file … |
| 11 | Code | 3 | Code for the preceding description; defines: extract_archives, find_files, _header_has_role, locate_metadata_csv |
| 12 | Markdown | 3 | Load the metadata file, print what is actually in it (columns, dtypes, unique values, missing-value encodings), resolve the column roles, and assert the dataset counts. |
| 13 | Code | 3 | Code for the preceding description; defines: is_missing_token, resolve_roles |
| 14 | Markdown | 3 | Verify that every metadata row has an image file, then build (or reload) the resized image cache. |
| 15 | Code | 3 | Code for the preceding description; defines: match_image, load_resized, _find |
| 16 | Markdown | 3 | Convert the metadata to a clean, typed table of model input fields, with all missing-value encodings mapped to a real missing value. |
| 17 | Code | 3 | Code for the preceding description |
| 18 | Markdown | 3 | Exploratory analysis: class distribution, images per lesion and per patient, metadata missingness (overall and per class), Fitzpatrick distribution, image sizes. |
| 19 | Code | 3 | Code for the preceding description |
| 20 | Markdown | 4 | Part 4. Patient-grouped splits — Create the outer 5-fold test splits and, inside every training fold, a validation split; assert that no patient is shared between the parts; save the assignments. |
| 21 | Code | 4 | Code for the preceding description; defines: make_splits, get_split |
| 22 | Markdown | 5 | Part 5. Preprocessing pipelines (fitted on training folds only) — For every split, fit the metadata preprocessing (imputation with missingness indicators, scaling, one-hot encoding) on the **training part only**, save the fitte… |
| 23 | Code | 5 | Code for the preceding description; defines: build_preprocessor, field_groups, all_missing_frame, mask_raw, apply_field_mask, ids_digest |
| 24 | Markdown | 6 | Part 6. Models and training — Shared infrastructure for all model runs: run bookkeeping (done-detection, configuration-hash check), the standard prediction file format, the test-time metadata masking conditions for RQ2, and the… |
| 25 | Code | 6 | Code for the preceding description; defines: get_prep, needs_robust, run_seed, run_files, is_done, log_softmax_np, softmax_np, save_predictions … |
| 26 | Markdown | 6 | Train the metadata-only baseline (gradient boosting) and its missingness-augmented variant for every split; save validation, test and masked-test predictions and the SHAP contributions. |
| 27 | Code | 6 | Code for the preceding description; defines: fit_lgbm, run_gbm |
| 28 | Markdown | 6 | Dataset class, augmentation and data loaders for the image models. |
| 29 | Code | 6 | Code for the preceding description; defines: LesionDataset, _worker_init, make_loader |
| 30 | Markdown | 6 | Define the image backbone wrapper and the four network heads: image-only, early concatenation, cross-attention fusion and MetaBlock-style gated fusion. |
| 31 | Code | 6 | Code for the preceding description; defines: Backbone, FieldTokenizer, FusionNet, build_model, count_parameters |
| 32 | Markdown | 6 | The training engine: optimiser and schedule, mixed precision, weighted loss, modality dropout, validation, early stopping, per-epoch checkpointing with automatic resume, and prediction on validation, test and masked-test data. |
| 33 | Code | 6 | Code for the preceding description; defines: drive_safe_remove, drive_safe_replace, recover_interrupted_write, _amp_enabled, md_mask_torch, encode_tokens, head_logits, train_deep_run |
| 34 | Markdown | 6 | Run every network for every split (image-only, the three trainable fusion networks, and their modality-dropout variants; plus the image-level leakage runs if enabled), skipping finished runs. |
| 35 | Code | 6 | Code for the preceding description |
| 36 | Markdown | 6 | Late (decision-level) fusion of the image-only network and the metadata model. |
| 37 | Code | 6 | Code for the preceding description; defines: read_logits, run_late_fusion |
| 38 | Markdown | 7 | Part 7. Out-of-fold prediction storage — Collect the per-run prediction files into aligned arrays, verify their completeness and integrity, fit the temperature for every run on its validation part, and write one consolidated ou… |
| 39 | Code | 7 | Code for the preceding description; defines: fit_temperature |
| 40 | Markdown | 8 | Part 8. RQ1 — discrimination — Define all discrimination metrics, validate them against scikit-learn, and build the patient-level bootstrap resamples that every later interval uses. |
| 41 | Code | 8 | Code for the preceding description; defines: auc_binary_fast, compute_metrics, validate_metrics_against_sklearn, make_boot_indices, percentile_ci |
| 42 | Markdown | 8 | Compute the RQ1 results: per-fold metrics (mean ± SD across folds), pooled out-of-fold point estimates and patient-level bootstrap 95% confidence intervals for every model. |
| 43 | Code | 8 | Code for the preceding description; defines: bootstrap_metrics |
| 44 | Markdown | 9 | Part 9. Statistical comparison — Test the pre-specified model comparisons: every model against the image-only reference, and every modality-dropout variant against its standard counterpart. |
| 45 | Code | 9 | Code for the preceding description; defines: holm_adjust, paired_bootstrap, delong_test |
| 46 | Markdown | 10 | Part 10. RQ2 — robustness to missing metadata — Evaluate every model when metadata is partly or fully missing at test time (random missingness at 0–100% and the structured scenarios), and compare standard training with modality… |
| 47 | Code | 10 | Code for the preceding description; defines: weighted_ba, weighted_rate, _trapz_mean |
| 48 | Markdown | 11 | Part 11. RQ3 — calibration and selective prediction — Measure calibration (ECE, Brier score, negative log-likelihood, reliability diagrams) before and after temperature scaling, and selective prediction (risk–coverage curves, A… |
| 49 | Code | 11 | Code for the preceding description; defines: ece_equal_width, ece_equal_mass, brier_multiclass, nll_multiclass, risk_coverage, aurc, reliability_bins |
| 50 | Markdown | 12 | Part 12. RQ4 — fairness across Fitzpatrick skin types and other subgroups — Report performance per Fitzpatrick skin type, age band and sex, with sample sizes and patient-level bootstrap intervals, and the gap between the best a… |
| 51 | Code | 12 | Code for the preceding description |
| 52 | Markdown | 13 | Part 13. Leakage analysis (optional) — Quantify how much an image-level split, which lets images of the same patient appear in training and test data, inflates the estimated performance compared with the patient-grouped protocol. |
| 53 | Code | 13 | Code for the preceding description |
| 54 | Markdown | 13 | Metadata-missingness audit (extension of Part 13) — Test whether the *pattern of missing metadata*, rather than the clinical content of the metadata, reveals the diagnosis, and show how every model performs separately on images… |
| 55 | Code | 13 | Code for the preceding description |
| 56 | Markdown | 13 | Sensitivity analysis: models restricted to low-missingness metadata (extension of Part 13) — Estimate how much of the benefit of metadata and of fusion remains when the models cannot exploit which fields are missing. |
| 57 | Code | 13 | Code for the preceding description |
| 58 | Markdown | 14 | Part 14. Explainability (supporting evidence only) — Produce Grad-CAM maps for correctly and incorrectly classified test images of every class, and aggregate the SHAP contributions of the metadata fields. |
| 59 | Code | 14 | Code for the preceding description; defines: gradcam_map |
| 60 | Markdown | 15 | Part 15. Export of tables and figures — Define the single plotting style used by every figure, the fixed colour / marker / line-style assignment of every model, and the function that saves a figure in both journal widths and al… |
| 61 | Code | 15 | Code for the preceding description; defines: mm_to_px, new_figure, audit_figure, save_figure, render_both_widths, skip_figure, panel_label, model_handle |
| 62 | Markdown | 15 | Figures F1–F4: dataset overview, model comparison with confidence intervals, confusion matrices, and one-vs-rest ROC curves. |
| 63 | Code | 15 | Code for the preceding description; defines: _annotate_heatmap, build_F1, build_F2, build_F3, build_F4 |
| 64 | Markdown | 15 | Figures F5–F9: reliability diagrams before and after temperature scaling, risk–coverage curves, robustness curves, the fairness forest plot and the leakage comparison. |
| 65 | Code | 15 | Code for the preceding description; defines: build_F5, build_F6, build_F7, build_F8, build_F9 |
| 66 | Markdown | 15 | Figures F10 (Grad-CAM panel) and F11 (SHAP importance of metadata fields). |
| 67 | Code | 15 | Code for the preceding description; defines: _overlay, _show_image, build_F10, build_F11 |
| 68 | Markdown | 15 | Table export utilities: number formatting, automatic bolding of the best value, and writers for CSV, Word (`python-docx`) and LaTeX (`booktabs`). |
| 69 | Code | 15 | Code for the preceding description; defines: na, fnum, fci, fest, fpm, fp, best_mask, tex_escape … |
| 70 | Markdown | 15 | Build and export Tables T1–T12 (T1 dataset, T2 main results, T3 statistical comparisons, T4 calibration, T5 robustness, T6 fairness, T7 settings, T8 literature, T9 software and hardware, T10 leakage, T11 and T12 metadata-missin… |
| 71 | Code | 15 | Code for the preceding description; defines: _rq1_table |
| 72 | Markdown | 16 | Part 16. Automated quality-control report — Verify the integrity of the experiment and of every exported file, print PASS / FAIL for each check, and stop if any check fails. |
| 73 | Code | 16 | Code for the preceding description; defines: qc_add, _run_check, _qc_overlap, _qc_preprocessing, _qc_no_test_tuning, _qc_metrics, _qc_fig_files, _qc_fig_png … |
| 74 | Markdown | 17 | Part 17. Results summary and release — Write a results summary in which RQ1–RQ4 are each answered by quoting the computed numbers, then zip the results folder for download. |
| 75 | Code | 17 | Code for the preceding description; defines: _rq1, _zero_text |